## 加载BETA数据及导入库

In [ ]:
import os
import scipy.io as sio

subject_id = 'S18'
base_folder = '/Users/mikuru/SSVEP/DATA/BETA DATA'
current_subject_file_path = os.path.join(base_folder, f'{subject_id}.mat')

print(subject_id)
print(current_subject_file_path)

In [ ]:
import matplotlib.pyplot as plt # 用于绘图
import numpy as np # 用于数值计算
import scipy.io as sio # 用于处理.mat文件
import pandas as pd
from scipy.signal import butter, filtfilt # 用于带通滤波器
from scipy.signal import welch # 用于做welch PSD
from scipy.linalg import eigh

In [ ]:

# ============================================================
# 结果保存工具：分批写入 Parquet，避免所有 trial/window 结果长期堆在内存中
# ============================================================

import os
import shutil
import gc
import time
import numpy as np
import pandas as pd

RESULT_SAVE_ENABLED = True
RESULT_OUTPUT_DIR = os.path.join(os.getcwd(), "ssvep_results_2win")
_RESULT_PART_COUNTERS = {}
_RESULT_PARQUET_AVAILABLE = None
_RESULT_CSV_FALLBACK_WARNED = False


def reset_result_output_dir_beta(output_dir=RESULT_OUTPUT_DIR):
    """
    清空并重新创建结果目录。
    建议在完整重新运行 notebook 前执行一次，避免旧结果和新结果混在一起。
    """
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    _RESULT_PART_COUNTERS.clear()
    print(f"结果输出目录已初始化: {output_dir}")


def _safe_float_beta(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan



def _can_write_parquet_beta():
    """
    检查当前环境是否安装了 pyarrow 或 fastparquet。
    只检查一次，避免没有 parquet 引擎时反复打印大量警告。
    """
    global _RESULT_PARQUET_AVAILABLE

    if _RESULT_PARQUET_AVAILABLE is not None:
        return bool(_RESULT_PARQUET_AVAILABLE)

    try:
        import pyarrow  # noqa: F401
        _RESULT_PARQUET_AVAILABLE = True
    except Exception:
        try:
            import fastparquet  # noqa: F401
            _RESULT_PARQUET_AVAILABLE = True
        except Exception:
            _RESULT_PARQUET_AVAILABLE = False

    return bool(_RESULT_PARQUET_AVAILABLE)


def _append_result_part_beta(df, table_name, output_dir=RESULT_OUTPUT_DIR):
    """
    将一个 DataFrame 作为一个 part 写入结果目录。
    目录名使用 *.parquet，因此之后可以直接：
        pd.read_parquet("ssvep_results_2win/trial_results.parquet")
    如果当前环境没有 pyarrow/fastparquet，会自动退回到 csv_parts。
    """
    global _RESULT_CSV_FALLBACK_WARNED

    if not RESULT_SAVE_ENABLED:
        return None

    if df is None or len(df) == 0:
        return None

    os.makedirs(output_dir, exist_ok=True)

    df_to_save = df.copy()

    # pandas/parquet 对 object 列中的 list/dict 有时不稳定，这里只保留表格型字段。
    for col in df_to_save.columns:
        if df_to_save[col].dtype == "object":
            df_to_save[col] = df_to_save[col].apply(
                lambda x: str(x) if isinstance(x, (list, tuple, dict, set)) else x
            )

    part_id = int(_RESULT_PART_COUNTERS.get(table_name, 0))
    _RESULT_PART_COUNTERS[table_name] = part_id + 1

    if _can_write_parquet_beta():
        table_dir = os.path.join(output_dir, f"{table_name}.parquet")
        os.makedirs(table_dir, exist_ok=True)
        parquet_path = os.path.join(table_dir, f"part_{part_id:06d}.parquet")

        try:
            df_to_save.to_parquet(parquet_path, index=False)
            return parquet_path
        except Exception as exc:
            if not _RESULT_CSV_FALLBACK_WARNED:
                print("[WARN] Parquet 写入失败，后续将退回 CSV parts。")
                print(f"       原因: {repr(exc)}")
                _RESULT_CSV_FALLBACK_WARNED = True

    else:
        if not _RESULT_CSV_FALLBACK_WARNED:
            print("[WARN] 当前环境没有 pyarrow/fastparquet，结果将退回保存为 CSV parts。")
            print("       若希望保存为 Parquet，请先安装：pip install pyarrow")
            _RESULT_CSV_FALLBACK_WARNED = True

    csv_dir = os.path.join(output_dir, f"{table_name}.csv_parts")
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = os.path.join(csv_dir, f"part_{part_id:06d}.csv")
    df_to_save.to_csv(csv_path, index=False, encoding="utf-8-sig")
    return csv_path


def read_saved_result_table_beta(table_name, output_dir=RESULT_OUTPUT_DIR):
    """
    读取通过 _append_result_part_beta 分批保存的结果。
    优先读取 parquet parts；如果不存在则读取 csv_parts。
    """
    parquet_dir = os.path.join(output_dir, f"{table_name}.parquet")
    csv_dir = os.path.join(output_dir, f"{table_name}.csv_parts")

    if os.path.isdir(parquet_dir):
        part_files = sorted(
            os.path.join(parquet_dir, x)
            for x in os.listdir(parquet_dir)
            if x.endswith(".parquet")
        )
        if len(part_files) > 0:
            return pd.concat([pd.read_parquet(x) for x in part_files], axis=0, ignore_index=True)

    if os.path.isdir(csv_dir):
        part_files = sorted(
            os.path.join(csv_dir, x)
            for x in os.listdir(csv_dir)
            if x.endswith(".csv")
        )
        if len(part_files) > 0:
            return pd.concat([pd.read_csv(x) for x in part_files], axis=0, ignore_index=True)

    return pd.DataFrame()


def _normalise_strategy_columns_beta(df, strategy_name, min_stable_windows=np.nan):
    df_out = df.copy()
    df_out["strategy_name"] = str(strategy_name)
    if "min_stable_windows" not in df_out.columns:
        df_out["min_stable_windows"] = min_stable_windows
    return df_out


def _add_common_fold_index_columns_beta(df, split_mode="block"):
    df_out = df.copy()

    if "split_value" in df_out.columns:
        if "outer_fold_idx" not in df_out.columns:
            df_out["outer_fold_idx"] = df_out["split_value"]
        if str(split_mode) == "block" and "test_block_idx" not in df_out.columns:
            df_out["test_block_idx"] = df_out["split_value"]

    if "test_block" in df_out.columns and "test_block_idx" not in df_out.columns:
        df_out["test_block_idx"] = df_out["test_block"]
    if "test_block_idx" in df_out.columns and "outer_fold_idx" not in df_out.columns:
        df_out["outer_fold_idx"] = df_out["test_block_idx"]

    return df_out


def _add_fold_cache_metadata_to_trial_results_beta(trial_results_df, fold_cache):
    """
    给 trial_results_df 补上 subject_id / block_idx / freq_idx_global / target_freq 等元数据。
    fold_cache 中的 test_meta_df 与 trial_results_df 行顺序一一对应。
    """
    df = trial_results_df.copy()

    if "is_correct" not in df.columns:
        df["is_correct"] = df["y_true"].astype(int).values == df["y_pred"].astype(int).values

    if fold_cache is None:
        return df

    meta_df_cache = fold_cache.get("test_meta_df", None)
    if isinstance(meta_df_cache, pd.DataFrame) and len(meta_df_cache) == len(df):
        meta_reset = meta_df_cache.reset_index(drop=True).copy()
        for col in [
            "subject_id",
            "block_idx",
            "freq_idx_global",
            "label_local",
            "target_freq",
            "latency_correction_sec"
        ]:
            if col in meta_reset.columns and col not in df.columns:
                df[col] = meta_reset[col].values

    split_mode = fold_cache.get("split_mode", None)
    split_value = fold_cache.get("split_value", None)

    if "outer_fold_idx" not in df.columns:
        df["outer_fold_idx"] = split_value

    if str(split_mode) == "block" and "test_block_idx" not in df.columns:
        if "block_idx" in df.columns:
            df["test_block_idx"] = df["block_idx"]
        else:
            df["test_block_idx"] = split_value

    return df


def _add_fold_identity_to_summary_row_beta(fold_summary_row, trial_results_df, split_mode, split_value):
    """
    给 fold_summary_row 补 subject_id / outer_fold_idx / test_block_idx。
    """
    row = dict(fold_summary_row)
    row["outer_fold_idx"] = split_value

    if str(split_mode) == "block":
        row["test_block_idx"] = split_value

    if isinstance(trial_results_df, pd.DataFrame) and len(trial_results_df) > 0:
        if "subject_id" in trial_results_df.columns:
            unique_subjects = pd.Series(trial_results_df["subject_id"]).dropna().astype(str).unique().tolist()
            if len(unique_subjects) == 1:
                row["subject_id"] = unique_subjects[0]

        if "block_idx" in trial_results_df.columns:
            unique_blocks = pd.Series(trial_results_df["block_idx"]).dropna().unique().tolist()
            if len(unique_blocks) == 1:
                row["test_block_idx"] = unique_blocks[0]
                row["outer_fold_idx"] = unique_blocks[0]

    return row


def save_summary_result_df_beta(summary_df, strategy_name, min_stable_windows=np.nan):
    df = _normalise_strategy_columns_beta(
        summary_df,
        strategy_name=strategy_name,
        min_stable_windows=min_stable_windows
    )
    _append_result_part_beta(df, "summary_results")


def save_fold_result_df_beta(fold_results_df, strategy_name, min_stable_windows=np.nan):
    if fold_results_df is None or len(fold_results_df) == 0:
        return

    df = _normalise_strategy_columns_beta(
        fold_results_df,
        strategy_name=strategy_name,
        min_stable_windows=min_stable_windows
    )

    split_mode = df["split_mode"].iloc[0] if "split_mode" in df.columns and len(df) > 0 else "block"
    df = _add_common_fold_index_columns_beta(df, split_mode=split_mode)

    _append_result_part_beta(df, "fold_results")


def _get_threshold_for_outer_fold_beta(fold_results_df, outer_split_value):
    if fold_results_df is None or len(fold_results_df) == 0:
        return np.nan

    df = fold_results_df.copy()
    if "split_value" in df.columns:
        sub_df = df.loc[df["split_value"].astype(str) == str(outer_split_value)].copy()
    elif "test_block_idx" in df.columns:
        sub_df = df.loc[df["test_block_idx"].astype(str) == str(outer_split_value)].copy()
    else:
        sub_df = df.copy()

    if len(sub_df) == 0:
        sub_df = df.copy()

    if "best_threshold_from_inner_cv" in sub_df.columns:
        return _safe_float_beta(sub_df.iloc[0]["best_threshold_from_inner_cv"])
    if "threshold" in sub_df.columns:
        return _safe_float_beta(sub_df.iloc[0]["threshold"])

    return np.nan


def _iter_leaf_result_pairs_beta(result_with_thresholds, result_with_caches=None):
    """
    同时遍历 result_with_thresholds 和 result_with_caches 的 subject_result_list。
    用于 stable 结果：threshold 来自 stable_result，cache 来自 nonstable_result。
    """
    if result_with_caches is None:
        result_with_caches = result_with_thresholds

    threshold_children = result_with_thresholds.get("subject_result_list", None) if isinstance(result_with_thresholds, dict) else None
    cache_children = result_with_caches.get("subject_result_list", None) if isinstance(result_with_caches, dict) else None

    if isinstance(threshold_children, list) and len(threshold_children) > 0:
        for idx, child_threshold in enumerate(threshold_children):
            if isinstance(cache_children, list) and idx < len(cache_children):
                child_cache = cache_children[idx]
            else:
                child_cache = child_threshold
            yield from _iter_leaf_result_pairs_beta(child_threshold, child_cache)
    else:
        yield result_with_thresholds, result_with_caches


def _iter_outer_fold_caches_beta(cache_leaf_result):
    cache_dict = {}
    if isinstance(cache_leaf_result, dict):
        cache_dict = cache_leaf_result.get("outer_fold_cache_by_outer_fold", {}) or {}

    for outer_split_value, fold_cache in cache_dict.items():
        yield outer_split_value, fold_cache


def _build_window_detail_df_from_fold_cache_beta(
    fold_cache,
    model_name,
    strategy_name="fixed_window",
    threshold=np.nan,
    min_stable_windows=np.nan
):
    pred_label_matrix = np.asarray(fold_cache["pred_label_matrix"], dtype=int)
    margin_matrix = np.asarray(fold_cache["margin_matrix"], dtype=float)
    y_true = np.asarray(fold_cache["y_true"], dtype=int)
    sample_indices = np.asarray(fold_cache["sample_indices"], dtype=int)
    window_sec_list = np.asarray(fold_cache["window_sec_list"], dtype=float)
    runtime_matrix = np.asarray(
        fold_cache.get("runtime_matrix", np.zeros_like(margin_matrix, dtype=float)),
        dtype=float
    )
    if runtime_matrix.shape != margin_matrix.shape:
        runtime_matrix = np.zeros_like(margin_matrix, dtype=float)
    gaze_shift_sec = float(fold_cache.get("gaze_shift_sec", 0.55))

    detail_parts = []
    for window_idx, window_sec in enumerate(window_sec_list):
        y_pred = pred_label_matrix[:, window_idx]
        one_df = pd.DataFrame({
            "model_name": str(model_name),
            "strategy_name": str(strategy_name),
            "sample_idx": sample_indices.astype(int),
            "window_sec": float(window_sec),
            "y_true": y_true.astype(int),
            "y_pred": y_pred.astype(int),
            "is_correct": (y_true.astype(int) == y_pred.astype(int)),
            "margin_value": margin_matrix[:, window_idx].astype(float),
            "recognition_runtime_sec": runtime_matrix[:, window_idx].astype(float),
            "decision_time_with_runtime_sec": float(window_sec + gaze_shift_sec) + runtime_matrix[:, window_idx].astype(float),
            "threshold": threshold,
            "min_stable_windows": min_stable_windows
        })
        one_df = _add_fold_cache_metadata_to_trial_results_beta(one_df, fold_cache)
        detail_parts.append(one_df)

    return pd.concat(detail_parts, axis=0, ignore_index=True)


def _build_window_sweep_df_from_fold_cache_beta(
    fold_cache,
    model_name,
    strategy_name="fixed_window",
    threshold=np.nan,
    min_stable_windows=np.nan
):
    pred_label_matrix = np.asarray(fold_cache["pred_label_matrix"], dtype=int)
    y_true = np.asarray(fold_cache["y_true"], dtype=int)
    window_sec_list = np.asarray(fold_cache["window_sec_list"], dtype=float)
    runtime_matrix = np.asarray(
        fold_cache.get("runtime_matrix", np.zeros_like(pred_label_matrix, dtype=float)),
        dtype=float
    )
    if runtime_matrix.shape != pred_label_matrix.shape:
        runtime_matrix = np.zeros_like(pred_label_matrix, dtype=float)
    gaze_shift_sec = float(fold_cache.get("gaze_shift_sec", 0.55))
    n_classes = int(fold_cache["n_classes"])

    meta_df_cache = fold_cache.get("test_meta_df", pd.DataFrame())
    subject_id_value = np.nan
    if isinstance(meta_df_cache, pd.DataFrame) and "subject_id" in meta_df_cache.columns:
        unique_subjects = pd.Series(meta_df_cache["subject_id"]).dropna().astype(str).unique().tolist()
        if len(unique_subjects) == 1:
            subject_id_value = unique_subjects[0]

    rows = []
    for window_idx, window_sec in enumerate(window_sec_list):
        y_pred = pred_label_matrix[:, window_idx]
        acc = float((y_true.astype(int) == y_pred.astype(int)).mean())
        decision_time_sec = float(window_sec + gaze_shift_sec)
        mean_recognition_runtime_sec = float(runtime_matrix[:, window_idx].mean()) if runtime_matrix.shape[0] > 0 else 0.0
        mean_decision_time_with_runtime_sec = float(decision_time_sec + mean_recognition_runtime_sec)
        itr_value = float(compute_itr_bits_per_min(
            n_classes=n_classes,
            accuracy=acc,
            decision_time_sec=decision_time_sec
        ))
        itr_with_runtime_value = float(compute_itr_bits_per_min(
            n_classes=n_classes,
            accuracy=acc,
            decision_time_sec=mean_decision_time_with_runtime_sec
        ))

        row = {
            "subject_id": subject_id_value,
            "model_name": str(model_name),
            "strategy_name": str(strategy_name),
            "split_mode": fold_cache.get("split_mode", "block"),
            "split_value": fold_cache.get("split_value", np.nan),
            "outer_fold_idx": fold_cache.get("split_value", np.nan),
            "test_block_idx": fold_cache.get("split_value", np.nan),
            "window_sec": float(window_sec),
            "threshold": threshold,
            "min_stable_windows": min_stable_windows,
            "fold_accuracy": acc,
            "mean_decision_time_sec": decision_time_sec,
            "mean_recognition_runtime_sec": mean_recognition_runtime_sec,
            "mean_decision_time_with_runtime_sec": mean_decision_time_with_runtime_sec,
            "itr_bits_per_min": itr_value,
            "itr_with_runtime_bits_per_min": itr_with_runtime_value,
            "n_test": int(len(y_true))
        }
        rows.append(row)

    return pd.DataFrame(rows)



def evaluate_fixed_window_from_cache_beta(
    fold_cache,
    model_name
):
    """
    固定窗无早停：只使用最大窗一次识别结果。
    recognition_runtime_sec 只取最大窗这一列，不累计前面更短窗口。
    """
    pred_label_matrix = np.asarray(fold_cache['pred_label_matrix'], dtype=int)
    margin_matrix = np.asarray(fold_cache['margin_matrix'], dtype=float)
    runtime_matrix = np.asarray(
        fold_cache.get('runtime_matrix', np.zeros_like(margin_matrix, dtype=float)),
        dtype=float
    )
    if runtime_matrix.shape != margin_matrix.shape:
        runtime_matrix = np.zeros_like(margin_matrix, dtype=float)

    y_true = np.asarray(fold_cache['y_true'], dtype=int)
    sample_indices = np.asarray(fold_cache['sample_indices'], dtype=int)
    window_sec_list = np.asarray(fold_cache['window_sec_list'], dtype=float)

    n_samples, n_windows = margin_matrix.shape
    stop_window_idx = n_windows - 1
    row_idx = np.arange(n_samples)

    y_pred = pred_label_matrix[row_idx, stop_window_idx]
    selected_margin = margin_matrix[row_idx, stop_window_idx]
    stop_window_sec = np.repeat(float(window_sec_list[stop_window_idx]), n_samples)
    decision_time_sec = stop_window_sec + float(fold_cache['gaze_shift_sec'])
    recognition_runtime_sec = runtime_matrix[row_idx, stop_window_idx]
    decision_time_with_runtime_sec = decision_time_sec + recognition_runtime_sec

    trial_results_df = pd.DataFrame({
        'sample_idx': sample_indices.astype(int),
        'y_true': y_true.astype(int),
        'y_pred': y_pred.astype(int),
        'margin_value': selected_margin.astype(float),
        'stop_window_sec': stop_window_sec.astype(float),
        'decision_time_sec': decision_time_sec.astype(float),
        'recognition_runtime_sec': recognition_runtime_sec.astype(float),
        'decision_time_with_runtime_sec': decision_time_with_runtime_sec.astype(float),
        'stopped_early': False,
        'stop_reason': 'fixed_window'
    })

    trial_results_df = _add_fold_cache_metadata_to_trial_results_beta(
        trial_results_df=trial_results_df,
        fold_cache=fold_cache
    )

    return summarize_one_fold_nonstable_early_stop_beta(
        trial_result_rows=trial_results_df.to_dict('records'),
        model_name=model_name,
        split_mode=fold_cache['split_mode'],
        split_value=fold_cache['split_value'],
        threshold=np.nan,
        n_classes=fold_cache['n_classes']
    )


def _save_final_trial_and_fold_from_cache_beta(
    fold_cache,
    model_name,
    strategy_name,
    threshold,
    min_stable_windows=np.nan
):
    if strategy_name == "stable_early_stop":
        eval_result = evaluate_stable_early_stop_from_cache_beta(
            fold_cache=fold_cache,
            threshold=threshold,
            model_name=model_name,
            min_stable_windows=int(min_stable_windows)
        )
    elif strategy_name == "fixed_window":
        eval_result = evaluate_fixed_window_from_cache_beta(
            fold_cache=fold_cache,
            model_name=model_name
        )
    else:
        eval_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=fold_cache,
            threshold=threshold,
            model_name=model_name
        )

    trial_df = eval_result["trial_results_df"].copy()
    trial_df = _normalise_strategy_columns_beta(
        trial_df,
        strategy_name=strategy_name,
        min_stable_windows=min_stable_windows
    )
    trial_df["model_name"] = str(model_name)
    trial_df["threshold"] = threshold

    if str(strategy_name) == "fixed_window":
        trial_df["threshold"] = np.nan
        trial_df["stopped_early"] = False
        trial_df["stop_reason"] = "fixed_window"

    trial_df = _add_fold_cache_metadata_to_trial_results_beta(trial_df, fold_cache)
    _append_result_part_beta(trial_df, "trial_results")

    fold_row = eval_result["fold_summary_row"].copy()
    if str(strategy_name) == "fixed_window":
        fold_row["threshold"] = np.nan
        fold_row["early_stop_rate"] = 0.0

    fold_df = pd.DataFrame([fold_row])
    fold_df = _normalise_strategy_columns_beta(
        fold_df,
        strategy_name=strategy_name,
        min_stable_windows=min_stable_windows
    )
    fold_df = _add_common_fold_index_columns_beta(
        fold_df,
        split_mode=fold_cache.get("split_mode", "block")
    )
    _append_result_part_beta(fold_df, "fold_results")


def save_threshold_sweep_result_beta(result, model_name, strategy_name, min_stable_windows=np.nan):
    """
    保存 threshold_search_by_outer_fold。
    每行是一个 outer fold 下的一个候选 threshold 的平均验证结果。
    """
    for leaf_result, _ in _iter_leaf_result_pairs_beta(result, result):
        fold_results_df = leaf_result.get("fold_results_df", pd.DataFrame())
        subject_id_value = np.nan
        if len(fold_results_df) > 0 and "subject_id" in fold_results_df.columns:
            unique_subjects = pd.Series(fold_results_df["subject_id"]).dropna().astype(str).unique().tolist()
            if len(unique_subjects) == 1:
                subject_id_value = unique_subjects[0]

        threshold_dict = leaf_result.get("threshold_search_by_outer_fold", {}) or {}
        for outer_split_value, search_df in threshold_dict.items():
            if search_df is None or len(search_df) == 0:
                continue

            df = search_df.copy()
            df["subject_id"] = subject_id_value
            df["model_name"] = str(model_name)
            df["strategy_name"] = str(strategy_name)
            df["split_mode"] = "block"
            df["outer_fold_idx"] = outer_split_value
            df["test_block_idx"] = outer_split_value
            df["min_stable_windows"] = min_stable_windows

            if "threshold" in df.columns:
                df["threshold_candidate"] = df["threshold"]

            _append_result_part_beta(df, "threshold_sweep_results")


def save_early_stop_tables_beta(
    result,
    model_name,
    strategy_name,
    cache_result=None,
    min_stable_windows=np.nan,
    save_window_cache=False,
    save_fixed_window_from_cache=False
):
    """
    保存一个模型一种早停策略的 summary/fold/trial/threshold 结果。
    - result: 当前策略结果，例如 svm_nonstable_result 或 svm_stable_result
    - cache_result: 提供 outer_fold_cache_by_outer_fold 的结果；stable 时传入对应 nonstable_result
    """
    if cache_result is None:
        cache_result = result

    save_summary_result_df_beta(
        result["summary_df"],
        strategy_name=strategy_name,
        min_stable_windows=min_stable_windows
    )

    # fold_results 会从 outer_fold_cache 重新生成并保存；
    # 这样可以确保 fold 表和 trial 表来自同一批最终 trial 结果，避免重复行。

    save_threshold_sweep_result_beta(
        result=result,
        model_name=model_name,
        strategy_name=strategy_name,
        min_stable_windows=min_stable_windows
    )

    for threshold_leaf, cache_leaf in _iter_leaf_result_pairs_beta(result, cache_result):
        leaf_fold_df = threshold_leaf.get("fold_results_df", pd.DataFrame())

        for outer_split_value, fold_cache in _iter_outer_fold_caches_beta(cache_leaf):
            threshold_value = _get_threshold_for_outer_fold_beta(
                fold_results_df=leaf_fold_df,
                outer_split_value=outer_split_value
            )

            _save_final_trial_and_fold_from_cache_beta(
                fold_cache=fold_cache,
                model_name=model_name,
                strategy_name=strategy_name,
                threshold=threshold_value,
                min_stable_windows=min_stable_windows
            )

            if save_window_cache:
                window_detail_df = _build_window_detail_df_from_fold_cache_beta(
                    fold_cache=fold_cache,
                    model_name=model_name,
                    strategy_name="fixed_window",
                    threshold=np.nan,
                    min_stable_windows=np.nan
                )
                _append_result_part_beta(window_detail_df, "window_detail_results")
                del window_detail_df

                window_sweep_df = _build_window_sweep_df_from_fold_cache_beta(
                    fold_cache=fold_cache,
                    model_name=model_name,
                    strategy_name="fixed_window",
                    threshold=np.nan,
                    min_stable_windows=np.nan
                )
                _append_result_part_beta(window_sweep_df, "window_sweep_results")
                del window_sweep_df

            if save_fixed_window_from_cache:
                _save_final_trial_and_fold_from_cache_beta(
                    fold_cache=fold_cache,
                    model_name=model_name,
                    strategy_name="fixed_window",
                    threshold=np.inf,
                    min_stable_windows=np.nan
                )

            gc.collect()


def release_prediction_caches_from_result_beta(result):
    """
    释放早停结果对象中的 prediction cache，降低后续 notebook 内存占用。
    注意：释放后仍保留 summary_df、fold_results_df、threshold_search_by_outer_fold。
    """
    if not isinstance(result, dict):
        return result

    if "subject_result_list" in result and isinstance(result["subject_result_list"], list):
        for child in result["subject_result_list"]:
            release_prediction_caches_from_result_beta(child)

    for key in ["inner_fold_cache_by_outer_fold", "outer_fold_cache_by_outer_fold"]:
        if key in result:
            result[key] = {}

    gc.collect()
    return result


def finalize_result_excel_beta(output_dir=RESULT_OUTPUT_DIR):
    """
    额外导出一个 Excel，主要用于快速查看 summary/fold/threshold 小表。
    大的 trial/window_detail 结果仍建议用 read_parquet 读取。
    """
    excel_path = os.path.join(output_dir, "summary_results.xlsx")

    table_names = [
        "summary_results",
        "fold_results",
        "window_sweep_results",
        "threshold_sweep_results"
    ]

    with pd.ExcelWriter(excel_path) as writer:
        for table_name in table_names:
            df = read_saved_result_table_beta(table_name, output_dir=output_dir)
            if len(df) == 0:
                continue

            sheet_name = table_name[:31]
            # Excel 不适合写入超大表，这里最多写 200000 行用于查看。
            df.head(200000).to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Excel 汇总文件已保存: {excel_path}")
    return excel_path


reset_result_output_dir_beta()


In [ ]:
mat_info = sio.whosmat(current_subject_file_path) # 获取.mat文件中的
print(mat_info) # 结果包含一个名为data的变量，维度为(1, 1)

In [ ]:
load_data = sio.loadmat(current_subject_file_path, struct_as_record = False, squeeze_me = True) # 加载.mat文件内容
data = load_data['data'] # 提取'data'变量
print(data)

In [ ]:
print(dir(data))

In [ ]:
eeg = data.EEG
info = data.suppl_info
print(eeg.shape)
print(dir(eeg))
print(dir(info))
# info打印出数据集携带的字段,包含的内容如下
# srate：采样率（Hz）
# freqs：刺激频率
# phases：每个刺激的初始相位
# chan：通道信息，通常是 64×4，最后一列是通道名，比如 Oz、POz。中间两个数字为极坐标,在EEG中,(0,1)为正前方,角度右转为正,左转为负
# sub：受试者编号，比如 'S1'。
# age / gender：受试者年龄、性别。
# bci_quotient：论文里提到的一个表现相关指标。
# narrow_snr / wide_snr：作者算好的信噪比，两种算法的结果。

In [ ]:
# print(info.chan)

In [ ]:
# print(info.chan) # 按前面的说明，打印本实验对象的年龄信息

In [ ]:
# print(info.freqs)

In [ ]:
# print(info.srate)

## 四个初始函数_第二版

### 函数1 输入通道和窗长，截出目标时间窗

In [ ]:
# ------------ 函数1 输入通道和窗长，截出目标时间窗 ------------
# 取所有通道名组成数组
chan_list = info.chan[:, -1]
chan_list_py = []
for i in chan_list:
  chan_list_py.append(str(i))

# 取所有频率组成数组
freqs_list = info.freqs
freqs_list_py = []
for i in freqs_list:
  freq = format(i, '.1f')
  freqs_list_py.append(str(freq))

print(chan_list_py)
print(freqs_list_py)

# 查找频率在数组中的位置
def select_freq(freq_name, freqs_list_py):
    for i in range(len(freqs_list_py)):
        if freqs_list_py[i] == freq_name:
            return i

# 截取函数
def extract_trial_window(
    eeg,
    chan_list_py,
    selected_chans,
    block_idx,
    freq_idx,
    srate,
    window_sec=None,
    pre_stim_sec=0.5,
    latency_correction_sec=None,
    T_sec=None,
    delta_sec=None
):
    if window_sec is None:
        if T_sec is None:
            raise ValueError('window_sec 和 T_sec 不能同时为空。')
        window_sec = T_sec

    if latency_correction_sec is None:
        if delta_sec is None:
            latency_correction_sec = 0.13
        else:
            latency_correction_sec = delta_sec

    if isinstance(selected_chans, str):
        selected_chans = [selected_chans]
    selected_chans = [str(ch) for ch in selected_chans]

    if len(selected_chans) == 0:
        raise ValueError('selected_chans 不能为空。')

    chan_name_to_idx = {}
    for i, ch_name in enumerate(chan_list_py):
        chan_name_to_idx[str(ch_name).upper()] = i

    selected_idx = []
    missing_chans = []
    for ch in selected_chans:
        ch_upper = ch.upper()
        if ch_upper in chan_name_to_idx:
            selected_idx.append(chan_name_to_idx[ch_upper])
        else:
            missing_chans.append(ch)

    if len(missing_chans) > 0:
        raise ValueError(f'以下通道在当前受试者中不存在: {missing_chans}')

    eeg = np.asarray(eeg, dtype=float)

    if eeg.ndim != 4:
        raise ValueError(f'eeg 应为4维数组 [channels, samples, blocks, freqs]，当前 ndim={eeg.ndim}')

    n_channels, n_total_samples, n_blocks, n_freqs = eeg.shape

    if block_idx < 0 or block_idx >= n_blocks:
        raise IndexError(f'block_idx={block_idx} 超出范围 [0, {n_blocks - 1}]')

    if freq_idx < 0 or freq_idx >= n_freqs:
        raise IndexError(f'freq_idx={freq_idx} 超出范围 [0, {n_freqs - 1}]')

    start_time_sec = float(pre_stim_sec + latency_correction_sec)
    start_sample = int(round(start_time_sec * srate))
    window_samples = int(round(float(window_sec) * srate))
    end_sample = start_sample + window_samples

    if start_sample < 0:
        raise ValueError(f'start_sample={start_sample} 非法，请检查 pre_stim_sec 和 latency_correction_sec。')

    if end_sample > n_total_samples:
        raise ValueError(
            f'截窗越界: end_sample={end_sample} > 总采样点数={n_total_samples}。'
            f' 请减小 window_sec，或检查 pre_stim_sec / latency_correction_sec。'
        )

    trial_window = eeg[selected_idx, start_sample:end_sample, block_idx, freq_idx]
    trial_window = np.asarray(trial_window, dtype=float)

    return trial_window

In [ ]:
# 函数1调用例子
freq_idx = select_freq('10.4', freqs_list_py) # 查找8.6Hz在数组中的位置索引

trial_window = extract_trial_window(
  eeg=eeg, # 原始eeg数据
  chan_list_py = chan_list_py, # 所有通道名称列表
  selected_chans = ['O1', 'OZ', 'O2', 'POZ'], # 选择的通道名称列表
  block_idx = 0, # 取哪个block
  freq_idx = freq_idx, # 取哪个刺激频率
  srate = info.srate, # 采样率，这里info中为250Hz
  T_sec = 1.2, # 窗长
  pre_stim_sec = 0.5, # 刺激前长度（秒）
  delta_sec = 0.13 # 视觉延迟时间（130ms = 0.13秒）
)

print(trial_window.shape) # 截出来的结果形状，本例中为（4, 250）
print(trial_window[:, :5]) # 打印实际内容中每行的前五个数

### 函数2 对一个通道片段做FFT，分别输出幅度谱和功率谱

In [ ]:
def bandpass_filter_signal(
    one_channel_signal,
    srate,
    lowcut=5.0,
    highcut=60.0,
    filter_order=4
):
    signal_array = np.asarray(one_channel_signal, dtype=float)

    nyquist = srate / 2.0
    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(
        filter_order,
        [low, high],
        btype='band'
    )

    filtered_signal = filtfilt(b, a, signal_array)
    return np.asarray(filtered_signal, dtype=float)


def compute_fft_amplitude_and_power(
    one_channel_signal,
    srate,
    use_bandpass=False,
    lowcut=5.0,
    highcut=60.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0
):
    signal_array = np.asarray(one_channel_signal, dtype=float)
    signal_centered = signal_array - np.mean(signal_array)

    if use_bandpass:
        signal_processed = bandpass_filter_signal(
            one_channel_signal=signal_centered,
            srate=srate,
            lowcut=lowcut,
            highcut=highcut,
            filter_order=filter_order
        )
    else:
        signal_processed = signal_centered

    if use_hann_window:
        window = np.hanning(len(signal_processed))
        signal_windowed = signal_processed * window
    else:
        signal_windowed = signal_processed

    original_length = len(signal_windowed)

    if use_zero_padding:
        if nfft is None:
            fft_length = int(round(srate * pad_total_sec))
        else:
            fft_length = int(nfft)
    else:
        fft_length = original_length

    fft_result = np.fft.rfft(signal_windowed, n=fft_length)
    freqs = np.fft.rfftfreq(fft_length, d=1.0 / srate)

    abs_fft = np.abs(fft_result)
    spectrum_amplitude = abs_fft / fft_length
    spectrum_power = (abs_fft ** 2) / fft_length

    return (
        np.asarray(freqs, dtype=float),
        np.asarray(spectrum_amplitude, dtype=float),
        np.asarray(spectrum_power, dtype=float)
    )


def compute_fft_amplitude(
    one_channel_signal,
    srate,
    use_bandpass=False,
    lowcut=5.0,
    highcut=60.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0
):
    freqs, spectrum_amplitude, _ = compute_fft_amplitude_and_power(
        one_channel_signal=one_channel_signal,
        srate=srate,
        use_bandpass=use_bandpass,
        lowcut=lowcut,
        highcut=highcut,
        filter_order=filter_order,
        use_hann_window=use_hann_window,
        use_zero_padding=use_zero_padding,
        nfft=nfft,
        pad_total_sec=pad_total_sec
    )
    return freqs, spectrum_amplitude


def compute_fft_power(
    one_channel_signal,
    srate,
    use_bandpass=False,
    lowcut=5.0,
    highcut=60.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0
):
    freqs, _, spectrum_power = compute_fft_amplitude_and_power(
        one_channel_signal=one_channel_signal,
        srate=srate,
        use_bandpass=use_bandpass,
        lowcut=lowcut,
        highcut=highcut,
        filter_order=filter_order,
        use_hann_window=use_hann_window,
        use_zero_padding=use_zero_padding,
        nfft=nfft,
        pad_total_sec=pad_total_sec
    )
    return freqs, spectrum_power

### 函数3 分别对单个通道求窄带SNR和宽带SNR



In [ ]:
# ------------ 函数3-1 对一个通道的幅度谱，计算40个候选频率的窄带SNR ------------

# BETA窄带SNR过程：
# ①依次取出每一个候选频率 f0
# ②找到最接近 f0 的频率点，取该点的幅度作为 signal_amplitude
# ③在该点左右各取5个邻近频率点（共10个点）
# ④对这10个邻近点的幅度求平均，作为 noise_amplitude
# ⑤用 20 * log10(signal_amplitude / noise_amplitude) 计算当前频率的窄带SNR
# ⑥对40个候选频率都重复一次，最终得到长度为40的 snr_vector

def compute_snr_vector_narrow_beta(
  freqs,
  spectrum_amplitude,
  candidate_freqs,
  neighbor_count = 5
):
  freqs_array = np.array(freqs, dtype=float)

  spectrum_amplitude_array = np.array(spectrum_amplitude, dtype=float)

  candidate_freqs_array = np.array(candidate_freqs, dtype=float)

  snr_vector = []

  eps = 1e-12

  for f0 in candidate_freqs_array:

      # ---------- 第1步：找到最接近f0的频率点 ----------
      min_diff = abs(freqs_array[0] - f0)
      signal_idx = 0

      for i in range(len(freqs_array)):
          current_diff = abs(freqs_array[i] - f0)

          if current_diff < min_diff:
              min_diff = current_diff
              signal_idx = i

      # ---------- 第2步：取目标点幅度作为signal_amplitude ----------
      signal_amplitude = spectrum_amplitude_array[signal_idx]

      # ---------- 第3步：取左右各5个邻近频率点作为noise ----------
      noise_values = []

      for k in range(1, neighbor_count + 1):
          left_idx = signal_idx - k
          right_idx = signal_idx + k

          if left_idx >= 0:
              noise_values.append(spectrum_amplitude_array[left_idx])

          if right_idx < len(spectrum_amplitude_array):
              noise_values.append(spectrum_amplitude_array[right_idx])

      # ---------- 第4步：计算noise平均幅度 ----------
      if len(noise_values) > 0:
          noise_amplitude = np.mean(noise_values)
      else:
          noise_amplitude = eps

      # ---------- 第5步：计算当前候选频率的窄带SNR ----------
      snr_value = 20.0 * np.log10((signal_amplitude + eps) / (noise_amplitude + eps))

      # ---------- 第6步：保存到SNR向量 ----------
      snr_vector.append(snr_value)

  snr_vector = np.array(snr_vector, dtype=float)

  return snr_vector

In [ ]:
# ------------ 函数3-2 对一个通道的功率谱，计算40个候选频率的宽带SNR ------------

# BETA宽带SNR过程：
# ①依次取出每一个候选频率 f0
# ②计算 f0 的前 n_harmonics 个谐波频率
# ③找到每个谐波频率在功率谱中最接近的频率点
# ④将这些谐波点的功率相加，作为 signal_power
# ⑤将整个功率谱的总功率减去 signal_power，作为 noise_power
# ⑥用 10 * log10(signal_power / noise_power) 计算当前频率的宽带SNR
# ⑦对40个候选频率都重复一次，最终得到长度为40的 snr_vector

def compute_snr_vector_wide_beta(
  freqs,
  spectrum_power,
  candidate_freqs,
  n_harmonics = 5
):
  freqs_array = np.array(freqs, dtype=float)

  spectrum_power_array = np.array(spectrum_power, dtype=float)

  candidate_freqs_array = np.array(candidate_freqs, dtype=float)

  snr_vector = []

  eps = 1e-12

  total_power = np.sum(spectrum_power_array)

  for f0 in candidate_freqs_array:

      # ---------- 第1步：收集当前候选频率的谐波索引 ----------
      harmonic_index_list = []

      for h in range(1, n_harmonics + 1):
          harmonic_freq = f0 * h

          # 如果谐波频率超过频率轴范围，就跳过
          if harmonic_freq > freqs_array[-1]:
              continue

          min_diff = abs(freqs_array[0] - harmonic_freq)
          harmonic_idx = 0

          for i in range(len(freqs_array)):
              current_diff = abs(freqs_array[i] - harmonic_freq)

              if current_diff < min_diff:
                  min_diff = current_diff
                  harmonic_idx = i

          harmonic_index_list.append(harmonic_idx)

      # 去掉重复索引，避免重复累加
      harmonic_index_list = list(set(harmonic_index_list))

      # ---------- 第2步：计算signal_power ----------
      signal_power = 0.0

      for idx in harmonic_index_list:
          signal_power = signal_power + spectrum_power_array[idx]

      # ---------- 第3步：计算noise_power ----------
      noise_power = total_power - signal_power

      if noise_power <= 0:
          noise_power = eps

      # ---------- 第4步：计算当前候选频率的宽带SNR ----------
      snr_value = 10.0 * np.log10((signal_power + eps) / (noise_power + eps))

      # ---------- 第5步：保存到SNR向量 ----------
      snr_vector.append(snr_value)

  snr_vector = np.array(snr_vector, dtype=float)

  return snr_vector

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import scipy.io as sio

# # ========= 你的参数 =========
# selected_subject_ids = [
#     'S23', 'S48',
#     'S1', 'S29',
#     'S26', 'S31',
#     'S47', 'S61'
# ]

# selected_chans = [
#     'PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2'
# ]

# T_sec = 2.0

# base_folder = '/content/drive/MyDrive/论文2025/BETA database'


# # ========= 保存结果 =========
# summary_results = []
# all_trial_results = []


# # ========= 开始遍历受试者 =========
# for subject_id in selected_subject_ids:

#     current_subject_file_path = os.path.join(base_folder, subject_id + '.mat')

#     load_data = sio.loadmat(
#         current_subject_file_path,
#         struct_as_record = False,
#         squeeze_me = True
#     )

#     data = load_data['data']

#     eeg = data.EEG
#     info = data.suppl_info

#     chan_list = info.chan[:, -1]

#     chan_list_py = []
#     for x in chan_list:
#         chan_list_py.append(str(x))

#     candidate_freqs = []
#     for x in info.freqs:
#         candidate_freqs.append(float(x))

#     n_blocks = eeg.shape[2]
#     n_freqs = len(candidate_freqs)

#     correct_count_narrow = 0
#     correct_count_wide = 0
#     total_count = 0

#     valid_chan_count = 0

#     print('========================================')
#     print('当前受试者 =', subject_id)

#     # ========= 遍历所选通道 =========
#     for one_chan in selected_chans:

#         chan_exists = False

#         for ch_name in chan_list_py:
#             if str(ch_name).upper() == str(one_chan).upper():
#                 chan_exists = True
#                 break

#         if chan_exists == False:
#             print('通道不存在，跳过 =', one_chan)
#             continue

#         valid_chan_count = valid_chan_count + 1

#         print('------------------------------')
#         print('当前通道 =', one_chan)

#         # ========= 遍历40个目标频率 =========
#         for freq_idx in range(n_freqs):

#             target_freq = candidate_freqs[freq_idx]

#             # ========= 遍历4个block =========
#             for block_idx in range(n_blocks):

#                 trial_window = extract_trial_window(
#                     eeg = eeg,
#                     chan_list_py = chan_list_py,
#                     selected_chans = [one_chan],
#                     block_idx = block_idx,
#                     freq_idx = freq_idx,
#                     srate = info.srate,
#                     T_sec = T_sec,
#                     pre_stim_sec = 0.5,
#                     delta_sec = 0.13
#                 )

#                 one_channel_signal = trial_window[0, :]

#                 # ===== 1) 幅度谱 -> 窄带SNR =====
#                 freqs_amp, spectrum_amplitude = compute_fft_amplitude(
#                     one_channel_signal = one_channel_signal,
#                     srate = info.srate,
#                     use_bandpass = True,
#                     lowcut = 3.0,
#                     highcut = 100.0,
#                     filter_order = 4,
#                     use_hann_window = False,
#                     use_zero_padding = True,
#                     nfft = None
#                 )

#                 snr_vector_narrow = compute_snr_vector_narrow_beta(
#                     freqs = freqs_amp,
#                     spectrum_amplitude = spectrum_amplitude,
#                     candidate_freqs = candidate_freqs,
#                     neighbor_count = 5
#                 )

#                 max_idx_narrow = np.argmax(snr_vector_narrow)
#                 predicted_freq_narrow = candidate_freqs[max_idx_narrow]
#                 max_snr_narrow = snr_vector_narrow[max_idx_narrow]

#                 is_correct_narrow = (predicted_freq_narrow == target_freq)

#                 # ===== 2) 功率谱 -> 宽带SNR =====
#                 freqs_power, spectrum_power = compute_fft_power(
#                     one_channel_signal = one_channel_signal,
#                     srate = info.srate,
#                     use_bandpass = True,
#                     lowcut = 3.0,
#                     highcut = 100.0,
#                     filter_order = 4,
#                     use_hann_window = False,
#                     use_zero_padding = True,
#                     nfft = None
#                 )

#                 snr_vector_wide = compute_snr_vector_wide_beta(
#                     freqs = freqs_power,
#                     spectrum_power = spectrum_power,
#                     candidate_freqs = candidate_freqs,
#                     n_harmonics = 5
#                 )

#                 max_idx_wide = np.argmax(snr_vector_wide)
#                 predicted_freq_wide = candidate_freqs[max_idx_wide]
#                 max_snr_wide = snr_vector_wide[max_idx_wide]

#                 is_correct_wide = (predicted_freq_wide == target_freq)

#                 # ===== 统计 =====
#                 if is_correct_narrow == True:
#                     correct_count_narrow = correct_count_narrow + 1

#                 if is_correct_wide == True:
#                     correct_count_wide = correct_count_wide + 1

#                 total_count = total_count + 1

#                 # ===== 保存每条trial结果（可选） =====
#                 one_trial_result = {
#                     'subject_id': subject_id,
#                     'channel': one_chan,
#                     'block_idx': block_idx,
#                     'target_freq': target_freq,
#                     'predicted_freq_narrow': predicted_freq_narrow,
#                     'predicted_freq_wide': predicted_freq_wide,
#                     'max_snr_narrow': max_snr_narrow,
#                     'max_snr_wide': max_snr_wide,
#                     'is_correct_narrow': is_correct_narrow,
#                     'is_correct_wide': is_correct_wide
#                 }

#                 all_trial_results.append(one_trial_result)

#     # ========= 受试者汇总 =========
#     if total_count > 0:
#         accuracy_narrow_percent = 100.0 * correct_count_narrow / total_count
#         accuracy_wide_percent = 100.0 * correct_count_wide / total_count
#     else:
#         accuracy_narrow_percent = np.nan
#         accuracy_wide_percent = np.nan

#     one_summary = {
#         'subject_id': subject_id,
#         'n_selected_chans_used': valid_chan_count,
#         'total_trial_number': total_count,
#         'narrow_snr_accuracy_percent': accuracy_narrow_percent,
#         'wide_snr_accuracy_percent': accuracy_wide_percent
#     }

#     summary_results.append(one_summary)


# # ========= 生成并显示表格 =========
# summary_df = pd.DataFrame(summary_results)

# result_table = summary_df[
#     ['subject_id', 'narrow_snr_accuracy_percent', 'wide_snr_accuracy_percent']
# ].copy()

# result_table['narrow_snr_accuracy_percent'] = result_table['narrow_snr_accuracy_percent'].round(2)
# result_table['wide_snr_accuracy_percent'] = result_table['wide_snr_accuracy_percent'].round(2)

# result_table = result_table.set_index('subject_id')

# print('========================================')
# print('各受试者两种SNR总体正确率表（已合并所选通道）')
# display(result_table)

### 函数4 将宽带和窄带SNR转换为一维向量

In [ ]:
# ------------ 函数4 将窄带SNR和宽带SNR写成机器学习可用的特征向量 ------------

# 过程：
# ①输入长度为40的窄带SNR向量 snr_vector_narrow
# ②输入长度为40的宽带SNR向量 snr_vector_wide
# ③根据 feature_mode 选择使用窄带、宽带或两者拼接
# ④根据 concat_style 决定拼接顺序
# ⑤输出一维 feature_vector，后续可直接用于 LDA / SVM / RandomForest / KNN

def build_feature_vector_from_snr(
  snr_vector_narrow,
  snr_vector_wide,
  feature_mode = 'both',
  # feature_mode = 'narrow' → 只用窄带SNR，40维
  # feature_mode = 'wide' → 只用宽带SNR，40维
  # feature_mode = 'both' → 窄带SNR + 宽带SNR，80维
  concat_style = 'by_type'
  # concat_style = 'by_type'
  # 输出：[全部窄带40维, 全部宽带40维]
  # concat_style = 'by_frequency'
  # 输出：[f1窄带, f1宽带, f2窄带, f2宽带, ...]
):
  snr_vector_narrow_array = np.array(snr_vector_narrow, dtype=float)
  snr_vector_wide_array = np.array(snr_vector_wide, dtype=float)

  if len(snr_vector_narrow_array) != len(snr_vector_wide_array):
      raise ValueError('snr_vector_narrow 和 snr_vector_wide 的长度不一致。')

  if feature_mode == 'narrow':
      feature_vector = snr_vector_narrow_array.copy()

  elif feature_mode == 'wide':
      feature_vector = snr_vector_wide_array.copy()

  elif feature_mode == 'both':

      if concat_style == 'by_type':
          feature_vector = np.concatenate([
              snr_vector_narrow_array,
              snr_vector_wide_array
          ])

      elif concat_style == 'by_frequency':
          feature_list = []

          for i in range(len(snr_vector_narrow_array)):
              feature_list.append(snr_vector_narrow_array[i])
              feature_list.append(snr_vector_wide_array[i])

          feature_vector = np.array(feature_list, dtype=float)

      else:
          raise ValueError("concat_style 只能写 'by_type' 或 'by_frequency'。")

  else:
      raise ValueError("feature_mode 只能写 'narrow'、'wide' 或 'both'。")

  feature_vector = np.array(feature_vector, dtype=float)

  return feature_vector

### 完整单个通道，单个trail的特征提取函数

In [ ]:
def extract_feature_vector_one_trial_one_channel_beta(
    one_channel_signal,
    srate,
    candidate_freqs_for_feature,
    feature_mode='wide',
    concat_style='by_type',
    use_bandpass=True,
    lowcut=5.0,
    highcut=60.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0,
    narrow_neighbor_count=5,
    wide_n_harmonics=5
):
    freqs, spectrum_amplitude, spectrum_power = compute_fft_amplitude_and_power(
        one_channel_signal=one_channel_signal,
        srate=srate,
        use_bandpass=use_bandpass,
        lowcut=lowcut,
        highcut=highcut,
        filter_order=filter_order,
        use_hann_window=use_hann_window,
        use_zero_padding=use_zero_padding,
        nfft=nfft,
        pad_total_sec=pad_total_sec
    )

    snr_vector_narrow = compute_snr_vector_narrow_beta(
        freqs=freqs,
        spectrum_amplitude=spectrum_amplitude,
        candidate_freqs=candidate_freqs_for_feature,
        neighbor_count=narrow_neighbor_count
    )

    snr_vector_wide = compute_snr_vector_wide_beta(
        freqs=freqs,
        spectrum_power=spectrum_power,
        candidate_freqs=candidate_freqs_for_feature,
        n_harmonics=wide_n_harmonics
    )

    feature_vector = build_feature_vector_from_snr(
        snr_vector_narrow=snr_vector_narrow,
        snr_vector_wide=snr_vector_wide,
        feature_mode=feature_mode,
        concat_style=concat_style
    )

    return feature_vector

## 无早停功能下训练与测试模型

### 构建数据集

In [ ]:
# ------------ 小规模数据集构建函数 ------------

# 过程：
# ① 读取所选受试者的 .mat 文件
# ② 按 selected_freq_indices 选择要做分类的频率类别
# ③ 对每个受试者、每个频率、每个block、每个通道提取特征
# ④ 将一个trial中多个通道的特征拼接成一个最终特征向量
# ⑤ 输出 X、y、meta_df、info_dict，方便之后单独给SVM或其他模型训练和测试

import os
import pandas as pd
import scipy.io as sio

def build_small_scale_dataset_beta(
    base_folder,
    selected_subject_ids,
    selected_chans,
    selected_freq_indices,
    T_sec=None,
    window_sec=None,
    feature_mode='wide',
    concat_style='by_type',
    use_selected_freqs_as_feature_candidates=True,
    pre_stim_sec=0.5,
    delta_sec=None,
    latency_correction_sec=None,
    gaze_shift_sec=0.55,
    use_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0,
    narrow_neighbor_count=5,
    wide_n_harmonics=5
):
    if window_sec is None:
        if T_sec is None:
            raise ValueError('window_sec 和 T_sec 不能同时为空。')
        window_sec = T_sec

    if latency_correction_sec is None:
        if delta_sec is None:
            latency_correction_sec = 0.13
        else:
            latency_correction_sec = delta_sec

    if isinstance(selected_chans, str):
        selected_chans = [selected_chans]
    selected_chans = [str(ch) for ch in selected_chans]

    X_list = []
    y_list = []
    meta_rows = []

    full_candidate_freqs_ref = None
    selected_freq_values_ref = None
    feature_candidate_freqs_ref = None
    srate_ref = None

    for subject_id in selected_subject_ids:
        current_subject_file_path = os.path.join(base_folder, subject_id + '.mat')

        load_data = sio.loadmat(
            current_subject_file_path,
            struct_as_record=False,
            squeeze_me=True
        )

        data = load_data['data']
        eeg = data.EEG
        info = data.suppl_info

        srate = float(info.srate)
        if srate_ref is None:
            srate_ref = srate
        elif not np.isclose(srate_ref, srate):
            raise ValueError(f'不同受试者采样率不一致: 之前={srate_ref}, 当前 {subject_id}={srate}')

        chan_list = info.chan[:, -1]
        chan_list_py = [str(x) for x in chan_list]

        full_candidate_freqs = [float(x) for x in info.freqs]

        if full_candidate_freqs_ref is None:
            full_candidate_freqs_ref = full_candidate_freqs.copy()

        selected_freq_values = [full_candidate_freqs[idx] for idx in selected_freq_indices]

        if selected_freq_values_ref is None:
            selected_freq_values_ref = selected_freq_values.copy()

        if use_selected_freqs_as_feature_candidates:
            candidate_freqs_for_feature = selected_freq_values.copy()
        else:
            candidate_freqs_for_feature = full_candidate_freqs.copy()

        if feature_candidate_freqs_ref is None:
            feature_candidate_freqs_ref = candidate_freqs_for_feature.copy()

        chan_upper_list = [x.upper() for x in chan_list_py]
        for one_chan in selected_chans:
            if one_chan.upper() not in chan_upper_list:
                raise ValueError(f'受试者 {subject_id} 中不存在通道 {one_chan}。')

        n_blocks = eeg.shape[2]

        for local_label, freq_idx in enumerate(selected_freq_indices):
            target_freq = full_candidate_freqs[freq_idx]

            for block_idx in range(n_blocks):
                one_trial_feature_parts = []

                for one_chan in selected_chans:
                    trial_window = extract_trial_window(
                        eeg=eeg,
                        chan_list_py=chan_list_py,
                        selected_chans=[one_chan],
                        block_idx=block_idx,
                        freq_idx=freq_idx,
                        srate=srate,
                        window_sec=window_sec,
                        pre_stim_sec=pre_stim_sec,
                        latency_correction_sec=latency_correction_sec
                    )

                    one_channel_signal = np.asarray(trial_window[0, :], dtype=float)

                    feature_vector = extract_feature_vector_one_trial_one_channel_beta(
                        one_channel_signal=one_channel_signal,
                        srate=srate,
                        candidate_freqs_for_feature=candidate_freqs_for_feature,
                        feature_mode=feature_mode,
                        concat_style=concat_style,
                        use_bandpass=use_bandpass,
                        lowcut=lowcut,
                        highcut=highcut,
                        filter_order=filter_order,
                        use_hann_window=use_hann_window,
                        use_zero_padding=use_zero_padding,
                        nfft=nfft,
                        pad_total_sec=pad_total_sec,
                        narrow_neighbor_count=narrow_neighbor_count,
                        wide_n_harmonics=wide_n_harmonics
                    )

                    feature_vector = np.asarray(feature_vector, dtype=float).reshape(-1)
                    one_trial_feature_parts.append(feature_vector)

                trial_feature_vector = np.concatenate(one_trial_feature_parts, axis=0)

                X_list.append(trial_feature_vector)
                y_list.append(local_label)

                meta_rows.append({
                    'subject_id': str(subject_id),
                    'block_idx': int(block_idx),
                    'freq_idx_global': int(freq_idx),
                    'label_local': int(local_label),
                    'target_freq': float(target_freq),
                    'window_sec': float(window_sec),
                    'latency_correction_sec': float(latency_correction_sec)
                })

    X = np.asarray(X_list, dtype=float)
    y = np.asarray(y_list, dtype=int)
    meta_df = pd.DataFrame(meta_rows)

    decision_time_sec_fixed = float(window_sec + gaze_shift_sec)

    info_dict = {
        'full_candidate_freqs': full_candidate_freqs_ref,
        'selected_freq_values': selected_freq_values_ref,
        'feature_candidate_freqs': feature_candidate_freqs_ref,
        'srate': float(srate_ref),
        'n_channels': int(len(selected_chans)),
        'selected_chans': selected_chans,
        'n_classes': int(len(selected_freq_values_ref)),
        'window_sec': float(window_sec),
        'T_sec': float(window_sec),
        'latency_correction_sec': float(latency_correction_sec),
        'delta_sec': float(latency_correction_sec),
        'gaze_shift_sec': float(gaze_shift_sec),
        'decision_time_sec_fixed': decision_time_sec_fixed,
        'pre_stim_sec': float(pre_stim_sec),
        'feature_mode': str(feature_mode),
        'concat_style': str(concat_style),
        'use_selected_freqs_as_feature_candidates': bool(use_selected_freqs_as_feature_candidates),
        'use_bandpass': bool(use_bandpass),
        'lowcut': float(lowcut),
        'highcut': float(highcut),
        'filter_order': int(filter_order),
        'use_hann_window': bool(use_hann_window),
        'use_zero_padding': bool(use_zero_padding),
        'nfft': None if nfft is None else int(nfft),
        'pad_total_sec': float(pad_total_sec),
        'narrow_neighbor_count': int(narrow_neighbor_count),
        'wide_n_harmonics': int(wide_n_harmonics)
    }

    return X, y, meta_df, info_dict

In [ ]:
# ------------ CCA数据集构建函数 ------------

# 过程：
# ① 读取所选受试者的 .mat 文件
# ② 按 selected_freq_indices 选择需要参与分类的目标频率
# ③ 对每个受试者、每个频率、每个block提取多通道EEG时间窗
# ④ 将每个trial的多通道窗口保存到列表中，并记录对应标签与元信息
# ⑤ 输出 trial_list、y、meta_df、info_dict，供后续CCA测试函数使用

import os
import pandas as pd
import scipy.io as sio

def build_cca_dataset_beta(
    base_folder,
    selected_subject_ids,
    selected_chans,
    selected_freq_indices,
    T_sec=None,
    window_sec=None,
    pre_stim_sec=0.5,
    delta_sec=None,
    latency_correction_sec=None,
    gaze_shift_sec=0.55
):
    if window_sec is None:
        if T_sec is None:
            raise ValueError('window_sec 和 T_sec 不能同时为空。')
        window_sec = T_sec

    if latency_correction_sec is None:
        if delta_sec is None:
            latency_correction_sec = 0.13
        else:
            latency_correction_sec = delta_sec

    if isinstance(selected_chans, str):
        selected_chans = [selected_chans]
    selected_chans = [str(ch) for ch in selected_chans]

    trial_list = []
    y_list = []
    meta_rows = []

    full_candidate_freqs_ref = None
    selected_freq_values_ref = None
    srate_ref = None

    for subject_id in selected_subject_ids:
        current_subject_file_path = os.path.join(base_folder, subject_id + '.mat')

        load_data = sio.loadmat(
            current_subject_file_path,
            struct_as_record=False,
            squeeze_me=True
        )

        data = load_data['data']
        eeg = data.EEG
        info = data.suppl_info

        srate = float(info.srate)
        if srate_ref is None:
            srate_ref = srate
        elif not np.isclose(srate_ref, srate):
            raise ValueError(f'不同受试者采样率不一致: 之前={srate_ref}, 当前 {subject_id}={srate}')

        chan_list = info.chan[:, -1]
        chan_list_py = [str(x) for x in chan_list]

        full_candidate_freqs = [float(x) for x in info.freqs]

        if full_candidate_freqs_ref is None:
            full_candidate_freqs_ref = full_candidate_freqs.copy()

        selected_freq_values = [full_candidate_freqs[idx] for idx in selected_freq_indices]

        if selected_freq_values_ref is None:
            selected_freq_values_ref = selected_freq_values.copy()

        chan_upper_list = [x.upper() for x in chan_list_py]
        for one_chan in selected_chans:
            if one_chan.upper() not in chan_upper_list:
                raise ValueError(f'受试者 {subject_id} 中不存在通道 {one_chan}。')

        n_blocks = eeg.shape[2]

        for local_label, freq_idx in enumerate(selected_freq_indices):
            target_freq = full_candidate_freqs[freq_idx]

            for block_idx in range(n_blocks):
                trial_window = extract_trial_window(
                    eeg=eeg,
                    chan_list_py=chan_list_py,
                    selected_chans=selected_chans,
                    block_idx=block_idx,
                    freq_idx=freq_idx,
                    srate=srate,
                    window_sec=window_sec,
                    pre_stim_sec=pre_stim_sec,
                    latency_correction_sec=latency_correction_sec
                )

                trial_window = np.asarray(trial_window, dtype=float)

                trial_list.append(trial_window)
                y_list.append(local_label)

                meta_rows.append({
                    'subject_id': str(subject_id),
                    'block_idx': int(block_idx),
                    'freq_idx_global': int(freq_idx),
                    'label_local': int(local_label),
                    'target_freq': float(target_freq),
                    'window_sec': float(window_sec),
                    'latency_correction_sec': float(latency_correction_sec)
                })

    y = np.asarray(y_list, dtype=int)
    meta_df = pd.DataFrame(meta_rows)

    decision_time_sec_fixed = float(window_sec + gaze_shift_sec)

    info_dict = {
        'full_candidate_freqs': full_candidate_freqs_ref,
        'selected_freq_values': selected_freq_values_ref,
        'srate': float(srate_ref),
        'n_channels': int(len(selected_chans)),
        'selected_chans': selected_chans,
        'n_classes': int(len(selected_freq_values_ref)),
        'window_sec': float(window_sec),
        'T_sec': float(window_sec),
        'latency_correction_sec': float(latency_correction_sec),
        'delta_sec': float(latency_correction_sec),
        'gaze_shift_sec': float(gaze_shift_sec),
        'decision_time_sec_fixed': decision_time_sec_fixed,
        'pre_stim_sec': float(pre_stim_sec)
    }

    return trial_list, y, meta_df, info_dict

In [ ]:
# # SVM和LDA用于训练和测试的数据集
# X, y, meta_df, info_dict = build_small_scale_dataset_beta(
#     base_folder = '/content/drive/MyDrive/论文2025/BETA database',

#     selected_subject_ids = ['S23', 'S1', 'S26', 'S47'],
#     # selected_subject_ids = [f'S{i}' for i in range(1, 71)],

#     selected_chans = ['POZ', 'O1', 'OZ', 'O2'],
#     # selected_chans = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],

#     selected_freq_indices = [1, 3, 5, 7],
#     # selected_freq_indices = list(range(40)),

#     T_sec = 1.5,
#     feature_mode = 'wide',
#     concat_style = 'by_type',
#     use_selected_freqs_as_feature_candidates = True
# )

In [ ]:
# # CCA类使用的数据集构建器
# trial_list, y_cca, meta_df_cca, info_dict_cca = build_cca_dataset_beta(
#     base_folder = '/content/drive/MyDrive/论文2025/BETA database',

#     selected_subject_ids = ['S23', 'S1', 'S26', 'S47'],
#     # selected_subject_ids = [f'S{i}' for i in range(1, 71)],

#     selected_chans = ['POZ', 'O1', 'OZ', 'O2'],
#     # selected_chans = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],

#     selected_freq_indices = [1, 3, 5, 7],
#     # selected_freq_indices = list(range(40)),

#     T_sec = 1.5
# )

In [ ]:
base_folder = '/Users/mikuru/SSVEP/DATA/BETA DATA'

selected_subject_ids = [
    'S16', 'S17', 'S18', 'S19', 'S20',
    'S21', 'S22', 'S23', 'S24', 'S25', 'S26', 'S27', 'S28', 'S29', 'S30',
    'S31', 'S32', 'S33', 'S34', 'S35', 'S36', 'S37', 'S38', 'S39', 'S40',
    'S41', 'S42', 'S43', 'S44', 'S45', 'S46', 'S47', 'S48', 'S49', 'S50',
    'S51', 'S52', 'S53', 'S54', 'S55', 'S56', 'S57', 'S58', 'S59', 'S60',
    'S61', 'S62', 'S63', 'S64', 'S65', 'S66', 'S67', 'S68', 'S69', 'S70'
]
selected_chans = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
selected_freq_indices = list(range(40))

window_sec = 2.0
latency_correction_sec = 0.13
gaze_shift_sec = 0.55

X, y, meta_df, info_dict = build_small_scale_dataset_beta(
    base_folder = base_folder,
    selected_subject_ids = selected_subject_ids,
    selected_chans = selected_chans,
    selected_freq_indices = selected_freq_indices,
    window_sec = window_sec,
    latency_correction_sec = latency_correction_sec,
    gaze_shift_sec = gaze_shift_sec,
    feature_mode = 'wide',
    concat_style = 'by_type',
    use_selected_freqs_as_feature_candidates = True,
    use_bandpass = True,
    lowcut = 6.0,
    highcut = 80.0,
    filter_order = 4,
    use_hann_window = False,
    use_zero_padding = True,
    nfft = None,
    pad_total_sec = 5.0,
    narrow_neighbor_count = 5,
    wide_n_harmonics = 5
)

trial_list, y_cca, meta_df_cca, info_dict_cca = build_cca_dataset_beta(
    base_folder = base_folder,
    selected_subject_ids = selected_subject_ids,
    selected_chans = selected_chans,
    selected_freq_indices = selected_freq_indices,
    window_sec = window_sec,
    latency_correction_sec = latency_correction_sec,
    gaze_shift_sec = gaze_shift_sec
)

### 公共函数：ITR、softmax、margin、稳定判定

In [ ]:
def compute_itr_bits_per_min(
    n_classes,
    accuracy,
    decision_time_sec
):
    N = int(n_classes)
    P = float(accuracy)
    T = float(decision_time_sec)

    if N <= 1 or T <= 0:
        return np.nan

    chance_level = 1.0 / N

    # ITR归零规则：当准确率低于或等于随机水平时，不给予由极短决策时间带来的ITR奖励。
    if P <= chance_level:
        return 0.0

    if P >= 1.0:
        bits_per_trial = np.log2(N)
    else:
        bits_per_trial = (
            np.log2(N)
            + P * np.log2(P)
            + (1.0 - P) * np.log2((1.0 - P) / (N - 1))
        )

    return float(60.0 * bits_per_trial / T)

def softmax_score_vector_beta(
    score_vector,
    temperature = 1.0
):
    score_array = np.array(score_vector, dtype=float)
    temperature = float(temperature)

    if temperature <= 0:
        raise ValueError('temperature 必须大于 0。')

    shifted = score_array - np.max(score_array)
    exp_value = np.exp(shifted / temperature)
    prob = exp_value / np.sum(exp_value)

    return prob


def compute_margin_from_prob_vector_beta(
    prob_vector
):
    prob_array = np.array(prob_vector, dtype=float)
    sorted_prob = np.sort(prob_array)[::-1]

    if len(sorted_prob) == 1:
        return 1.0

    return float(sorted_prob[0] - sorted_prob[1])


def check_recent_predictions_stable_beta(
    pred_history,
    min_stable_windows = 1
):
    K = int(min_stable_windows)

    if K <= 1:
        return True

    if len(pred_history) < K:
        return False

    recent_pred = pred_history[-K:]

    return len(set(recent_pred)) == 1

In [ ]:
# 给无早停结果加上速度和ITR显示

def _should_run_subjectwise_block_cv_beta(meta_df, split_mode):
    if str(split_mode) != 'block':
        return False
    if meta_df is None or 'subject_id' not in meta_df.columns:
        return False
    subject_ids = pd.Series(meta_df['subject_id']).astype(str).unique().tolist()
    return len(subject_ids) > 1


def _aggregate_fixed_window_subject_results_beta(
    subject_result_list,
    model_name,
    split_mode,
    meta_df,
    info_dict
):
    subject_summary_df = pd.concat(
        [one_result['summary_df'].copy() for one_result in subject_result_list],
        axis=0,
        ignore_index=True
    )

    fold_results_df = pd.concat(
        [one_result['fold_results_df'].copy() for one_result in subject_result_list],
        axis=0,
        ignore_index=True
    )

    all_y_true = np.concatenate(
        [np.asarray(one_result['all_y_true'], dtype=int) for one_result in subject_result_list],
        axis=0
    )
    all_y_pred = np.concatenate(
        [np.asarray(one_result['all_y_pred'], dtype=int) for one_result in subject_result_list],
        axis=0
    )

    all_score_vectors = []
    all_prob_vectors = []
    for one_result in subject_result_list:
        if 'all_score_vectors' in one_result and one_result['all_score_vectors'] is not None:
            all_score_vectors.extend(one_result['all_score_vectors'])
        if 'all_prob_vectors' in one_result and one_result['all_prob_vectors'] is not None:
            all_prob_vectors.extend(one_result['all_prob_vectors'])

    first_summary_row = subject_summary_df.iloc[0].to_dict()

    summary_row = dict(first_summary_row)
    summary_row['model_name'] = str(model_name)
    summary_row['split_mode'] = str(split_mode)
    summary_row['overall_accuracy'] = float(subject_summary_df['overall_accuracy'].mean())
    summary_row['mean_fold_accuracy'] = float(subject_summary_df['mean_fold_accuracy'].mean())
    summary_row['std_fold_accuracy'] = float(subject_summary_df['mean_fold_accuracy'].std(ddof=0))
    summary_row['n_samples'] = int(len(all_y_true))

    if 'mean_recognition_runtime_sec' in subject_summary_df.columns:
        summary_row['mean_recognition_runtime_sec'] = float(subject_summary_df['mean_recognition_runtime_sec'].mean())
        summary_row['std_recognition_runtime_sec'] = float(subject_summary_df['mean_recognition_runtime_sec'].std(ddof=0))
    if 'mean_decision_time_with_runtime_sec' in subject_summary_df.columns:
        summary_row['mean_decision_time_with_runtime_sec'] = float(subject_summary_df['mean_decision_time_with_runtime_sec'].mean())
        summary_row['std_decision_time_with_runtime_sec'] = float(subject_summary_df['mean_decision_time_with_runtime_sec'].std(ddof=0))

    if 'n_blocks' in summary_row:
        summary_row['n_blocks'] = int(meta_df['block_idx'].nunique())

    summary_df = pd.DataFrame([summary_row])

    return {
        'summary_df': summary_df,
        'fold_results_df': fold_results_df,
        'all_y_true': all_y_true,
        'all_y_pred': all_y_pred,
        'all_score_vectors': all_score_vectors,
        'all_prob_vectors': all_prob_vectors,
        'subject_summary_df': subject_summary_df
    }


def append_fixed_window_speed_itr_to_result_beta(
    result_dict,
    info_dict
):
    summary_df = result_dict['summary_df'].copy()

    if 'fold_results_df' in result_dict and result_dict['fold_results_df'] is not None:
        fold_results_df = result_dict['fold_results_df'].copy()
    else:
        fold_results_df = pd.DataFrame()

    subject_summary_df = result_dict.get('subject_summary_df', None)
    if subject_summary_df is not None:
        subject_summary_df = subject_summary_df.copy()

    window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
    gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
    decision_time_sec = float(window_sec + gaze_shift_sec)

    if 'n_classes' in summary_df.columns:
        n_classes = int(summary_df.loc[0, 'n_classes'])
    else:
        n_classes = int(info_dict['n_classes'])

    summary_df['window_sec'] = float(window_sec)
    summary_df['gaze_shift_sec'] = float(gaze_shift_sec)
    summary_df['mean_decision_time_sec'] = float(decision_time_sec)
    if len(fold_results_df) > 0 and 'mean_recognition_runtime_sec' in fold_results_df.columns:
        mean_recognition_runtime_sec = float(fold_results_df['mean_recognition_runtime_sec'].mean())
    elif 'mean_recognition_runtime_sec' in summary_df.columns:
        mean_recognition_runtime_sec = float(summary_df.loc[0, 'mean_recognition_runtime_sec'])
    else:
        mean_recognition_runtime_sec = 0.0
    summary_df['mean_recognition_runtime_sec'] = float(mean_recognition_runtime_sec)
    summary_df['mean_decision_time_with_runtime_sec'] = float(decision_time_sec + mean_recognition_runtime_sec)
    summary_df['decision_rate_trials_per_min'] = float(60.0 / decision_time_sec)

    if len(fold_results_df) > 0 and 'fold_accuracy' in fold_results_df.columns:
        fold_itr_list = []
        fold_itr_with_runtime_list = []
        for fold_acc in fold_results_df['fold_accuracy'].tolist():
            fold_itr = compute_itr_bits_per_min(
                n_classes=n_classes,
                accuracy=float(fold_acc),
                decision_time_sec=decision_time_sec
            )
            fold_itr_list.append(float(fold_itr))

            if 'mean_recognition_runtime_sec' in fold_results_df.columns:
                current_runtime = float(fold_results_df.loc[len(fold_itr_list) - 1, 'mean_recognition_runtime_sec'])
            else:
                current_runtime = 0.0
            fold_itr_with_runtime = compute_itr_bits_per_min(
                n_classes=n_classes,
                accuracy=float(fold_acc),
                decision_time_sec=float(decision_time_sec + current_runtime)
            )
            fold_itr_with_runtime_list.append(float(fold_itr_with_runtime))

        fold_results_df['window_sec'] = float(window_sec)
        fold_results_df['gaze_shift_sec'] = float(gaze_shift_sec)
        fold_results_df['decision_time_sec'] = float(decision_time_sec)
        if 'mean_recognition_runtime_sec' not in fold_results_df.columns:
            fold_results_df['mean_recognition_runtime_sec'] = 0.0
        fold_results_df['decision_time_with_runtime_sec'] = (
            fold_results_df['decision_time_sec'].astype(float)
            + fold_results_df['mean_recognition_runtime_sec'].astype(float)
        )
        fold_results_df['fold_itr_bits_per_min'] = np.asarray(fold_itr_list, dtype=float)
        fold_results_df['fold_itr_with_runtime_bits_per_min'] = np.asarray(fold_itr_with_runtime_list, dtype=float)

    if subject_summary_df is not None and len(subject_summary_df) > 0:
        subject_itr_list = []
        for subject_acc in subject_summary_df['overall_accuracy'].tolist():
            subject_itr = compute_itr_bits_per_min(
                n_classes=n_classes,
                accuracy=float(subject_acc),
                decision_time_sec=decision_time_sec
            )
            subject_itr_list.append(float(subject_itr))

        summary_df['mean_fold_itr_bits_per_min'] = float(np.mean(subject_itr_list))
        summary_df['std_fold_itr_bits_per_min'] = float(np.std(subject_itr_list, ddof=0))
        summary_df['itr_bits_per_min'] = float(np.mean(subject_itr_list))

        subject_itr_with_runtime_list = []
        for _, subject_row in subject_summary_df.iterrows():
            subject_acc = float(subject_row['overall_accuracy'])
            subject_runtime = float(subject_row['mean_recognition_runtime_sec']) if 'mean_recognition_runtime_sec' in subject_summary_df.columns else 0.0
            subject_time_with_runtime = float(decision_time_sec + subject_runtime)
            subject_itr_with_runtime = compute_itr_bits_per_min(
                n_classes=n_classes,
                accuracy=subject_acc,
                decision_time_sec=subject_time_with_runtime
            )
            subject_itr_with_runtime_list.append(float(subject_itr_with_runtime))
        summary_df['itr_with_runtime_bits_per_min'] = float(np.mean(subject_itr_with_runtime_list))
        summary_df['std_fold_itr_with_runtime_bits_per_min'] = float(np.std(subject_itr_with_runtime_list, ddof=0))
    elif len(fold_results_df) > 0:
        summary_df['mean_fold_itr_bits_per_min'] = float(fold_results_df['fold_itr_bits_per_min'].mean())
        summary_df['std_fold_itr_bits_per_min'] = float(fold_results_df['fold_itr_bits_per_min'].std(ddof=0))
        summary_df['itr_bits_per_min'] = float(fold_results_df['fold_itr_bits_per_min'].mean())
        summary_df['itr_with_runtime_bits_per_min'] = float(fold_results_df['fold_itr_with_runtime_bits_per_min'].mean()) if 'fold_itr_with_runtime_bits_per_min' in fold_results_df.columns else float(fold_results_df['fold_itr_bits_per_min'].mean())
        summary_df['std_fold_itr_with_runtime_bits_per_min'] = float(fold_results_df['fold_itr_with_runtime_bits_per_min'].std(ddof=0)) if 'fold_itr_with_runtime_bits_per_min' in fold_results_df.columns else float(fold_results_df['fold_itr_bits_per_min'].std(ddof=0))
    else:
        summary_df['mean_fold_itr_bits_per_min'] = np.nan
        summary_df['std_fold_itr_bits_per_min'] = np.nan
        summary_df['itr_with_runtime_bits_per_min'] = np.nan
        summary_df['std_fold_itr_with_runtime_bits_per_min'] = np.nan

    overall_accuracy = float(summary_df.loc[0, 'overall_accuracy'])
    overall_itr = compute_itr_bits_per_min(
        n_classes=n_classes,
        accuracy=overall_accuracy,
        decision_time_sec=decision_time_sec
    )
    summary_df['overall_accuracy_based_itr_bits_per_min'] = float(overall_itr)
    overall_itr_with_runtime = compute_itr_bits_per_min(
        n_classes=n_classes,
        accuracy=overall_accuracy,
        decision_time_sec=float(decision_time_sec + summary_df.loc[0, 'mean_recognition_runtime_sec'])
    )
    summary_df['overall_accuracy_based_itr_with_runtime_bits_per_min'] = float(overall_itr_with_runtime)

    result_dict['summary_df'] = summary_df
    if len(fold_results_df) > 0:
        result_dict['fold_results_df'] = fold_results_df
    if subject_summary_df is not None:
        result_dict['subject_summary_df'] = subject_summary_df

    return result_dict

### SVM参数定义与测试训练
### Definición de parámetros SVM y entrenamiento/prueba

In [ ]:
# ------------ SVM模型参数定义函数 ------------

# 过程：
# ① 定义SVM的基础参数，例如核函数、C、gamma
# ② 使用StandardScaler对特征做标准化
# ③ 早停阶段打开 probability，使用前两名预测概率差作为置信度
# ④ 输出可直接用于训练和预测的SVM模型

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


def define_svm_model_beta(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=False,
    random_state=42
):
    svm_model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(
            C=C,
            kernel=kernel,
            gamma=gamma,
            probability=probability,
            random_state=random_state
        ))
    ])

    return svm_model

In [ ]:
# ------------ SVM统一训练与测试函数 ------------

# 过程：
# ① 输入已经构造好的特征矩阵 X、标签 y、样本信息 meta_df、频率信息 info_dict
# ② 根据 split_mode 选择按 block 或按 subject 做切分
# ③ 用定义好的SVM模型进行训练和测试，并统计每一折的准确率
# ④ 汇总总体准确率、平均折准确率、标准差、样本数、特征维度等整体结果

def train_test_svm_beta(
  X,
  y,
  meta_df,
  info_dict,
  split_mode = 'block',
  C = 1.0,
  kernel = 'rbf',
  gamma = 'scale',
  probability = False,
  random_state = 42
):
  if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
      subject_result_list = []

      subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

      for one_subject_id in subject_id_list:
          subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)

          one_subject_result = train_test_svm_beta(
              X = X[subject_mask],
              y = y[subject_mask],
              meta_df = meta_df.loc[subject_mask].reset_index(drop = True),
              info_dict = info_dict,
              split_mode = split_mode,
              C = C,
              kernel = kernel,
              gamma = gamma,
              probability = probability,
              random_state = random_state
          )
          subject_result_list.append(one_subject_result)

      return _aggregate_fixed_window_subject_results_beta(
          subject_result_list = subject_result_list,
          model_name = 'SVM',
          split_mode = split_mode,
          meta_df = meta_df,
          info_dict = info_dict
      )

  svm_model = define_svm_model_beta(
      C = C,
      kernel = kernel,
      gamma = gamma,
      probability = probability,
      random_state = random_state
  )

  if split_mode == 'block':
      split_key = 'block_idx'
      split_values = sorted(meta_df['block_idx'].unique().tolist())
      fold_name = 'test_block'
  elif split_mode == 'subject':
      split_key = 'subject_id'
      split_values = sorted(meta_df['subject_id'].unique().tolist())
      fold_name = 'test_subject_id'
  else:
      raise ValueError("split_mode 只能写 'block' 或 'subject'。")

  window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
  gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
  decision_time_sec = float(window_sec + gaze_shift_sec)

  fold_result_rows = []

  all_y_true = []
  all_y_pred = []
  all_score_vectors = []
  all_prob_vectors = []

  for one_split_value in split_values:
      train_mask = meta_df[split_key].values != one_split_value
      test_mask = meta_df[split_key].values == one_split_value

      X_train = X[train_mask]
      y_train = y[train_mask]

      X_test = X[test_mask]
      y_test = y[test_mask]

      current_model = clone(svm_model)
      current_model.fit(X_train, y_train)

      _runtime_start = time.perf_counter()
      y_pred = current_model.predict(X_test)

      if hasattr(current_model, 'decision_function'):
          decision_matrix = current_model.decision_function(X_test)
          decision_matrix = np.asarray(decision_matrix, dtype=float)
          if decision_matrix.ndim == 1:
              decision_matrix = decision_matrix.reshape(-1, 1)
          score_matrix = decision_matrix
          prob_matrix = None
      else:
          score_matrix = None
          prob_matrix = None

      _runtime_elapsed_sec = float(time.perf_counter() - _runtime_start)
      mean_recognition_runtime_sec = _runtime_elapsed_sec / max(len(X_test), 1)

      fold_acc = accuracy_score(y_test, y_pred)

      fold_result_rows.append({
          'model_name': 'SVM',
          fold_name: one_split_value,
          'fold_accuracy': float(fold_acc),
          'n_train': int(len(X_train)),
          'n_test': int(len(X_test)),
          'window_sec': float(window_sec),
          'decision_time_sec': float(decision_time_sec),
          'mean_recognition_runtime_sec': float(mean_recognition_runtime_sec),
          'decision_time_with_runtime_sec': float(decision_time_sec + mean_recognition_runtime_sec)
      })

      all_y_true.extend(y_test.tolist())
      all_y_pred.extend(y_pred.tolist())

      if score_matrix is not None:
          for row in score_matrix:
              all_score_vectors.append(np.asarray(row, dtype=float))

      if prob_matrix is not None:
          for row in prob_matrix:
              all_prob_vectors.append(np.asarray(row, dtype=float))

  all_y_true = np.array(all_y_true, dtype=int)
  all_y_pred = np.array(all_y_pred, dtype=int)

  overall_accuracy = accuracy_score(all_y_true, all_y_pred)

  fold_results_df = pd.DataFrame(fold_result_rows)

  summary_row = {
      'model_name': 'SVM',
      'split_mode': split_mode,
      'overall_accuracy': float(overall_accuracy),
      'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
      'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof = 0)),
      'n_samples': int(len(all_y_true)),
      'n_features': int(X.shape[1]),
      'n_classes': int(len(info_dict['selected_freq_values'])),
      'window_sec': float(window_sec),
      'gaze_shift_sec': float(gaze_shift_sec),
      'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()),
      'mean_decision_time_with_runtime_sec': float(fold_results_df['decision_time_with_runtime_sec'].mean()),
      'C': float(C),
      'kernel': str(kernel),
      'gamma': str(gamma),
      'probability': bool(probability)
  }

  if split_mode == 'subject':
      summary_row['n_subjects'] = int(len(split_values))
  elif split_mode == 'block':
      summary_row['n_blocks'] = int(len(split_values))

  summary_df = pd.DataFrame([summary_row])

  result_dict = {
      'summary_df': summary_df,
      'fold_results_df': fold_results_df,
      'all_y_true': all_y_true,
      'all_y_pred': all_y_pred,
      'all_score_vectors': all_score_vectors,
      'all_prob_vectors': all_prob_vectors
  }

  return result_dict

In [ ]:
# SVM结果展示函数
def print_svm_result_with_notes_beta(
  svm_result
):
#   print('=' * 80)
#   print('SVM summary_df')
  print(svm_result['summary_df'])
#   print('=' * 80)

In [ ]:
# 非跨受试者
# lda_result_block = train_test_lda_beta(
#     X = X,
#     y = y,
#     meta_df = meta_df,
#     info_dict = info_dict,
#     split_mode = 'block',
#     solver = 'lsqr',
#     shrinkage = 'auto'
# )

# lda_result_block = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = lda_result_block,
#     info_dict = info_dict
# )

# print_lda_result_with_notes_beta(
#     lda_result = lda_result_block
# )

# lda_result_block['summary_df']

In [ ]:
# # 跨受试者调用例
# svm_result_subject = train_test_svm_beta(
#     X = X,
#     y = y,
#     meta_df = meta_df,
#     info_dict = info_dict,
#     split_mode = 'subject',
#     C = 1.0,
#     kernel = 'rbf',
#     gamma = 'scale',
#     probability = False,
#     random_state = 42
# )

# svm_result_subject = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = svm_result_subject,
#     info_dict = info_dict
# )

# print_svm_result_with_notes_beta(
#     svm_result = svm_result_subject
# )

# svm_result_subject['summary_df']

In [ ]:
# svm_result_block = train_test_svm_beta(
#     X = X,
#     y = y,
#     meta_df = meta_df,
#     info_dict = info_dict,
#     split_mode = 'block',
#     C = 1.0,
#     kernel = 'rbf',
#     gamma = 'scale',
#     probability = False,
#     random_state = 42
# )

# svm_result_block = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = svm_result_block,
#     info_dict = info_dict
# )

# svm_result_block['summary_df']

### LDA参数定义与测试训练

In [ ]:
# ------------ LDA模型参数定义函数 ------------

# 过程：
# ① 定义LDA的基础参数，例如 solver 和 shrinkage
# ② 使用StandardScaler对特征做标准化
# ③ 将标准化和LDA放入同一个Pipeline中
# ④ 输出可直接用于训练和预测的LDA模型

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score


def define_lda_model_beta(
  solver = 'lsqr',
  shrinkage = 'auto'
):
  lda_model = Pipeline([
      ('scaler', StandardScaler()),
      ('clf', LinearDiscriminantAnalysis(
          solver = solver,
          shrinkage = shrinkage
      ))
  ])

  return lda_model

In [ ]:
# ------------ LDA统一训练与测试函数 ------------

# 过程：
# ① 输入已经构造好的特征矩阵 X、标签 y、样本信息 meta_df、频率信息 info_dict
# ② 根据 split_mode 选择按 block 或按 subject 做切分
# ③ 用定义好的LDA模型进行训练和测试，并统计每一折的准确率
# ④ 汇总总体准确率、平均折准确率、标准差、样本数、特征维度等整体结果

def train_test_lda_beta(
  X,
  y,
  meta_df,
  info_dict,
  split_mode = 'block',
  solver = 'lsqr',
  shrinkage = 'auto'
):
  if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
      subject_result_list = []

      subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

      for one_subject_id in subject_id_list:
          subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)

          one_subject_result = train_test_lda_beta(
              X = X[subject_mask],
              y = y[subject_mask],
              meta_df = meta_df.loc[subject_mask].reset_index(drop = True),
              info_dict = info_dict,
              split_mode = split_mode,
              solver = solver,
              shrinkage = shrinkage
          )
          subject_result_list.append(one_subject_result)

      return _aggregate_fixed_window_subject_results_beta(
          subject_result_list = subject_result_list,
          model_name = 'LDA',
          split_mode = split_mode,
          meta_df = meta_df,
          info_dict = info_dict
      )

  lda_model = define_lda_model_beta(
      solver = solver,
      shrinkage = shrinkage
  )

  if split_mode == 'block':
      split_key = 'block_idx'
      split_values = sorted(meta_df['block_idx'].unique().tolist())
      fold_name = 'test_block'
  elif split_mode == 'subject':
      split_key = 'subject_id'
      split_values = sorted(meta_df['subject_id'].unique().tolist())
      fold_name = 'test_subject_id'
  else:
      raise ValueError("split_mode 只能写 'block' 或 'subject'。")

  window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
  gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
  decision_time_sec = float(window_sec + gaze_shift_sec)

  fold_result_rows = []

  all_y_true = []
  all_y_pred = []
  all_score_vectors = []
  all_prob_vectors = []

  for one_split_value in split_values:
      train_mask = meta_df[split_key].values != one_split_value
      test_mask = meta_df[split_key].values == one_split_value

      X_train = X[train_mask]
      y_train = y[train_mask]

      X_test = X[test_mask]
      y_test = y[test_mask]

      current_model = clone(lda_model)
      current_model.fit(X_train, y_train)

      _runtime_start = time.perf_counter()
      y_pred = current_model.predict(X_test)

      if hasattr(current_model, 'decision_function'):
          decision_matrix = current_model.decision_function(X_test)
          decision_matrix = np.asarray(decision_matrix, dtype=float)
          if decision_matrix.ndim == 1:
              decision_matrix = decision_matrix.reshape(-1, 1)
          score_matrix = decision_matrix
          prob_matrix = None
      else:
          score_matrix = None
          prob_matrix = None

      _runtime_elapsed_sec = float(time.perf_counter() - _runtime_start)
      mean_recognition_runtime_sec = _runtime_elapsed_sec / max(len(X_test), 1)

      fold_acc = accuracy_score(y_test, y_pred)

      fold_result_rows.append({
          'model_name': 'LDA',
          fold_name: one_split_value,
          'fold_accuracy': float(fold_acc),
          'n_train': int(len(X_train)),
          'n_test': int(len(X_test)),
          'window_sec': float(window_sec),
          'decision_time_sec': float(decision_time_sec),
          'mean_recognition_runtime_sec': float(mean_recognition_runtime_sec),
          'decision_time_with_runtime_sec': float(decision_time_sec + mean_recognition_runtime_sec)
      })

      all_y_true.extend(y_test.tolist())
      all_y_pred.extend(y_pred.tolist())

      if score_matrix is not None:
          for row in score_matrix:
              all_score_vectors.append(np.asarray(row, dtype=float))

      if prob_matrix is not None:
          for row in prob_matrix:
              all_prob_vectors.append(np.asarray(row, dtype=float))

  all_y_true = np.array(all_y_true, dtype=int)
  all_y_pred = np.array(all_y_pred, dtype=int)

  overall_accuracy = accuracy_score(all_y_true, all_y_pred)

  fold_results_df = pd.DataFrame(fold_result_rows)

  summary_row = {
      'model_name': 'LDA',
      'split_mode': split_mode,
      'overall_accuracy': float(overall_accuracy),
      'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
      'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof = 0)),
      'n_samples': int(len(all_y_true)),
      'n_features': int(X.shape[1]),
      'n_classes': int(len(info_dict['selected_freq_values'])),
      'window_sec': float(window_sec),
      'gaze_shift_sec': float(gaze_shift_sec),
      'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()),
      'mean_decision_time_with_runtime_sec': float(fold_results_df['decision_time_with_runtime_sec'].mean()),
      'solver': str(solver),
      'shrinkage': str(shrinkage)
  }

  if split_mode == 'subject':
      summary_row['n_subjects'] = int(len(split_values))
  elif split_mode == 'block':
      summary_row['n_blocks'] = int(len(split_values))

  summary_df = pd.DataFrame([summary_row])

  result_dict = {
      'summary_df': summary_df,
      'fold_results_df': fold_results_df,
      'all_y_true': all_y_true,
      'all_y_pred': all_y_pred,
      'all_score_vectors': all_score_vectors,
      'all_prob_vectors': all_prob_vectors
  }

  return result_dict

In [ ]:
# LDA结果展示函数
def print_lda_result_with_notes_beta(
  lda_result
):
#   print('=' * 80)
#   print('LDA summary_df')
  print(lda_result['summary_df'])
#   print('=' * 80)

In [ ]:
# # 非跨受试者
# lda_result_block = train_test_lda_beta(
#     X = X,
#     y = y,
#     meta_df = meta_df,
#     info_dict = info_dict,
#     split_mode = 'block',
#     solver = 'lsqr',
#     shrinkage = 'auto'
# )

# print_lda_result_with_notes_beta(
#     lda_result = lda_result_block
# )

In [ ]:
# # 跨受试者
# lda_result_subject = train_test_lda_beta(
#     X = X,
#     y = y,
#     meta_df = meta_df,
#     info_dict = info_dict,
#     split_mode = 'subject',
#     solver = 'lsqr',
#     shrinkage = 'auto'
# )

# lda_result_subject = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = lda_result_subject,
#     info_dict = info_dict
# )

# print_lda_result_with_notes_beta(
#     lda_result = lda_result_subject
# )

# lda_result_subject['summary_df']

In [ ]:
# lda_result_block = train_test_lda_beta(
#     X = X,
#     y = y,
#     meta_df = meta_df,
#     info_dict = info_dict,
#     split_mode = 'block',
#     solver = 'lsqr',
#     shrinkage = 'auto'
# )

# lda_result_block = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = lda_result_block,
#     info_dict = info_dict
# )

# lda_result_block['summary_df']

### CCA参数定义与测试

In [ ]:
# ------------ CCA模型参数定义函数 ------------

# 过程：
# ① 定义CCA识别时需要的核心参数，例如谐波数 n_harmonics
# ② 将这些参数整理为一个字典，方便后续统一调用
# ③ 该模型不需要训练数据拟合，因此这里只保存参数
# ④ 输出可用于后续CCA识别的参数字典

def define_cca_model_beta(
  n_harmonics = 5
):
  cca_model_dict = {
      'model_name': 'CCA',
      'n_harmonics': int(n_harmonics)
  }

  return cca_model_dict

In [ ]:
# ------------ CCA参考信号生成与单trial预测函数 ------------

# 过程：
# ① 根据候选频率、谐波数、采样率和时间点数生成每个频率的参考信号
# ② 对一个trial的多通道EEG窗口和每个候选频率参考信号分别做CCA
# ③ 取相关系数最大的候选频率作为预测类别
# ④ 输出预测标签以及各候选频率对应的相关系数分数

import numpy as np
from sklearn.cross_decomposition import CCA

def generate_cca_reference_signals_beta(
  candidate_freqs,
  srate,
  n_samples,
  n_harmonics = 5
):
  t = np.arange(n_samples, dtype=float) / float(srate)

  reference_signal_list = []

  for freq in candidate_freqs:
      ref_components = []

      for h in range(1, n_harmonics + 1):
          ref_components.append(np.sin(2 * np.pi * h * freq * t))
          ref_components.append(np.cos(2 * np.pi * h * freq * t))

      ref_matrix = np.array(ref_components, dtype=float)
      reference_signal_list.append(ref_matrix)

  return reference_signal_list
# 已清理：旧版 compute_cca_score_one_trial_beta 已删除，统一使用后面保留的最终版本。

def predict_one_trial_cca_beta(
  trial_eeg,
  reference_signal_list
):
  cca_score_list = []

  for ref_signal in reference_signal_list:
      one_score = compute_cca_score_one_trial_beta(
          trial_eeg = trial_eeg,
          ref_signal = ref_signal
      )
      cca_score_list.append(one_score)

  cca_score_array = np.array(cca_score_list, dtype=float)

  pred_label = int(np.argmax(cca_score_array))

  return pred_label, cca_score_array


In [ ]:
# ------------ CCA统一测试函数 ------------

# 过程：
# ① 输入已经构造好的trial_list、标签y、样本信息meta_df、频率信息info_dict
# ② 根据 split_mode 选择按 block 或按 subject 做切分
# ③ 由于标准CCA为training-free方法，因此每一折不进行模型拟合，只在测试集上直接识别
# ④ 汇总总体准确率、平均折准确率、标准差、样本数、类别数等整体结果

from sklearn.metrics import accuracy_score

def train_test_cca_beta(
  trial_list,
  y,
  meta_df,
  info_dict,
  split_mode = 'block',
  n_harmonics = 5,
  apply_bandpass = True,
  lowcut = 6.0,
  highcut = 80.0,
  filter_order = 4
):
  if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
      subject_result_list = []

      subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

      for one_subject_id in subject_id_list:
          subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)
          subject_indices = np.where(subject_mask)[0]

          one_subject_result = train_test_cca_beta(
              trial_list = [trial_list[idx] for idx in subject_indices],
              y = y[subject_indices],
              meta_df = meta_df.loc[subject_mask].reset_index(drop = True),
              info_dict = info_dict,
              split_mode = split_mode,
              n_harmonics = n_harmonics,
              apply_bandpass = apply_bandpass,
              lowcut = lowcut,
              highcut = highcut,
              filter_order = filter_order
          )
          subject_result_list.append(one_subject_result)

      return _aggregate_fixed_window_subject_results_beta(
          subject_result_list = subject_result_list,
          model_name = 'CCA',
          split_mode = split_mode,
          meta_df = meta_df,
          info_dict = info_dict
      )

  if split_mode == 'block':
      split_key = 'block_idx'
      split_values = sorted(meta_df['block_idx'].unique().tolist())
      fold_name = 'test_block'
  elif split_mode == 'subject':
      split_key = 'subject_id'
      split_values = sorted(meta_df['subject_id'].unique().tolist())
      fold_name = 'test_subject_id'
  else:
      raise ValueError("split_mode 只能写 'block' 或 'subject'。")

  selected_freq_values = info_dict['selected_freq_values']
  srate = info_dict['srate']
  n_samples = int(trial_list[0].shape[1])

  window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
  gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
  decision_time_sec = float(window_sec + gaze_shift_sec)

  reference_signal_list = generate_cca_reference_signals_beta(
      candidate_freqs = selected_freq_values,
      srate = srate,
      n_samples = n_samples,
      n_harmonics = n_harmonics
  )

  fold_result_rows = []

  all_y_true = []
  all_y_pred = []
  all_score_vectors = []

  for one_split_value in split_values:
      test_mask = meta_df[split_key].values == one_split_value
      test_indices = np.where(test_mask)[0]

      y_test = y[test_indices]
      y_pred_list = []
      runtime_list = []

      for idx in test_indices:
          trial_eeg = np.asarray(trial_list[idx], dtype=float)
          _runtime_start = time.perf_counter()

          if apply_bandpass:
              trial_eeg = bandpass_filter_multichannel_signal_beta(
                  multichannel_signal = trial_eeg,
                  srate = srate,
                  lowcut = lowcut,
                  highcut = highcut,
                  filter_order = filter_order
              )

          pred_label, cca_score_array = predict_one_trial_cca_beta(
              trial_eeg = trial_eeg,
              reference_signal_list = reference_signal_list
          )

          runtime_list.append(float(time.perf_counter() - _runtime_start))

          y_pred_list.append(int(pred_label))
          all_score_vectors.append(np.asarray(cca_score_array, dtype=float))

      y_pred = np.array(y_pred_list, dtype=int)

      fold_acc = accuracy_score(y_test, y_pred)
      mean_recognition_runtime_sec = float(np.mean(runtime_list)) if len(runtime_list) > 0 else 0.0

      fold_result_rows.append({
          'model_name': 'CCA',
          fold_name: one_split_value,
          'fold_accuracy': float(fold_acc),
          'n_train': np.nan,
          'n_test': int(len(y_test)),
          'window_sec': float(window_sec),
          'decision_time_sec': float(decision_time_sec),
          'mean_recognition_runtime_sec': float(mean_recognition_runtime_sec),
          'decision_time_with_runtime_sec': float(decision_time_sec + mean_recognition_runtime_sec)
      })

      all_y_true.extend(y_test.tolist())
      all_y_pred.extend(y_pred.tolist())

  all_y_true = np.array(all_y_true, dtype=int)
  all_y_pred = np.array(all_y_pred, dtype=int)

  overall_accuracy = accuracy_score(all_y_true, all_y_pred)

  fold_results_df = pd.DataFrame(fold_result_rows)

  summary_row = {
      'model_name': 'CCA',
      'split_mode': split_mode,
      'overall_accuracy': float(overall_accuracy),
      'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
      'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof = 0)),
      'n_samples': int(len(all_y_true)),
      'n_channels': int(info_dict['n_channels']),
      'n_classes': int(info_dict['n_classes']),
      'window_sec': float(window_sec),
      'gaze_shift_sec': float(gaze_shift_sec),
      'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()),
      'mean_decision_time_with_runtime_sec': float(fold_results_df['decision_time_with_runtime_sec'].mean()),
      'n_harmonics': int(n_harmonics),
      'apply_bandpass': bool(apply_bandpass),
      'lowcut': float(lowcut),
      'highcut': float(highcut),
      'filter_order': int(filter_order)
  }

  if split_mode == 'subject':
      summary_row['n_subjects'] = int(len(split_values))
  elif split_mode == 'block':
      summary_row['n_blocks'] = int(len(split_values))

  summary_df = pd.DataFrame([summary_row])

  result_dict = {
      'summary_df': summary_df,
      'fold_results_df': fold_results_df,
      'all_y_true': all_y_true,
      'all_y_pred': all_y_pred,
      'all_score_vectors': all_score_vectors
  }

  return result_dict

In [ ]:
# CCA结果展示函数

def print_cca_result_with_notes_beta(
  cca_result
):
#   print('=' * 80)
#   print('CCA summary_df')
  print(cca_result['summary_df'])
#   print('=' * 80)

In [ ]:
# # 跨受试者（CCA不需要训练，不存在非跨受试者）
# cca_result_subject = train_test_cca_beta(
#     trial_list = trial_list,
#     y = y_cca,
#     meta_df = meta_df_cca,
#     info_dict = info_dict_cca,
#     split_mode = 'subject',
#     n_harmonics = 5
# )

# cca_result_subject = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = cca_result_subject,
#     info_dict = info_dict_cca
# )

# print_cca_result_with_notes_beta(
#     cca_result = cca_result_subject
# )

# cca_result_subject['summary_df']

In [ ]:
# cca_result_block = train_test_cca_beta(
#     trial_list = trial_list,
#     y = y_cca,
#     meta_df = meta_df_cca,
#     info_dict = info_dict_cca,
#     split_mode = 'block',
#     n_harmonics = 5
# )

# cca_result_block = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = cca_result_block,
#     info_dict = info_dict_cca
# )

# cca_result_block['summary_df']

### FBCCA参数定义与测试

In [ ]:
# ------------ FBCCA模型参数定义函数 ------------

# 过程：
# ① 定义FBCCA识别时需要的核心参数，例如谐波数、子带数、滤波器组边界、权重参数
# ② 将这些参数整理为一个字典，方便后续统一调用
# ③ 该模型不需要训练数据拟合，因此这里只保存参数
# ④ 输出可用于后续FBCCA识别的参数字典

def define_fbcca_model_beta(
  n_harmonics = 5,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25
):
  if subband_low_list is None:
      subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0]

  fbcca_model_dict = {
      'model_name': 'FBCCA',
      'n_harmonics': int(n_harmonics),
      'subband_low_list': [float(x) for x in subband_low_list],
      'subband_highcut': float(subband_highcut),
      'filter_order': int(filter_order),
      'weight_a': float(weight_a),
      'weight_b': float(weight_b)
  }

  return fbcca_model_dict

In [ ]:
# ------------ FBCCA参考信号生成函数 ------------

# 过程：
# ① 根据候选频率、谐波数、采样率和时间点数生成每个频率的参考信号
# ② 每个候选频率的参考信号由多个谐波的正弦与余弦组成
# ③ 输出所有候选频率对应的参考信号列表
# ④ 后续FBCCA将在每个子带上分别与这些参考信号进行CCA比较

import numpy as np

def generate_fbcca_reference_signals_beta(
  candidate_freqs,
  srate,
  n_samples,
  n_harmonics = 5
):
  t = np.arange(n_samples, dtype=float) / float(srate)

  reference_signal_list = []

  for freq in candidate_freqs:
      ref_components = []

      for h in range(1, n_harmonics + 1):
          ref_components.append(np.sin(2 * np.pi * h * freq * t))
          ref_components.append(np.cos(2 * np.pi * h * freq * t))

      ref_matrix = np.array(ref_components, dtype=float)
      reference_signal_list.append(ref_matrix)

  return reference_signal_list

In [ ]:
# ------------ FBCCA滤波器组函数 ------------

# 过程：
# ① 对一个trial的多通道EEG信号进行带通滤波
# ② 根据子带下限频率列表，对同一个trial生成多个子带版本
# ③ 每个子带都保留相同的通道数和时间长度
# ④ 输出由多个子带EEG组成的列表，供后续FBCCA逐子带做CCA

from scipy.signal import butter, filtfilt

def bandpass_filter_multichannel_signal_beta(
  multichannel_signal,
  srate,
  lowcut,
  highcut,
  filter_order = 4
):
  signal_array = np.array(multichannel_signal, dtype=float)

  nyquist = srate / 2.0
  low = lowcut / nyquist
  high = highcut / nyquist

  if low <= 0:
      low = 1e-6
  if high >= 1:
      high = 0.999999

  b, a = butter(
      filter_order,
      [low, high],
      btype = 'band'
  )

  filtered_signal = filtfilt(
      b,
      a,
      signal_array,
      axis = 1
  )

  filtered_signal = np.array(filtered_signal, dtype=float)

  return filtered_signal


def generate_fbcca_subband_signals_beta(
  trial_eeg,
  srate,
  subband_low_list,
  subband_highcut = 90.0,
  filter_order = 4
):
  subband_signal_list = []

  for lowcut in subband_low_list:
      one_subband_signal = bandpass_filter_multichannel_signal_beta(
          multichannel_signal = trial_eeg,
          srate = srate,
          lowcut = lowcut,
          highcut = subband_highcut,
          filter_order = filter_order
      )

      subband_signal_list.append(one_subband_signal)

  return subband_signal_list

In [ ]:
# ------------ 单个trial的CCA分数函数 ------------

# 过程：
# ① 输入一个trial的多通道EEG窗口和某个候选频率的参考信号
# ② 对EEG和参考信号分别去均值
# ③ 使用CCA计算第一对典型变量
# ④ 输出该候选频率对应的典型相关系数，作为匹配分数

from sklearn.cross_decomposition import CCA

def compute_cca_score_one_trial_beta(
  trial_eeg,
  ref_signal
):
  X = np.array(trial_eeg, dtype=float).T
  Y = np.array(ref_signal, dtype=float).T

  X = X - np.mean(X, axis = 0, keepdims = True)
  Y = Y - np.mean(Y, axis = 0, keepdims = True)

  cca = CCA(
    n_components = 1,
    max_iter = 2000,
    tol = 1e-5
)
  cca.fit(X, Y)

  X_c, Y_c = cca.transform(X, Y)

  corr_value = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]

  if np.isnan(corr_value):
      corr_value = 0.0

  return float(corr_value)

In [ ]:
# ------------ 单个trial的FBCCA预测函数 ------------

# 过程：
# ① 将一个trial的多通道EEG窗口分解为多个子带信号
# ② 在每个子带上，分别计算该trial与每个候选频率参考信号的CCA相关系数
# ③ 对各子带CCA分数按权重进行加权融合，得到每个候选频率的最终分数
# ④ 取最终分数最大的候选频率作为预测类别

def compute_fbcca_weights_beta(
  n_subbands,
  weight_a = 1.25,
  weight_b = 0.25
):
  weight_list = []

  for m in range(1, n_subbands + 1):
      one_weight = (m ** (-weight_a)) + weight_b
      weight_list.append(float(one_weight))

  weight_array = np.array(weight_list, dtype=float)

  return weight_array


def predict_one_trial_fbcca_beta(
  trial_eeg,
  srate,
  reference_signal_list,
  subband_low_list,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25
):
  subband_signal_list = generate_fbcca_subband_signals_beta(
      trial_eeg = trial_eeg,
      srate = srate,
      subband_low_list = subband_low_list,
      subband_highcut = subband_highcut,
      filter_order = filter_order
  )

  n_subbands = len(subband_signal_list)
  n_classes = len(reference_signal_list)

  weight_array = compute_fbcca_weights_beta(
      n_subbands = n_subbands,
      weight_a = weight_a,
      weight_b = weight_b
  )

  fbcca_score_array = np.zeros(n_classes, dtype=float)

  for subband_idx, subband_eeg in enumerate(subband_signal_list):
      one_weight = weight_array[subband_idx]

      for class_idx, ref_signal in enumerate(reference_signal_list):
          one_cca_score = compute_cca_score_one_trial_beta(
              trial_eeg = subband_eeg,
              ref_signal = ref_signal
          )

          # 这里使用 rho^2 再加权，是FBCCA中常见的融合方式
          fbcca_score_array[class_idx] += one_weight * (one_cca_score ** 2)

  pred_label = int(np.argmax(fbcca_score_array))

  return pred_label, fbcca_score_array

In [ ]:
# ------------ FBCCA统一评估函数 ------------

# 过程：
# ① 输入已经构造好的trial_list、标签y、样本信息meta_df、频率信息info_dict
# ② 根据 split_mode 选择按 block 或按 subject 分组统计结果
# ③ 对每个测试trial直接执行FBCCA识别，不进行训练
# ④ 汇总总体准确率、平均折准确率、标准差、样本数、类别数等整体结果

from sklearn.metrics import accuracy_score
import pandas as pd

def evaluate_fbcca_beta(
  trial_list,
  y,
  meta_df,
  info_dict,
  split_mode = 'block',
  n_harmonics = 5,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25
):
  if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
      subject_result_list = []

      subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

      for one_subject_id in subject_id_list:
          subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)
          subject_indices = np.where(subject_mask)[0]

          one_subject_result = evaluate_fbcca_beta(
              trial_list = [trial_list[idx] for idx in subject_indices],
              y = y[subject_indices],
              meta_df = meta_df.loc[subject_mask].reset_index(drop = True),
              info_dict = info_dict,
              split_mode = split_mode,
              n_harmonics = n_harmonics,
              subband_low_list = subband_low_list,
              subband_highcut = subband_highcut,
              filter_order = filter_order,
              weight_a = weight_a,
              weight_b = weight_b
          )
          subject_result_list.append(one_subject_result)

      return _aggregate_fixed_window_subject_results_beta(
          subject_result_list = subject_result_list,
          model_name = 'FBCCA',
          split_mode = split_mode,
          meta_df = meta_df,
          info_dict = info_dict
      )

  fbcca_model_dict = define_fbcca_model_beta(
      n_harmonics = n_harmonics,
      subband_low_list = subband_low_list,
      subband_highcut = subband_highcut,
      filter_order = filter_order,
      weight_a = weight_a,
      weight_b = weight_b
  )

  if split_mode == 'block':
      split_key = 'block_idx'
      split_values = sorted(meta_df['block_idx'].unique().tolist())
      fold_name = 'test_block'
  elif split_mode == 'subject':
      split_key = 'subject_id'
      split_values = sorted(meta_df['subject_id'].unique().tolist())
      fold_name = 'test_subject_id'
  else:
      raise ValueError("split_mode 只能写 'block' 或 'subject'。")

  selected_freq_values = info_dict['selected_freq_values']
  srate = info_dict['srate']
  n_samples = int(trial_list[0].shape[1])

  window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
  gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
  decision_time_sec = float(window_sec + gaze_shift_sec)

  reference_signal_list = generate_fbcca_reference_signals_beta(
      candidate_freqs = selected_freq_values,
      srate = srate,
      n_samples = n_samples,
      n_harmonics = n_harmonics
  )

  fold_result_rows = []

  all_y_true = []
  all_y_pred = []
  all_score_vectors = []

  for one_split_value in split_values:
      test_mask = meta_df[split_key].values == one_split_value
      test_indices = np.where(test_mask)[0]

      y_test = y[test_indices]
      y_pred_list = []
      runtime_list = []

      for idx in test_indices:
          trial_eeg = np.asarray(trial_list[idx], dtype=float)

          _runtime_start = time.perf_counter()
          pred_label, fbcca_score_array = predict_one_trial_fbcca_beta(
              trial_eeg = trial_eeg,
              srate = srate,
              reference_signal_list = reference_signal_list,
              subband_low_list = fbcca_model_dict['subband_low_list'],
              subband_highcut = fbcca_model_dict['subband_highcut'],
              filter_order = fbcca_model_dict['filter_order'],
              weight_a = fbcca_model_dict['weight_a'],
              weight_b = fbcca_model_dict['weight_b']
          )

          runtime_list.append(float(time.perf_counter() - _runtime_start))

          y_pred_list.append(int(pred_label))
          all_score_vectors.append(np.asarray(fbcca_score_array, dtype=float))

      y_pred = np.array(y_pred_list, dtype=int)

      fold_acc = accuracy_score(y_test, y_pred)
      mean_recognition_runtime_sec = float(np.mean(runtime_list)) if len(runtime_list) > 0 else 0.0

      fold_result_rows.append({
          'model_name': 'FBCCA',
          fold_name: one_split_value,
          'fold_accuracy': float(fold_acc),
          'n_train': np.nan,
          'n_test': int(len(y_test)),
          'window_sec': float(window_sec),
          'decision_time_sec': float(decision_time_sec),
          'mean_recognition_runtime_sec': float(mean_recognition_runtime_sec),
          'decision_time_with_runtime_sec': float(decision_time_sec + mean_recognition_runtime_sec)
      })

      all_y_true.extend(y_test.tolist())
      all_y_pred.extend(y_pred.tolist())

  all_y_true = np.array(all_y_true, dtype=int)
  all_y_pred = np.array(all_y_pred, dtype=int)

  overall_accuracy = accuracy_score(all_y_true, all_y_pred)

  fold_results_df = pd.DataFrame(fold_result_rows)

  summary_row = {
      'model_name': 'FBCCA',
      'split_mode': split_mode,
      'overall_accuracy': float(overall_accuracy),
      'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
      'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof = 0)),
      'n_samples': int(len(all_y_true)),
      'n_channels': int(info_dict['n_channels']),
      'n_classes': int(info_dict['n_classes']),
      'window_sec': float(window_sec),
      'gaze_shift_sec': float(gaze_shift_sec),
      'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()),
      'mean_decision_time_with_runtime_sec': float(fold_results_df['decision_time_with_runtime_sec'].mean()),
      'n_harmonics': int(fbcca_model_dict['n_harmonics']),
      'n_subbands': int(len(fbcca_model_dict['subband_low_list'])),
      'subband_highcut': float(fbcca_model_dict['subband_highcut']),
      'filter_order': int(fbcca_model_dict['filter_order']),
      'weight_a': float(fbcca_model_dict['weight_a']),
      'weight_b': float(fbcca_model_dict['weight_b'])
  }

  if split_mode == 'subject':
      summary_row['n_subjects'] = int(len(split_values))
  elif split_mode == 'block':
      summary_row['n_blocks'] = int(len(split_values))

  summary_df = pd.DataFrame([summary_row])

  result_dict = {
      'summary_df': summary_df,
      'fold_results_df': fold_results_df,
      'all_y_true': all_y_true,
      'all_y_pred': all_y_pred,
      'all_score_vectors': all_score_vectors
  }

  return result_dict

In [ ]:
# FBCCA结果展示函数

def print_fbcca_result_with_notes_beta(
  fbcca_result
):
#   print('=' * 80)
#   print('FBCCA summary_df')
  print(fbcca_result['summary_df'])
#   print('=' * 80)

In [ ]:
# fbcca_result_subject = evaluate_fbcca_beta(
#     trial_list = trial_list,
#     y = y_cca,
#     meta_df = meta_df_cca,
#     info_dict = info_dict_cca,
#     split_mode = 'subject',
#     n_harmonics = 5,
#     subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0],
#     subband_highcut = 90.0,
#     filter_order = 4,
#     weight_a = 1.25,
#     weight_b = 0.25
# )

# fbcca_result_subject = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = fbcca_result_subject,
#     info_dict = info_dict_cca
# )

# print_fbcca_result_with_notes_beta(
#     fbcca_result = fbcca_result_subject
# )

# fbcca_result_subject['summary_df']

In [ ]:
# fbcca_result_block = evaluate_fbcca_beta(
#     trial_list = trial_list,
#     y = y_cca,
#     meta_df = meta_df_cca,
#     info_dict = info_dict_cca,
#     split_mode = 'block',
#     n_harmonics = 5,
#     subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0],
#     subband_highcut = 90.0,
#     filter_order = 4,
#     weight_a = 1.25,
#     weight_b = 0.25
# )

# fbcca_result_block = append_fixed_window_speed_itr_to_result_beta(
#     result_dict = fbcca_result_block,
#     info_dict = info_dict_cca
# )

# fbcca_result_block['summary_df']

### TRCA参数定义与测试

In [ ]:
# ------------ TRCA / msTRCA辅助函数 ------------

# 过程：
# ① 提供TRCA / msTRCA都会用到的基础函数，例如去均值、相关系数、邻频模板选择
# ② 对训练trial构造TRCA所需的广义特征值分解问题
# ③ 训练每个类别、每个子带的空间滤波器与模板
# ④ 为后续“无早停”版TRCA / msTRCA统一评估函数做准备

def _center_multichannel_signal_beta(
  multichannel_signal
):
  signal_array = np.array(multichannel_signal, dtype=float)
  signal_array = signal_array - np.mean(signal_array, axis=1, keepdims=True)

  return signal_array


def _corrcoef_1d_safe_beta(
  x,
  y
):
  x = np.asarray(x, dtype=float).reshape(-1)
  y = np.asarray(y, dtype=float).reshape(-1)

  if len(x) != len(y):
      raise ValueError('x 和 y 的长度必须一致。')

  x_std = np.std(x)
  y_std = np.std(y)

  if x_std <= 1e-12 or y_std <= 1e-12:
      return 0.0

  corr_value = np.corrcoef(x, y)[0, 1]

  if np.isnan(corr_value):
      corr_value = 0.0

  return float(corr_value)


def _stack_trials_one_class_beta(
  trial_list,
  y,
  selected_indices,
  class_label
):
  class_trial_list = []

  for idx in selected_indices:
      if int(y[idx]) == int(class_label):
          class_trial_list.append(np.asarray(trial_list[idx], dtype=float))

  if len(class_trial_list) == 0:
      raise ValueError(f'类别 {class_label} 在当前训练集中没有trial，无法训练TRCA/msTRCA。')

  class_trial_array = np.stack(class_trial_list, axis=0)

  return class_trial_array


def _build_class_trials_list_beta(
  trial_list,
  y,
  selected_indices,
  n_classes
):
  class_trials_list = []

  for class_label in range(int(n_classes)):
      class_trial_array = _stack_trials_one_class_beta(
          trial_list = trial_list,
          y = y,
          selected_indices = selected_indices,
          class_label = class_label
      )
      class_trials_list.append(class_trial_array)

  return class_trials_list


def _filter_trial_array_to_one_subband_beta(
  trial_array,
  srate,
  lowcut,
  highcut,
  filter_order = 4
):
  trial_array = np.asarray(trial_array, dtype=float)

  filtered_trial_list = []

  for one_trial in trial_array:
      filtered_trial = bandpass_filter_multichannel_signal_beta(
          multichannel_signal = one_trial,
          srate = srate,
          lowcut = lowcut,
          highcut = highcut,
          filter_order = filter_order
      )
      filtered_trial_list.append(np.asarray(filtered_trial, dtype=float))

  filtered_trial_array = np.stack(filtered_trial_list, axis=0)

  return filtered_trial_array


def _prepare_subband_class_trials_beta(
  class_trials_list,
  srate,
  subband_low_list,
  subband_highcut = 90.0,
  filter_order = 4
):
  n_classes = len(class_trials_list)

  subband_class_trials = []

  for class_idx in range(n_classes):
      one_class_trial_array = np.asarray(class_trials_list[class_idx], dtype=float)

      one_class_subband_list = []

      for lowcut in subband_low_list:
          filtered_trial_array = _filter_trial_array_to_one_subband_beta(
              trial_array = one_class_trial_array,
              srate = srate,
              lowcut = lowcut,
              highcut = subband_highcut,
              filter_order = filter_order
          )
          one_class_subband_list.append(filtered_trial_array)

      subband_class_trials.append(one_class_subband_list)

  return subband_class_trials


def _solve_trca_generalized_eigen_beta(
  S,
  Q,
  regularization = 1e-8
):
  S = np.asarray(S, dtype=float)
  Q = np.asarray(Q, dtype=float)

  n_channels = S.shape[0]
  reg_value = float(regularization)

  S = 0.5 * (S + S.T)
  Q = 0.5 * (Q + Q.T)

  Q_reg = Q + reg_value * np.eye(n_channels)

  eigvals, eigvecs = eigh(S, Q_reg)

  best_vec = np.asarray(eigvecs[:, np.argmax(eigvals)], dtype=float).reshape(-1)

  vec_norm = np.linalg.norm(best_vec)
  if vec_norm <= 1e-12:
      best_vec = np.ones(n_channels, dtype=float) / np.sqrt(n_channels)
  else:
      best_vec = best_vec / vec_norm

  return best_vec


def _train_trca_spatial_filter_beta(
  trial_array,
  regularization = 1e-8
):
  trial_array = np.asarray(trial_array, dtype=float)

  if trial_array.ndim != 3:
      raise ValueError('trial_array 必须是 [n_trials, n_channels, n_samples] 三维数组。')

  n_trials, n_channels, _ = trial_array.shape

  if n_trials < 2:
      return np.ones(n_channels, dtype=float) / np.sqrt(n_channels)

  centered_trial_list = []
  Q = np.zeros((n_channels, n_channels), dtype=float)

  for trial_idx in range(n_trials):
      one_trial = _center_multichannel_signal_beta(trial_array[trial_idx])
      centered_trial_list.append(one_trial)
      Q += one_trial @ one_trial.T

  trial_sum = np.sum(np.stack(centered_trial_list, axis=0), axis=0)
  S = trial_sum @ trial_sum.T - Q

  spatial_filter = _solve_trca_generalized_eigen_beta(
      S = S,
      Q = Q,
      regularization = regularization
  )

  return spatial_filter


def _build_frequency_neighbor_map_beta(
  selected_freq_values,
  neighbor_template_count = 2
):
  """
  为 ms-eTRCA 构造邻近频率表。

  注意：这里的 neighbor_template_count 表示“目标频率之外额外加入的邻近频率数量”。
  因此 neighbor_template_count=2 表示：目标频率 + 2 个邻近频率，共 3 个频率参与 ms-eTRCA 空间滤波器训练。
  """
  selected_freq_values = [float(x) for x in selected_freq_values]
  n_classes = len(selected_freq_values)
  k_neighbors = int(neighbor_template_count)

  neighbor_map = {}

  for class_idx in range(n_classes):
      if k_neighbors <= 0:
          neighbor_map[class_idx] = []
          continue

      distance_rows = []

      for other_idx in range(n_classes):
          if other_idx == class_idx:
              continue

          freq_distance = abs(selected_freq_values[class_idx] - selected_freq_values[other_idx])

          distance_rows.append(
              (float(freq_distance), float(selected_freq_values[other_idx]), int(other_idx))
          )

      distance_rows = sorted(distance_rows, key=lambda x: (x[0], x[1], x[2]))

      chosen_neighbors = [int(x[2]) for x in distance_rows[:k_neighbors]]
      neighbor_map[class_idx] = chosen_neighbors

  return neighbor_map


def _accumulate_trca_covariance_from_one_stimulus_beta(
  trial_array
):
  """
  对单个刺激频率内部的多个 trial 构造 TRCA 的 S 和 Q。
  该函数只使用同一刺激频率内部 trial 之间的可重复成分，不引入不同频率之间的交叉协方差。
  """
  trial_array = np.asarray(trial_array, dtype=float)

  if trial_array.ndim != 3:
      raise ValueError('trial_array 必须是 [n_trials, n_channels, n_samples] 三维数组。')

  n_trials, n_channels, _ = trial_array.shape

  S = np.zeros((n_channels, n_channels), dtype=float)
  Q = np.zeros((n_channels, n_channels), dtype=float)

  if n_trials < 2:
      return S, Q

  centered_trial_list = []

  for trial_idx in range(n_trials):
      one_trial = _center_multichannel_signal_beta(trial_array[trial_idx])
      centered_trial_list.append(one_trial)
      Q += one_trial @ one_trial.T

  trial_sum = np.sum(np.stack(centered_trial_list, axis=0), axis=0)
  S = trial_sum @ trial_sum.T - Q

  return S, Q


def _train_msetrca_spatial_filter_beta(
  subband_class_trials,
  subband_idx,
  selected_stimulus_indices,
  regularization = 1e-8
):
  """
  按原始 ms-eTRCA 思路训练空间滤波器。

  与之前“把目标频率和邻近频率 trial 直接拼接成一个大类”不同，
  这里先分别计算每个刺激频率内部的 TRCA 协方差信息，
  再把这些频率内部的 S/Q 矩阵相加。

  这样邻近频率只提供各自内部的可重复成分，不会把不同频率之间的交叉协方差混入训练。
  """
  selected_stimulus_indices = [int(x) for x in selected_stimulus_indices]

  first_trial_array = np.asarray(
      subband_class_trials[selected_stimulus_indices[0]][subband_idx],
      dtype=float
  )
  n_channels = int(first_trial_array.shape[1])

  total_S = np.zeros((n_channels, n_channels), dtype=float)
  total_Q = np.zeros((n_channels, n_channels), dtype=float)
  valid_stimulus_count = 0

  for stimulus_idx in selected_stimulus_indices:
      one_trial_array = np.asarray(
          subband_class_trials[int(stimulus_idx)][subband_idx],
          dtype=float
      )

      one_S, one_Q = _accumulate_trca_covariance_from_one_stimulus_beta(
          trial_array = one_trial_array
      )

      if np.any(np.isfinite(one_S)) and np.any(np.abs(one_Q) > 0):
          total_S += one_S
          total_Q += one_Q
          valid_stimulus_count += 1

  if valid_stimulus_count <= 0 or np.all(np.abs(total_Q) <= 1e-12):
      return np.ones(n_channels, dtype=float) / np.sqrt(n_channels)

  spatial_filter = _solve_trca_generalized_eigen_beta(
      S = total_S,
      Q = total_Q,
      regularization = regularization
  )

  return spatial_filter


def _fit_trca_like_model_beta(
  trial_list,
  y,
  train_indices,
  info_dict,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25,
  regularization = 1e-8,
  use_ensemble = True,
  use_multistimulus = False,
  neighbor_template_count = 2
):
  srate = float(info_dict['srate'])
  n_classes = int(info_dict['n_classes'])
  selected_freq_values = [float(x) for x in info_dict['selected_freq_values']]

  if subband_low_list is None:
      subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0]
  subband_low_list = [float(x) for x in subband_low_list]

  class_trials_list = _build_class_trials_list_beta(
      trial_list = trial_list,
      y = y,
      selected_indices = train_indices,
      n_classes = n_classes
  )

  subband_class_trials = _prepare_subband_class_trials_beta(
      class_trials_list = class_trials_list,
      srate = srate,
      subband_low_list = subband_low_list,
      subband_highcut = subband_highcut,
      filter_order = filter_order
  )

  if use_multistimulus:
      neighbor_map = _build_frequency_neighbor_map_beta(
          selected_freq_values = selected_freq_values,
          neighbor_template_count = neighbor_template_count
      )
  else:
      neighbor_map = {class_idx: [] for class_idx in range(n_classes)}

  class_template_list_by_subband = []
  class_filter_list_by_subband = []
  ensemble_filter_matrix_list = []

  for subband_idx in range(len(subband_low_list)):
      one_subband_template_list = []
      one_subband_filter_list = []

      for class_idx in range(n_classes):
          target_trial_array = np.asarray(
              subband_class_trials[class_idx][subband_idx],
              dtype=float
          )

          # 测试匹配模板保持为目标频率自身的平均模板。
          # 邻近频率只参与 ms-eTRCA 空间滤波器训练，不直接混入目标频率模板。
          target_template = np.mean(target_trial_array, axis=0)
          one_subband_template_list.append(np.asarray(target_template, dtype=float))

          if use_multistimulus and int(neighbor_template_count) > 0:
              selected_stimulus_indices = [int(class_idx)] + [int(x) for x in neighbor_map[class_idx]]

              spatial_filter = _train_msetrca_spatial_filter_beta(
                  subband_class_trials = subband_class_trials,
                  subband_idx = subband_idx,
                  selected_stimulus_indices = selected_stimulus_indices,
                  regularization = regularization
              )
          else:
              spatial_filter = _train_trca_spatial_filter_beta(
                  trial_array = target_trial_array,
                  regularization = regularization
              )

          one_subband_filter_list.append(np.asarray(spatial_filter, dtype=float))

      one_subband_ensemble_filter = np.column_stack(one_subband_filter_list)

      class_template_list_by_subband.append(one_subband_template_list)
      class_filter_list_by_subband.append(one_subband_filter_list)
      ensemble_filter_matrix_list.append(one_subband_ensemble_filter)

  model_state = {
      'subband_low_list': subband_low_list,
      'subband_highcut': float(subband_highcut),
      'filter_order': int(filter_order),
      'weight_a': float(weight_a),
      'weight_b': float(weight_b),
      'regularization': float(regularization),
      'use_ensemble': bool(use_ensemble),
      'use_multistimulus': bool(use_multistimulus),
      'neighbor_template_count': int(neighbor_template_count),
      'neighbor_map': neighbor_map,
      'ms_trca_mode': 'original_ms_eTRCA_like_internal_covariance_sum' if use_multistimulus else 'standard_TRCA',
      'class_template_list_by_subband': class_template_list_by_subband,
      'class_filter_list_by_subband': class_filter_list_by_subband,
      'ensemble_filter_matrix_list': ensemble_filter_matrix_list
  }

  return model_state

def _predict_one_trial_trca_like_beta(
  trial_eeg,
  srate,
  model_state
):
  trial_eeg = np.asarray(trial_eeg, dtype=float)

  subband_low_list = model_state['subband_low_list']
  subband_highcut = float(model_state['subband_highcut'])
  filter_order = int(model_state['filter_order'])
  weight_a = float(model_state['weight_a'])
  weight_b = float(model_state['weight_b'])
  use_ensemble = bool(model_state['use_ensemble'])

  subband_signal_list = generate_fbcca_subband_signals_beta(
      trial_eeg = trial_eeg,
      srate = srate,
      subband_low_list = subband_low_list,
      subband_highcut = subband_highcut,
      filter_order = filter_order
  )

  n_subbands = len(subband_signal_list)
  n_classes = len(model_state['class_template_list_by_subband'][0])

  weight_array = compute_fbcca_weights_beta(
      n_subbands = n_subbands,
      weight_a = weight_a,
      weight_b = weight_b
  )

  score_array = np.zeros(n_classes, dtype=float)

  for subband_idx in range(n_subbands):
      test_subband = np.asarray(subband_signal_list[subband_idx], dtype=float)

      for class_idx in range(n_classes):
          template = np.asarray(
              model_state['class_template_list_by_subband'][subband_idx][class_idx],
              dtype=float
          )

          if use_ensemble:
              W = np.asarray(
                  model_state['ensemble_filter_matrix_list'][subband_idx],
                  dtype=float
              )

              projected_test = test_subband.T @ W
              projected_template = template.T @ W

              corr_value = _corrcoef_1d_safe_beta(
                  projected_test.reshape(-1),
                  projected_template.reshape(-1)
              )
          else:
              w = np.asarray(
                  model_state['class_filter_list_by_subband'][subband_idx][class_idx],
                  dtype=float
              ).reshape(-1)

              projected_test = test_subband.T @ w
              projected_template = template.T @ w

              corr_value = _corrcoef_1d_safe_beta(
                  projected_test,
                  projected_template
              )

          score_array[class_idx] += weight_array[subband_idx] * (corr_value ** 2)

  pred_label = int(np.argmax(score_array))

  return pred_label, score_array

In [ ]:
# ------------ TRCA模型参数定义与无早停统一测试函数 ------------

# 过程：
# ① 定义TRCA默认参数，尽量贴近BETA论文中的设置：ensemble + filter bank
# ② 在每一折中使用训练block学习TRCA空间滤波器和类别模板
# ③ 对测试trial做子带融合识别，得到每个候选频率的TRCA分数
# ④ 汇总整体准确率、折准确率、窗口长度、ITR相关基础字段

def define_trca_model_beta(
  use_ensemble = True,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25,
  regularization = 1e-8
):
  if subband_low_list is None:
      subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0]

  trca_model_dict = {
      'model_name': 'TRCA',
      'use_ensemble': bool(use_ensemble),
      'subband_low_list': [float(x) for x in subband_low_list],
      'subband_highcut': float(subband_highcut),
      'filter_order': int(filter_order),
      'weight_a': float(weight_a),
      'weight_b': float(weight_b),
      'regularization': float(regularization)
  }

  return trca_model_dict


def train_test_trca_beta(
  trial_list,
  y,
  meta_df,
  info_dict,
  split_mode = 'block',
  use_ensemble = True,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25,
  regularization = 1e-8
):
  if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
      subject_result_list = []

      subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

      for one_subject_id in subject_id_list:
          subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)
          subject_indices = np.where(subject_mask)[0]

          one_subject_result = train_test_trca_beta(
              trial_list = [trial_list[idx] for idx in subject_indices],
              y = y[subject_indices],
              meta_df = meta_df.loc[subject_mask].reset_index(drop = True),
              info_dict = info_dict,
              split_mode = split_mode,
              use_ensemble = use_ensemble,
              subband_low_list = subband_low_list,
              subband_highcut = subband_highcut,
              filter_order = filter_order,
              weight_a = weight_a,
              weight_b = weight_b,
              regularization = regularization
          )
          subject_result_list.append(one_subject_result)

      return _aggregate_fixed_window_subject_results_beta(
          subject_result_list = subject_result_list,
          model_name = 'TRCA',
          split_mode = split_mode,
          meta_df = meta_df,
          info_dict = info_dict
      )

  trca_model_dict = define_trca_model_beta(
      use_ensemble = use_ensemble,
      subband_low_list = subband_low_list,
      subband_highcut = subband_highcut,
      filter_order = filter_order,
      weight_a = weight_a,
      weight_b = weight_b,
      regularization = regularization
  )

  if split_mode == 'block':
      split_key = 'block_idx'
      split_values = sorted(meta_df['block_idx'].unique().tolist())
      fold_name = 'test_block'
  elif split_mode == 'subject':
      split_key = 'subject_id'
      split_values = sorted(meta_df['subject_id'].unique().tolist())
      fold_name = 'test_subject_id'
  else:
      raise ValueError("split_mode 只能写 'block' 或 'subject'。")

  srate = float(info_dict['srate'])
  window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
  gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
  decision_time_sec = float(window_sec + gaze_shift_sec)

  fold_result_rows = []

  all_y_true = []
  all_y_pred = []
  all_score_vectors = []

  for one_split_value in split_values:
      train_mask = meta_df[split_key].values != one_split_value
      test_mask = meta_df[split_key].values == one_split_value

      train_indices = np.where(train_mask)[0]
      test_indices = np.where(test_mask)[0]

      model_state = _fit_trca_like_model_beta(
          trial_list = trial_list,
          y = y,
          train_indices = train_indices,
          info_dict = info_dict,
          subband_low_list = trca_model_dict['subband_low_list'],
          subband_highcut = trca_model_dict['subband_highcut'],
          filter_order = trca_model_dict['filter_order'],
          weight_a = trca_model_dict['weight_a'],
          weight_b = trca_model_dict['weight_b'],
          regularization = trca_model_dict['regularization'],
          use_ensemble = trca_model_dict['use_ensemble'],
          use_multistimulus = False,
          neighbor_template_count = 0
      )

      y_test = y[test_indices]
      y_pred_list = []
      runtime_list = []

      for idx in test_indices:
          trial_eeg = np.asarray(trial_list[idx], dtype=float)

          _runtime_start = time.perf_counter()
          pred_label, score_array = _predict_one_trial_trca_like_beta(
              trial_eeg = trial_eeg,
              srate = srate,
              model_state = model_state
          )

          runtime_list.append(float(time.perf_counter() - _runtime_start))

          y_pred_list.append(int(pred_label))
          all_score_vectors.append(np.asarray(score_array, dtype=float))

      y_pred = np.asarray(y_pred_list, dtype=int)
      fold_acc = accuracy_score(y_test, y_pred)
      mean_recognition_runtime_sec = float(np.mean(runtime_list)) if len(runtime_list) > 0 else 0.0

      fold_result_rows.append({
          'model_name': 'TRCA',
          fold_name: one_split_value,
          'fold_accuracy': float(fold_acc),
          'n_train': int(len(train_indices)),
          'n_test': int(len(test_indices)),
          'window_sec': float(window_sec),
          'decision_time_sec': float(decision_time_sec),
          'mean_recognition_runtime_sec': float(mean_recognition_runtime_sec),
          'decision_time_with_runtime_sec': float(decision_time_sec + mean_recognition_runtime_sec)
      })

      all_y_true.extend(y_test.tolist())
      all_y_pred.extend(y_pred.tolist())

  all_y_true = np.asarray(all_y_true, dtype=int)
  all_y_pred = np.asarray(all_y_pred, dtype=int)

  overall_accuracy = accuracy_score(all_y_true, all_y_pred)

  fold_results_df = pd.DataFrame(fold_result_rows)

  summary_row = {
      'model_name': 'TRCA',
      'split_mode': split_mode,
      'overall_accuracy': float(overall_accuracy),
      'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
      'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof = 0)),
      'n_samples': int(len(all_y_true)),
      'n_channels': int(info_dict['n_channels']),
      'n_classes': int(info_dict['n_classes']),
      'window_sec': float(window_sec),
      'gaze_shift_sec': float(gaze_shift_sec),
      'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()),
      'mean_decision_time_with_runtime_sec': float(fold_results_df['decision_time_with_runtime_sec'].mean()),
      'use_ensemble': bool(trca_model_dict['use_ensemble']),
      'n_subbands': int(len(trca_model_dict['subband_low_list'])),
      'subband_highcut': float(trca_model_dict['subband_highcut']),
      'filter_order': int(trca_model_dict['filter_order']),
      'weight_a': float(trca_model_dict['weight_a']),
      'weight_b': float(trca_model_dict['weight_b']),
      'regularization': float(trca_model_dict['regularization'])
  }

  if split_mode == 'subject':
      summary_row['n_subjects'] = int(len(split_values))
  elif split_mode == 'block':
      summary_row['n_blocks'] = int(len(split_values))

  summary_df = pd.DataFrame([summary_row])

  result_dict = {
      'summary_df': summary_df,
      'fold_results_df': fold_results_df,
      'all_y_true': all_y_true,
      'all_y_pred': all_y_pred,
      'all_score_vectors': all_score_vectors
  }

  return result_dict

### msTRCA参数定义与测试

In [ ]:
# ------------ msTRCA模型参数定义与无早停统一测试函数 ------------

# 过程：
# ① 定义msTRCA默认参数：ensemble + filter bank + 2个邻近频率（目标频率+邻近频率共3个）
# ② 在每次训练中，按 ms-eTRCA 思路分别构造目标频率及邻近频率各自内部的TRCA协方差信息，再组合训练空间滤波器
# ③ 测试阶段仍使用目标频率自身的平均模板做相关匹配，输出各类别msTRCA分数
# ④ 汇总整体准确率、折准确率、窗口长度、ITR相关基础字段

def define_mstrca_model_beta(
  use_ensemble = True,
  neighbor_template_count = 2,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25,
  regularization = 1e-8
):
  if subband_low_list is None:
      subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0]

  mstrca_model_dict = {
      'model_name': 'msTRCA',
      'use_ensemble': bool(use_ensemble),
      'neighbor_template_count': int(neighbor_template_count),
      'subband_low_list': [float(x) for x in subband_low_list],
      'subband_highcut': float(subband_highcut),
      'filter_order': int(filter_order),
      'weight_a': float(weight_a),
      'weight_b': float(weight_b),
      'regularization': float(regularization)
  }

  return mstrca_model_dict


def train_test_mstrca_beta(
  trial_list,
  y,
  meta_df,
  info_dict,
  split_mode = 'block',
  use_ensemble = True,
  neighbor_template_count = 2,
  subband_low_list = None,
  subband_highcut = 90.0,
  filter_order = 4,
  weight_a = 1.25,
  weight_b = 0.25,
  regularization = 1e-8
):
  if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
      subject_result_list = []

      subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

      for one_subject_id in subject_id_list:
          subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)
          subject_indices = np.where(subject_mask)[0]

          one_subject_result = train_test_mstrca_beta(
              trial_list = [trial_list[idx] for idx in subject_indices],
              y = y[subject_indices],
              meta_df = meta_df.loc[subject_mask].reset_index(drop = True),
              info_dict = info_dict,
              split_mode = split_mode,
              use_ensemble = use_ensemble,
              neighbor_template_count = neighbor_template_count,
              subband_low_list = subband_low_list,
              subband_highcut = subband_highcut,
              filter_order = filter_order,
              weight_a = weight_a,
              weight_b = weight_b,
              regularization = regularization
          )
          subject_result_list.append(one_subject_result)

      return _aggregate_fixed_window_subject_results_beta(
          subject_result_list = subject_result_list,
          model_name = 'msTRCA',
          split_mode = split_mode,
          meta_df = meta_df,
          info_dict = info_dict
      )

  mstrca_model_dict = define_mstrca_model_beta(
      use_ensemble = use_ensemble,
      neighbor_template_count = neighbor_template_count,
      subband_low_list = subband_low_list,
      subband_highcut = subband_highcut,
      filter_order = filter_order,
      weight_a = weight_a,
      weight_b = weight_b,
      regularization = regularization
  )

  if split_mode == 'block':
      split_key = 'block_idx'
      split_values = sorted(meta_df['block_idx'].unique().tolist())
      fold_name = 'test_block'
  elif split_mode == 'subject':
      split_key = 'subject_id'
      split_values = sorted(meta_df['subject_id'].unique().tolist())
      fold_name = 'test_subject_id'
  else:
      raise ValueError("split_mode 只能写 'block' 或 'subject'。")

  srate = float(info_dict['srate'])
  window_sec = float(info_dict.get('window_sec', info_dict.get('T_sec')))
  gaze_shift_sec = float(info_dict.get('gaze_shift_sec', 0.55))
  decision_time_sec = float(window_sec + gaze_shift_sec)

  fold_result_rows = []

  all_y_true = []
  all_y_pred = []
  all_score_vectors = []

  for one_split_value in split_values:
      train_mask = meta_df[split_key].values != one_split_value
      test_mask = meta_df[split_key].values == one_split_value

      train_indices = np.where(train_mask)[0]
      test_indices = np.where(test_mask)[0]

      model_state = _fit_trca_like_model_beta(
          trial_list = trial_list,
          y = y,
          train_indices = train_indices,
          info_dict = info_dict,
          subband_low_list = mstrca_model_dict['subband_low_list'],
          subband_highcut = mstrca_model_dict['subband_highcut'],
          filter_order = mstrca_model_dict['filter_order'],
          weight_a = mstrca_model_dict['weight_a'],
          weight_b = mstrca_model_dict['weight_b'],
          regularization = mstrca_model_dict['regularization'],
          use_ensemble = mstrca_model_dict['use_ensemble'],
          use_multistimulus = True,
          neighbor_template_count = mstrca_model_dict['neighbor_template_count']
      )

      y_test = y[test_indices]
      y_pred_list = []
      runtime_list = []

      for idx in test_indices:
          trial_eeg = np.asarray(trial_list[idx], dtype=float)

          _runtime_start = time.perf_counter()
          pred_label, score_array = _predict_one_trial_trca_like_beta(
              trial_eeg = trial_eeg,
              srate = srate,
              model_state = model_state
          )

          runtime_list.append(float(time.perf_counter() - _runtime_start))

          y_pred_list.append(int(pred_label))
          all_score_vectors.append(np.asarray(score_array, dtype=float))

      y_pred = np.asarray(y_pred_list, dtype=int)
      fold_acc = accuracy_score(y_test, y_pred)
      mean_recognition_runtime_sec = float(np.mean(runtime_list)) if len(runtime_list) > 0 else 0.0

      fold_result_rows.append({
          'model_name': 'msTRCA',
          fold_name: one_split_value,
          'fold_accuracy': float(fold_acc),
          'n_train': int(len(train_indices)),
          'n_test': int(len(test_indices)),
          'window_sec': float(window_sec),
          'decision_time_sec': float(decision_time_sec),
          'mean_recognition_runtime_sec': float(mean_recognition_runtime_sec),
          'decision_time_with_runtime_sec': float(decision_time_sec + mean_recognition_runtime_sec)
      })

      all_y_true.extend(y_test.tolist())
      all_y_pred.extend(y_pred.tolist())

  all_y_true = np.asarray(all_y_true, dtype=int)
  all_y_pred = np.asarray(all_y_pred, dtype=int)

  overall_accuracy = accuracy_score(all_y_true, all_y_pred)

  fold_results_df = pd.DataFrame(fold_result_rows)

  summary_row = {
      'model_name': 'msTRCA',
      'split_mode': split_mode,
      'overall_accuracy': float(overall_accuracy),
      'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
      'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof = 0)),
      'n_samples': int(len(all_y_true)),
      'n_channels': int(info_dict['n_channels']),
      'n_classes': int(info_dict['n_classes']),
      'window_sec': float(window_sec),
      'gaze_shift_sec': float(gaze_shift_sec),
      'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()),
      'mean_decision_time_with_runtime_sec': float(fold_results_df['decision_time_with_runtime_sec'].mean()),
      'use_ensemble': bool(mstrca_model_dict['use_ensemble']),
      'neighbor_template_count': int(mstrca_model_dict['neighbor_template_count']),
      'n_subbands': int(len(mstrca_model_dict['subband_low_list'])),
      'subband_highcut': float(mstrca_model_dict['subband_highcut']),
      'filter_order': int(mstrca_model_dict['filter_order']),
      'weight_a': float(mstrca_model_dict['weight_a']),
      'weight_b': float(mstrca_model_dict['weight_b']),
      'regularization': float(mstrca_model_dict['regularization'])
  }

  if split_mode == 'subject':
      summary_row['n_subjects'] = int(len(split_values))
  elif split_mode == 'block':
      summary_row['n_blocks'] = int(len(split_values))

  summary_df = pd.DataFrame([summary_row])

  result_dict = {
      'summary_df': summary_df,
      'fold_results_df': fold_results_df,
      'all_y_true': all_y_true,
      'all_y_pred': all_y_pred,
      'all_score_vectors': all_score_vectors
  }

  return result_dict

In [ ]:
# ------------ TRCA / msTRCA结果展示函数 ------------

def print_trca_result_with_notes_beta(
  trca_result
):
  print(trca_result['summary_df'])


def print_mstrca_result_with_notes_beta(
  mstrca_result
):
  print(mstrca_result['summary_df'])

### 统一调用六个模型

In [ ]:
svm_result_block = train_test_svm_beta(
    X = X,
    y = y,
    meta_df = meta_df,
    info_dict = info_dict,
    split_mode = 'block',
    C = 1.0,
    kernel = 'rbf',
    gamma = 'scale',
    probability = False,
    random_state = 42
)
svm_result_block = append_fixed_window_speed_itr_to_result_beta(
    result_dict = svm_result_block,
    info_dict = info_dict
)

lda_result_block = train_test_lda_beta(
    X = X,
    y = y,
    meta_df = meta_df,
    info_dict = info_dict,
    split_mode = 'block',
    solver = 'lsqr',
    shrinkage = 'auto'
)
lda_result_block = append_fixed_window_speed_itr_to_result_beta(
    result_dict = lda_result_block,
    info_dict = info_dict
)

cca_result_block = train_test_cca_beta(
    trial_list = trial_list,
    y = y_cca,
    meta_df = meta_df_cca,
    info_dict = info_dict_cca,
    split_mode = 'block',
    n_harmonics = 5,
    apply_bandpass = True,
    lowcut = 6.0,
    highcut = 80.0,
    filter_order = 4
)
cca_result_block = append_fixed_window_speed_itr_to_result_beta(
    result_dict = cca_result_block,
    info_dict = info_dict_cca
)

fbcca_result_block = evaluate_fbcca_beta(
    trial_list = trial_list,
    y = y_cca,
    meta_df = meta_df_cca,
    info_dict = info_dict_cca,
    split_mode = 'block',
    n_harmonics = 5,
    subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0],
    subband_highcut = 90.0,
    filter_order = 4,
    weight_a = 1.25,
    weight_b = 0.25
)
fbcca_result_block = append_fixed_window_speed_itr_to_result_beta(
    result_dict = fbcca_result_block,
    info_dict = info_dict_cca
)

# ------------ 无早停：加入TRCA和msTRCA的主调用示例 ------------

trca_result_block = train_test_trca_beta(
    trial_list = trial_list,
    y = y_cca,
    meta_df = meta_df_cca,
    info_dict = info_dict_cca,
    split_mode = 'block',
    use_ensemble = True,
    subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0],
    subband_highcut = 90.0,
    filter_order = 4,
    weight_a = 1.25,
    weight_b = 0.25,
    regularization = 1e-8
)
trca_result_block = append_fixed_window_speed_itr_to_result_beta(
    result_dict = trca_result_block,
    info_dict = info_dict_cca
)

mstrca_result_block = train_test_mstrca_beta(
    trial_list = trial_list,
    y = y_cca,
    meta_df = meta_df_cca,
    info_dict = info_dict_cca,
    split_mode = 'block',
    use_ensemble = True,
    neighbor_template_count = 2,
    subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0],
    subband_highcut = 90.0,
    filter_order = 4,
    weight_a = 1.25,
    weight_b = 0.25,
    regularization = 1e-8
)
mstrca_result_block = append_fixed_window_speed_itr_to_result_beta(
    result_dict = mstrca_result_block,
    info_dict = info_dict_cca
)

In [ ]:
# 查看在固定窗长下，六个模型的结果
result_summary_df = pd.concat([
    svm_result_block['summary_df'],
    lda_result_block['summary_df'],
    cca_result_block['summary_df'],
    fbcca_result_block['summary_df'],
    trca_result_block['summary_df'],
    mstrca_result_block['summary_df']
], ignore_index=True)

display(result_summary_df[[
    'model_name',
    'overall_accuracy',
    'mean_fold_accuracy',
    'mean_decision_time_sec',
    'mean_recognition_runtime_sec',
    'mean_decision_time_with_runtime_sec',
    'itr_bits_per_min',
    'itr_with_runtime_bits_per_min',
    'overall_accuracy_based_itr_bits_per_min',
    'overall_accuracy_based_itr_with_runtime_bits_per_min'
]])

# 保存固定窗最终 summary 结果。
save_summary_result_df_beta(
    summary_df=result_summary_df,
    strategy_name='fixed_window',
    min_stable_windows=np.nan
)


### 按BETA方法找出最大ITR

In [ ]:
# # 输出为单个受试者下扫描多种窗长后，拿到最佳窗长，得到CCA和FBCCA模型在该窗长下的最好结果
# import numpy as np
# import pandas as pd

# def get_beta_window_list_for_subject_beta(
#     subject_id
# ):
#     subject_str = str(subject_id).upper().replace(' ', '')
#     if not subject_str.startswith('S'):
#         raise ValueError('subject_id 应该类似 S6 或 S67。')
#     subject_num = int(subject_str[1:])
#     max_window_sec = 2.0 if subject_num <= 15 else 3.0
#     window_sec_list = np.arange(0.2, max_window_sec + 0.0001, 0.2)
#     window_sec_list = [round(float(x), 1) for x in window_sec_list]
#     return window_sec_list

# def scan_single_subject_max_itr_beta(
#     base_folder,
#     subject_id,
#     selected_chans,
#     selected_freq_indices,
#     model_name='FBCCA',
#     window_sec_list=None,
#     latency_correction_sec=0.13,
#     gaze_shift_sec=0.55,
#     cca_n_harmonics=5,
#     cca_apply_bandpass=True,
#     cca_lowcut=6.0,
#     cca_highcut=80.0,
#     cca_filter_order=4,
#     fbcca_n_harmonics=5,
#     fbcca_subband_low_list=None,
#     fbcca_subband_highcut=90.0,
#     fbcca_filter_order=4,
#     fbcca_weight_a=1.25,
#     fbcca_weight_b=0.25
# ):
#     model_name_upper = str(model_name).upper()

#     if model_name_upper not in ['CCA', 'FBCCA']:
#         raise ValueError("model_name 只能写 'CCA' 或 'FBCCA'。")

#     if window_sec_list is None:
#         window_sec_list = get_beta_window_list_for_subject_beta(subject_id)

#     all_curve_rows = []
#     all_result_dict_by_window = {}

#     for window_sec in window_sec_list:
#         trial_list, y, meta_df, info_dict = build_cca_dataset_beta(
#             base_folder=base_folder,
#             selected_subject_ids=[subject_id],
#             selected_chans=selected_chans,
#             selected_freq_indices=selected_freq_indices,
#             window_sec=window_sec,
#             latency_correction_sec=latency_correction_sec,
#             gaze_shift_sec=gaze_shift_sec
#         )

#         if model_name_upper == 'CCA':
#             result_dict = train_test_cca_beta(
#                 trial_list=trial_list,
#                 y=y,
#                 meta_df=meta_df,
#                 info_dict=info_dict,
#                 split_mode='block',
#                 n_harmonics=cca_n_harmonics,
#                 apply_bandpass=cca_apply_bandpass,
#                 lowcut=cca_lowcut,
#                 highcut=cca_highcut,
#                 filter_order=cca_filter_order
#             )
#         else:
#             result_dict = evaluate_fbcca_beta(
#                 trial_list=trial_list,
#                 y=y,
#                 meta_df=meta_df,
#                 info_dict=info_dict,
#                 split_mode='block',
#                 n_harmonics=fbcca_n_harmonics,
#                 subband_low_list=fbcca_subband_low_list,
#                 subband_highcut=fbcca_subband_highcut,
#                 filter_order=fbcca_filter_order,
#                 weight_a=fbcca_weight_a,
#                 weight_b=fbcca_weight_b
#             )

#         result_dict = append_fixed_window_speed_itr_to_result_beta(
#             result_dict=result_dict,
#             info_dict=info_dict
#         )

#         summary_df = result_dict['summary_df'].copy()
#         row_dict = summary_df.iloc[0].to_dict()
#         row_dict['subject_id'] = str(subject_id)
#         row_dict['window_sec'] = float(window_sec)

#         all_curve_rows.append(row_dict)
#         all_result_dict_by_window[float(window_sec)] = result_dict

#     curve_df = pd.DataFrame(all_curve_rows).sort_values(
#         by='window_sec',
#         ascending=True
#     ).reset_index(drop=True)

#     best_idx = curve_df['itr_bits_per_min'].astype(float).idxmax()
#     best_window_sec = float(curve_df.loc[best_idx, 'window_sec'])
#     best_result_dict = all_result_dict_by_window[best_window_sec]

#     best_summary_df = best_result_dict['summary_df'].copy()
#     best_fold_results_df = best_result_dict['fold_results_df'].copy()

#     return {
#         'subject_id': str(subject_id),
#         'model_name': model_name_upper,
#         'window_sec_list': window_sec_list,
#         'curve_df': curve_df,
#         'best_window_sec': best_window_sec,
#         'best_summary_df': best_summary_df,
#         'best_fold_results_df': best_fold_results_df,
#         'best_result_dict': best_result_dict
#     }

# def compare_cca_fbcca_single_subject_max_itr_beta(
#     base_folder,
#     subject_id,
#     selected_chans,
#     selected_freq_indices,
#     window_sec_list=None,
#     latency_correction_sec=0.13,
#     gaze_shift_sec=0.55,
#     cca_n_harmonics=5,
#     cca_apply_bandpass=True,
#     cca_lowcut=6.0,
#     cca_highcut=80.0,
#     cca_filter_order=4,
#     fbcca_n_harmonics=5,
#     fbcca_subband_low_list=None,
#     fbcca_subband_highcut=90.0,
#     fbcca_filter_order=4,
#     fbcca_weight_a=1.25,
#     fbcca_weight_b=0.25
# ):
#     cca_scan = scan_single_subject_max_itr_beta(
#         base_folder=base_folder,
#         subject_id=subject_id,
#         selected_chans=selected_chans,
#         selected_freq_indices=selected_freq_indices,
#         model_name='CCA',
#         window_sec_list=window_sec_list,
#         latency_correction_sec=latency_correction_sec,
#         gaze_shift_sec=gaze_shift_sec,
#         cca_n_harmonics=cca_n_harmonics,
#         cca_apply_bandpass=cca_apply_bandpass,
#         cca_lowcut=cca_lowcut,
#         cca_highcut=cca_highcut,
#         cca_filter_order=cca_filter_order
#     )

#     fbcca_scan = scan_single_subject_max_itr_beta(
#         base_folder=base_folder,
#         subject_id=subject_id,
#         selected_chans=selected_chans,
#         selected_freq_indices=selected_freq_indices,
#         model_name='FBCCA',
#         window_sec_list=window_sec_list,
#         latency_correction_sec=latency_correction_sec,
#         gaze_shift_sec=gaze_shift_sec,
#         fbcca_n_harmonics=fbcca_n_harmonics,
#         fbcca_subband_low_list=fbcca_subband_low_list,
#         fbcca_subband_highcut=fbcca_subband_highcut,
#         fbcca_filter_order=fbcca_filter_order,
#         fbcca_weight_a=fbcca_weight_a,
#         fbcca_weight_b=fbcca_weight_b
#     )

#     compare_curve_df = pd.concat([
#         cca_scan['curve_df'][[
#             'subject_id',
#             'model_name',
#             'window_sec',
#             'overall_accuracy',
#             'mean_fold_accuracy',
#             'mean_decision_time_sec',
#             'itr_bits_per_min',
#             'overall_accuracy_based_itr_bits_per_min'
#         ]],
#         fbcca_scan['curve_df'][[
#             'subject_id',
#             'model_name',
#             'window_sec',
#             'overall_accuracy',
#             'mean_fold_accuracy',
#             'mean_decision_time_sec',
#             'itr_bits_per_min',
#             'overall_accuracy_based_itr_bits_per_min'
#         ]]
#     ], ignore_index=True)

#     best_compare_df = pd.concat([
#         cca_scan['best_summary_df'],
#         fbcca_scan['best_summary_df']
#     ], ignore_index=True)

#     return {
#         'cca_scan': cca_scan,
#         'fbcca_scan': fbcca_scan,
#         'compare_curve_df': compare_curve_df,
#         'best_compare_df': best_compare_df
#     }

# base_folder = '/content/drive/MyDrive/论文2025/BETA database'
# subject_id = 'S67'
# selected_chans = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
# selected_freq_indices = list(range(40))

# result_compare = compare_cca_fbcca_single_subject_max_itr_beta(
#     base_folder=base_folder,
#     subject_id=subject_id,
#     selected_chans=selected_chans,
#     selected_freq_indices=selected_freq_indices,
#     window_sec_list=None,
#     latency_correction_sec=0.13,
#     gaze_shift_sec=0.55,
#     cca_n_harmonics=5,
#     cca_apply_bandpass=True,
#     cca_lowcut=6.0,
#     cca_highcut=80.0,
#     cca_filter_order=4,
#     fbcca_n_harmonics=5,
#     fbcca_subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
#     fbcca_subband_highcut=90.0,
#     fbcca_filter_order=4,
#     fbcca_weight_a=1.25,
#     fbcca_weight_b=0.25
# )

# compare_curve_df = result_compare['compare_curve_df'].copy()
# best_compare_df = result_compare['best_compare_df'].copy()

# # display(compare_curve_df.sort_values(['model_name', 'window_sec']).reset_index(drop=True)[[
# #     'subject_id',
# #     'model_name',
# #     'window_sec',
# #     'overall_accuracy',
# #     'mean_fold_accuracy',
# #     'mean_decision_time_sec',
# #     'itr_bits_per_min',
# #     'overall_accuracy_based_itr_bits_per_min'
# # ]])

# display(best_compare_df[[
#     'model_name',
#     'overall_accuracy',
#     'mean_fold_accuracy',
#     'mean_decision_time_sec',
#     'itr_bits_per_min',
#     'overall_accuracy_based_itr_bits_per_min'
# ]])

# print('CCA best window_sec =', result_compare['cca_scan']['best_window_sec'])
# print('FBCCA best window_sec =', result_compare['fbcca_scan']['best_window_sec'])

# # display(result_compare['cca_scan']['best_fold_results_df'])
# # display(result_compare['fbcca_scan']['best_fold_results_df'])

In [ ]:
# # 输出为单个受试者下扫描多种窗长后，拿到最佳窗长，得到CCA和FBCCA模型在该窗长下的最好结果
# # 一次性得到多个受试者的结果
# import os
# import numpy as np
# import pandas as pd
# import scipy.io as sio

# from joblib import Parallel, delayed
# from threadpoolctl import threadpool_limits


# def _subject_num_beta(subject_id):
#     subject_str = str(subject_id).upper().replace(' ', '')
#     if not subject_str.startswith('S'):
#         raise ValueError('subject_id 应该类似 S6 或 S67。')
#     return int(subject_str[1:])


# def get_beta_window_list_for_subject_beta(subject_id):
#     subject_num = _subject_num_beta(subject_id)
#     max_window_sec = 2.0 if subject_num <= 15 else 3.0
#     window_sec_list = np.arange(0.2, max_window_sec + 0.0001, 0.2)
#     return [round(float(x), 1) for x in window_sec_list]


# def _load_single_subject_beta_for_batch(
#     base_folder,
#     subject_id,
#     selected_chans
# ):
#     subject_file_path = os.path.join(base_folder, f'{subject_id}.mat')

#     load_data = sio.loadmat(
#         subject_file_path,
#         struct_as_record=False,
#         squeeze_me=True
#     )

#     data = load_data['data']
#     eeg = np.asarray(data.EEG, dtype=float)
#     info = data.suppl_info

#     srate = float(info.srate)
#     chan_list = info.chan[:, -1]
#     chan_list_py = [str(x) for x in chan_list]
#     full_candidate_freqs = [float(x) for x in info.freqs]

#     chan_name_to_idx = {}
#     for i, ch_name in enumerate(chan_list_py):
#         chan_name_to_idx[str(ch_name).upper()] = i

#     selected_idx = []
#     missing_chans = []
#     for ch in selected_chans:
#         ch_upper = str(ch).upper()
#         if ch_upper in chan_name_to_idx:
#             selected_idx.append(chan_name_to_idx[ch_upper])
#         else:
#             missing_chans.append(ch)

#     if len(missing_chans) > 0:
#         raise ValueError(f'受试者 {subject_id} 缺少这些通道: {missing_chans}')

#     return {
#         'subject_id': str(subject_id),
#         'eeg': eeg,
#         'srate': srate,
#         'selected_idx': selected_idx,
#         'full_candidate_freqs': full_candidate_freqs,
#         'n_blocks': int(eeg.shape[2]),
#         'n_freqs': int(eeg.shape[3]),
#         'n_total_samples': int(eeg.shape[1])
#     }


# def _extract_trial_window_from_loaded_subject_beta(
#     subject_data,
#     block_idx,
#     freq_idx,
#     window_sec,
#     pre_stim_sec=0.5,
#     latency_correction_sec=0.13
# ):
#     eeg = subject_data['eeg']
#     selected_idx = subject_data['selected_idx']
#     srate = subject_data['srate']
#     n_total_samples = subject_data['n_total_samples']

#     start_time_sec = float(pre_stim_sec + latency_correction_sec)
#     start_sample = int(round(start_time_sec * srate))
#     window_samples = int(round(float(window_sec) * srate))
#     end_sample = start_sample + window_samples

#     if end_sample > n_total_samples:
#         raise ValueError(
#             f"受试者 {subject_data['subject_id']} 在 window_sec={window_sec} 时截窗越界: "
#             f"end_sample={end_sample}, total={n_total_samples}"
#         )

#     trial_window = eeg[selected_idx, start_sample:end_sample, block_idx, freq_idx]
#     return np.asarray(trial_window, dtype=float)


# def _build_trials_one_window_from_loaded_subject_beta(
#     subject_data,
#     selected_freq_indices,
#     window_sec,
#     latency_correction_sec=0.13,
#     pre_stim_sec=0.5
# ):
#     trial_list = []
#     y_list = []
#     block_idx_list = []

#     n_blocks = subject_data['n_blocks']

#     for local_label, freq_idx in enumerate(selected_freq_indices):
#         for block_idx in range(n_blocks):
#             trial_window = _extract_trial_window_from_loaded_subject_beta(
#                 subject_data=subject_data,
#                 block_idx=block_idx,
#                 freq_idx=freq_idx,
#                 window_sec=window_sec,
#                 pre_stim_sec=pre_stim_sec,
#                 latency_correction_sec=latency_correction_sec
#             )
#             trial_list.append(trial_window)
#             y_list.append(local_label)
#             block_idx_list.append(block_idx)

#     y = np.asarray(y_list, dtype=int)
#     block_idx_array = np.asarray(block_idx_list, dtype=int)

#     return trial_list, y, block_idx_array


# def _summarize_best_row_from_predictions_beta(
#     subject_id,
#     model_name,
#     y_true,
#     y_pred,
#     block_idx_array,
#     window_sec,
#     gaze_shift_sec,
#     n_classes
# ):
#     y_true = np.asarray(y_true, dtype=int)
#     y_pred = np.asarray(y_pred, dtype=int)
#     block_idx_array = np.asarray(block_idx_array, dtype=int)

#     decision_time_sec = float(window_sec + gaze_shift_sec)
#     unique_blocks = sorted(np.unique(block_idx_array).tolist())

#     fold_acc_list = []
#     fold_itr_list = []

#     for one_block in unique_blocks:
#         mask = (block_idx_array == one_block)
#         fold_acc = float(np.mean(y_true[mask] == y_pred[mask]))
#         fold_itr = float(compute_itr_bits_per_min(
#             n_classes=n_classes,
#             accuracy=fold_acc,
#             decision_time_sec=decision_time_sec
#         ))
#         fold_acc_list.append(fold_acc)
#         fold_itr_list.append(fold_itr)

#     overall_accuracy = float(np.mean(y_true == y_pred))
#     overall_accuracy_based_itr = float(compute_itr_bits_per_min(
#         n_classes=n_classes,
#         accuracy=overall_accuracy,
#         decision_time_sec=decision_time_sec
#     ))

#     row = {
#         'subject_id': str(subject_id),
#         'model_name': str(model_name),
#         'best_window_sec': float(window_sec),
#         'overall_accuracy': float(overall_accuracy),
#         'mean_fold_accuracy': float(np.mean(fold_acc_list)),
#         'mean_decision_time_sec': float(decision_time_sec),
#         'itr_bits_per_min': float(np.mean(fold_itr_list)),
#         'overall_accuracy_based_itr_bits_per_min': float(overall_accuracy_based_itr)
#     }

#     return row


# def _evaluate_cca_one_window_fast_beta(
#     subject_data,
#     selected_freq_indices,
#     selected_freq_values,
#     trial_list,
#     y,
#     block_idx_array,
#     window_sec,
#     gaze_shift_sec=0.55,
#     n_harmonics=5,
#     apply_bandpass=True,
#     lowcut=6.0,
#     highcut=80.0,
#     filter_order=4
# ):
#     srate = subject_data['srate']
#     n_samples = int(trial_list[0].shape[1])

#     reference_signal_list = generate_cca_reference_signals_beta(
#         candidate_freqs=selected_freq_values,
#         srate=srate,
#         n_samples=n_samples,
#         n_harmonics=n_harmonics
#     )

#     y_pred = np.empty_like(y)

#     for i, trial_eeg in enumerate(trial_list):
#         one_trial = trial_eeg

#         if apply_bandpass:
#             one_trial = bandpass_filter_multichannel_signal_beta(
#                 multichannel_signal=one_trial,
#                 srate=srate,
#                 lowcut=lowcut,
#                 highcut=highcut,
#                 filter_order=filter_order
#             )

#         pred_label, _ = predict_one_trial_cca_beta(
#             trial_eeg=one_trial,
#             reference_signal_list=reference_signal_list
#         )
#         y_pred[i] = int(pred_label)

#     return _summarize_best_row_from_predictions_beta(
#         subject_id=subject_data['subject_id'],
#         model_name='CCA',
#         y_true=y,
#         y_pred=y_pred,
#         block_idx_array=block_idx_array,
#         window_sec=window_sec,
#         gaze_shift_sec=gaze_shift_sec,
#         n_classes=len(selected_freq_indices)
#     )


# def _evaluate_fbcca_one_window_fast_beta(
#     subject_data,
#     selected_freq_indices,
#     selected_freq_values,
#     trial_list,
#     y,
#     block_idx_array,
#     window_sec,
#     gaze_shift_sec=0.55,
#     n_harmonics=5,
#     subband_low_list=None,
#     subband_highcut=90.0,
#     filter_order=4,
#     weight_a=1.25,
#     weight_b=0.25
# ):
#     srate = subject_data['srate']
#     n_samples = int(trial_list[0].shape[1])

#     if subband_low_list is None:
#         subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0]

#     reference_signal_list = generate_fbcca_reference_signals_beta(
#         candidate_freqs=selected_freq_values,
#         srate=srate,
#         n_samples=n_samples,
#         n_harmonics=n_harmonics
#     )

#     y_pred = np.empty_like(y)

#     for i, trial_eeg in enumerate(trial_list):
#         pred_label, _ = predict_one_trial_fbcca_beta(
#             trial_eeg=trial_eeg,
#             srate=srate,
#             reference_signal_list=reference_signal_list,
#             subband_low_list=subband_low_list,
#             subband_highcut=subband_highcut,
#             filter_order=filter_order,
#             weight_a=weight_a,
#             weight_b=weight_b
#         )
#         y_pred[i] = int(pred_label)

#     return _summarize_best_row_from_predictions_beta(
#         subject_id=subject_data['subject_id'],
#         model_name='FBCCA',
#         y_true=y,
#         y_pred=y_pred,
#         block_idx_array=block_idx_array,
#         window_sec=window_sec,
#         gaze_shift_sec=gaze_shift_sec,
#         n_classes=len(selected_freq_indices)
#     )


# def _scan_one_subject_best_rows_beta(
#     base_folder,
#     subject_id,
#     selected_chans,
#     selected_freq_indices,
#     models_to_run=('CCA', 'FBCCA'),
#     latency_correction_sec=0.13,
#     gaze_shift_sec=0.55,
#     pre_stim_sec=0.5,
#     window_sec_list=None,
#     cca_n_harmonics=5,
#     cca_apply_bandpass=True,
#     cca_lowcut=6.0,
#     cca_highcut=80.0,
#     cca_filter_order=4,
#     fbcca_n_harmonics=5,
#     fbcca_subband_low_list=None,
#     fbcca_subband_highcut=90.0,
#     fbcca_filter_order=4,
#     fbcca_weight_a=1.25,
#     fbcca_weight_b=0.25
# ):
#     with threadpool_limits(limits=1):
#         subject_data = _load_single_subject_beta_for_batch(
#             base_folder=base_folder,
#             subject_id=subject_id,
#             selected_chans=selected_chans
#         )

#         if window_sec_list is None:
#             window_sec_list_use = get_beta_window_list_for_subject_beta(subject_id)
#         else:
#             window_sec_list_use = [round(float(x), 1) for x in window_sec_list]

#         selected_freq_values = [
#             subject_data['full_candidate_freqs'][idx]
#             for idx in selected_freq_indices
#         ]

#         best_row_by_model = {}
#         best_itr_by_model = {}

#         for model_name in models_to_run:
#             model_name_upper = str(model_name).upper()
#             best_row_by_model[model_name_upper] = None
#             best_itr_by_model[model_name_upper] = -np.inf

#         for window_sec in window_sec_list_use:
#             trial_list, y, block_idx_array = _build_trials_one_window_from_loaded_subject_beta(
#                 subject_data=subject_data,
#                 selected_freq_indices=selected_freq_indices,
#                 window_sec=window_sec,
#                 latency_correction_sec=latency_correction_sec,
#                 pre_stim_sec=pre_stim_sec
#             )

#             if 'CCA' in [str(x).upper() for x in models_to_run]:
#                 cca_row = _evaluate_cca_one_window_fast_beta(
#                     subject_data=subject_data,
#                     selected_freq_indices=selected_freq_indices,
#                     selected_freq_values=selected_freq_values,
#                     trial_list=trial_list,
#                     y=y,
#                     block_idx_array=block_idx_array,
#                     window_sec=window_sec,
#                     gaze_shift_sec=gaze_shift_sec,
#                     n_harmonics=cca_n_harmonics,
#                     apply_bandpass=cca_apply_bandpass,
#                     lowcut=cca_lowcut,
#                     highcut=cca_highcut,
#                     filter_order=cca_filter_order
#                 )

#                 if float(cca_row['itr_bits_per_min']) > best_itr_by_model['CCA']:
#                     best_itr_by_model['CCA'] = float(cca_row['itr_bits_per_min'])
#                     best_row_by_model['CCA'] = cca_row

#             if 'FBCCA' in [str(x).upper() for x in models_to_run]:
#                 fbcca_row = _evaluate_fbcca_one_window_fast_beta(
#                     subject_data=subject_data,
#                     selected_freq_indices=selected_freq_indices,
#                     selected_freq_values=selected_freq_values,
#                     trial_list=trial_list,
#                     y=y,
#                     block_idx_array=block_idx_array,
#                     window_sec=window_sec,
#                     gaze_shift_sec=gaze_shift_sec,
#                     n_harmonics=fbcca_n_harmonics,
#                     subband_low_list=fbcca_subband_low_list,
#                     subband_highcut=fbcca_subband_highcut,
#                     filter_order=fbcca_filter_order,
#                     weight_a=fbcca_weight_a,
#                     weight_b=fbcca_weight_b
#                 )

#                 if float(fbcca_row['itr_bits_per_min']) > best_itr_by_model['FBCCA']:
#                     best_itr_by_model['FBCCA'] = float(fbcca_row['itr_bits_per_min'])
#                     best_row_by_model['FBCCA'] = fbcca_row

#         output_rows = []
#         for model_name in models_to_run:
#             model_name_upper = str(model_name).upper()
#             if best_row_by_model[model_name_upper] is not None:
#                 output_rows.append(best_row_by_model[model_name_upper])

#         return output_rows


# def batch_run_best_table_beta(
#     base_folder,
#     selected_subject_ids,
#     selected_chans,
#     selected_freq_indices,
#     models_to_run=('CCA', 'FBCCA'),
#     latency_correction_sec=0.13,
#     gaze_shift_sec=0.55,
#     pre_stim_sec=0.5,
#     window_sec_list=None,
#     cca_n_harmonics=5,
#     cca_apply_bandpass=True,
#     cca_lowcut=6.0,
#     cca_highcut=80.0,
#     cca_filter_order=4,
#     fbcca_n_harmonics=5,
#     fbcca_subband_low_list=None,
#     fbcca_subband_highcut=90.0,
#     fbcca_filter_order=4,
#     fbcca_weight_a=1.25,
#     fbcca_weight_b=0.25,
#     n_jobs_subject=-1,
#     verbose=10
# ):
#     if fbcca_subband_low_list is None:
#         fbcca_subband_low_list = [6.0, 14.0, 22.0, 30.0, 38.0]

#     if n_jobs_subject is None or int(n_jobs_subject) == 0:
#         n_jobs_subject = 1

#     subject_rows_nested = Parallel(
#         n_jobs=n_jobs_subject,
#         backend='loky',
#         verbose=verbose
#     )(
#         delayed(_scan_one_subject_best_rows_beta)(
#             base_folder=base_folder,
#             subject_id=subject_id,
#             selected_chans=selected_chans,
#             selected_freq_indices=selected_freq_indices,
#             models_to_run=models_to_run,
#             latency_correction_sec=latency_correction_sec,
#             gaze_shift_sec=gaze_shift_sec,
#             pre_stim_sec=pre_stim_sec,
#             window_sec_list=window_sec_list,
#             cca_n_harmonics=cca_n_harmonics,
#             cca_apply_bandpass=cca_apply_bandpass,
#             cca_lowcut=cca_lowcut,
#             cca_highcut=cca_highcut,
#             cca_filter_order=cca_filter_order,
#             fbcca_n_harmonics=fbcca_n_harmonics,
#             fbcca_subband_low_list=fbcca_subband_low_list,
#             fbcca_subband_highcut=fbcca_subband_highcut,
#             fbcca_filter_order=fbcca_filter_order,
#             fbcca_weight_a=fbcca_weight_a,
#             fbcca_weight_b=fbcca_weight_b
#         )
#         for subject_id in selected_subject_ids
#     )

#     all_rows = []
#     for rows_one_subject in subject_rows_nested:
#         all_rows.extend(rows_one_subject)

#     best_table_df = pd.DataFrame(all_rows)

#     if len(best_table_df) == 0:
#         return best_table_df

#     best_table_df['subject_num'] = best_table_df['subject_id'].apply(_subject_num_beta)

#     model_order_map = {}
#     for i, model_name in enumerate([str(x).upper() for x in models_to_run]):
#         model_order_map[model_name] = i

#     best_table_df['model_order'] = best_table_df['model_name'].map(model_order_map)

#     best_table_df = best_table_df.sort_values(
#         by=['subject_num', 'model_order'],
#         ascending=[True, True]
#     ).reset_index(drop=True)

#     best_table_df = best_table_df[[
#         'subject_id',
#         'model_name',
#         'best_window_sec',
#         'overall_accuracy',
#         'mean_fold_accuracy',
#         'mean_decision_time_sec',
#         'itr_bits_per_min',
#         'overall_accuracy_based_itr_bits_per_min'
#     ]]

#     return best_table_df

# # 以下设置参数
# base_folder = '/content/drive/MyDrive/论文2025/BETA database'

# selected_subject_ids = [
#     'S36'
# ]

# selected_chans = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
# selected_freq_indices = list(range(40))

# best_table_df = batch_run_best_table_beta(
#     base_folder=base_folder,
#     selected_subject_ids=selected_subject_ids,
#     selected_chans=selected_chans,
#     selected_freq_indices=selected_freq_indices,
#     models_to_run=('CCA', 'FBCCA'),
#     latency_correction_sec=0.13,
#     gaze_shift_sec=0.55,
#     cca_n_harmonics=5,
#     cca_apply_bandpass=True,
#     cca_lowcut=6.0,
#     cca_highcut=80.0,
#     cca_filter_order=4,
#     fbcca_n_harmonics=5,
#     fbcca_subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
#     fbcca_subband_highcut=90.0,
#     fbcca_filter_order=4,
#     fbcca_weight_a=1.25,
#     fbcca_weight_b=0.25,
#     n_jobs_subject=-1,
#     verbose=10
# )

# display(best_table_df)

## 非稳定早停下的模型

### 多窗口数据包与公共汇总函数

In [ ]:
# 根据 split_mode 返回对应的划分列名。
def get_split_key_beta(split_mode='block'):
    if split_mode == 'block':
        return 'block_idx'
    elif split_mode == 'subject':
        return 'subject_id'
    else:
        raise ValueError("split_mode 只能写 'block' 或 'subject'。")


import os
import numpy as np
import pandas as pd
import scipy.io as sio


def _load_single_subject_beta_once(
    base_folder,
    subject_id,
    selected_chans,
    selected_freq_indices,
    use_selected_freqs_as_feature_candidates=True
):
    current_subject_file_path = os.path.join(base_folder, str(subject_id) + '.mat')

    load_data = sio.loadmat(
        current_subject_file_path,
        struct_as_record=False,
        squeeze_me=True
    )

    data = load_data['data']
    eeg = np.asarray(data.EEG, dtype=float)
    info = data.suppl_info

    srate = float(info.srate)

    chan_list = info.chan[:, -1]
    chan_list_py = [str(x) for x in chan_list]

    full_candidate_freqs = [float(x) for x in info.freqs]
    selected_freq_values = [full_candidate_freqs[idx] for idx in selected_freq_indices]

    if use_selected_freqs_as_feature_candidates:
        candidate_freqs_for_feature = selected_freq_values.copy()
    else:
        candidate_freqs_for_feature = full_candidate_freqs.copy()

    chan_name_to_idx = {}
    for i, ch_name in enumerate(chan_list_py):
        chan_name_to_idx[str(ch_name).upper()] = i

    selected_idx = []
    missing_chans = []

    for ch in selected_chans:
        ch_upper = str(ch).upper()
        if ch_upper in chan_name_to_idx:
            selected_idx.append(chan_name_to_idx[ch_upper])
        else:
            missing_chans.append(ch)

    if len(missing_chans) > 0:
        raise ValueError(f'受试者 {subject_id} 中不存在通道 {missing_chans}。')

    if eeg.ndim != 4:
        raise ValueError(f'eeg 应为4维数组 [channels, samples, blocks, freqs]，当前 ndim={eeg.ndim}')

    n_channels, n_total_samples, n_blocks, n_freqs = eeg.shape

    return {
        'subject_id': str(subject_id),
        'eeg': eeg,
        'srate': srate,
        'chan_list_py': chan_list_py,
        'selected_idx': selected_idx,
        'selected_chans': [str(ch) for ch in selected_chans],
        'n_channels_total': int(n_channels),
        'n_total_samples': int(n_total_samples),
        'n_blocks': int(n_blocks),
        'n_freqs': int(n_freqs),
        'full_candidate_freqs': full_candidate_freqs,
        'selected_freq_values': selected_freq_values,
        'candidate_freqs_for_feature': candidate_freqs_for_feature
    }


def _extract_trial_window_from_loaded_subject_beta(
    loaded_subject,
    block_idx,
    freq_idx,
    window_sec,
    pre_stim_sec=0.5,
    latency_correction_sec=0.13
):
    eeg = loaded_subject['eeg']
    srate = float(loaded_subject['srate'])
    selected_idx = loaded_subject['selected_idx']

    n_channels, n_total_samples, n_blocks, n_freqs = eeg.shape

    if block_idx < 0 or block_idx >= n_blocks:
        raise IndexError(f'block_idx={block_idx} 超出范围 [0, {n_blocks - 1}]')

    if freq_idx < 0 or freq_idx >= n_freqs:
        raise IndexError(f'freq_idx={freq_idx} 超出范围 [0, {n_freqs - 1}]')

    start_time_sec = float(pre_stim_sec + latency_correction_sec)
    start_sample = int(round(start_time_sec * srate))
    window_samples = int(round(float(window_sec) * srate))
    end_sample = start_sample + window_samples

    if start_sample < 0:
        raise ValueError(f'start_sample={start_sample} 非法，请检查 pre_stim_sec 和 latency_correction_sec。')

    if end_sample > n_total_samples:
        raise ValueError(
            f'截窗越界: end_sample={end_sample} > 总采样点数={n_total_samples}。'
            f' 请减小 window_sec，或检查 pre_stim_sec / latency_correction_sec。'
        )

    trial_window = eeg[selected_idx, start_sample:end_sample, block_idx, freq_idx]
    return np.asarray(trial_window, dtype=float)


def _check_subject_consistency_beta(
    loaded_subjects
):
    if len(loaded_subjects) == 0:
        raise ValueError('loaded_subjects 不能为空。')

    srate_ref = float(loaded_subjects[0]['srate'])
    full_candidate_freqs_ref = loaded_subjects[0]['full_candidate_freqs'].copy()
    selected_freq_values_ref = loaded_subjects[0]['selected_freq_values'].copy()
    feature_candidate_freqs_ref = loaded_subjects[0]['candidate_freqs_for_feature'].copy()

    for loaded_subject in loaded_subjects[1:]:
        srate_now = float(loaded_subject['srate'])

        if not np.isclose(srate_ref, srate_now):
            raise ValueError(f"不同受试者采样率不一致: 之前={srate_ref}, 当前={srate_now}")

        if len(full_candidate_freqs_ref) != len(loaded_subject['full_candidate_freqs']):
            raise ValueError('不同受试者 full_candidate_freqs 长度不一致。')

        if not np.allclose(full_candidate_freqs_ref, loaded_subject['full_candidate_freqs']):
            raise ValueError('不同受试者 full_candidate_freqs 不一致。')

        if len(selected_freq_values_ref) != len(loaded_subject['selected_freq_values']):
            raise ValueError('不同受试者 selected_freq_values 长度不一致。')

        if not np.allclose(selected_freq_values_ref, loaded_subject['selected_freq_values']):
            raise ValueError('不同受试者 selected_freq_values 不一致。')

        if len(feature_candidate_freqs_ref) != len(loaded_subject['candidate_freqs_for_feature']):
            raise ValueError('不同受试者 candidate_freqs_for_feature 长度不一致。')

        if not np.allclose(feature_candidate_freqs_ref, loaded_subject['candidate_freqs_for_feature']):
            raise ValueError('不同受试者 candidate_freqs_for_feature 不一致。')

    return {
        'srate_ref': srate_ref,
        'full_candidate_freqs_ref': full_candidate_freqs_ref,
        'selected_freq_values_ref': selected_freq_values_ref,
        'feature_candidate_freqs_ref': feature_candidate_freqs_ref
    }


# 为 SVM / LDA 构造“多窗口特征包”。
# 新版：每个 subject 的 .mat 只读取一次，然后在内存里循环不同 window_sec。
def build_multiscale_feature_package_beta(
    base_folder,
    selected_subject_ids,
    selected_chans,
    selected_freq_indices,
    window_sec_list,
    latency_correction_sec=0.13,
    gaze_shift_sec=0.55,
    feature_mode='wide',
    concat_style='by_type',
    use_selected_freqs_as_feature_candidates=True,
    pre_stim_sec=0.5,
    use_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0,
    narrow_neighbor_count=5,
    wide_n_harmonics=5
):
    if isinstance(selected_chans, str):
        selected_chans = [selected_chans]
    selected_chans = [str(ch) for ch in selected_chans]

    loaded_subjects = []
    for subject_id in selected_subject_ids:
        loaded_subject = _load_single_subject_beta_once(
            base_folder=base_folder,
            subject_id=subject_id,
            selected_chans=selected_chans,
            selected_freq_indices=selected_freq_indices,
            use_selected_freqs_as_feature_candidates=use_selected_freqs_as_feature_candidates
        )
        loaded_subjects.append(loaded_subject)

    ref_info = _check_subject_consistency_beta(loaded_subjects)

    data_by_window = {}

    for window_sec in window_sec_list:
        X_list = []
        y_list = []
        meta_rows = []

        for loaded_subject in loaded_subjects:
            subject_id = loaded_subject['subject_id']
            srate = float(loaded_subject['srate'])
            n_blocks = int(loaded_subject['n_blocks'])
            full_candidate_freqs = loaded_subject['full_candidate_freqs']
            candidate_freqs_for_feature = loaded_subject['candidate_freqs_for_feature']

            for local_label, freq_idx in enumerate(selected_freq_indices):
                target_freq = float(full_candidate_freqs[freq_idx])

                for block_idx in range(n_blocks):
                    trial_window = _extract_trial_window_from_loaded_subject_beta(
                        loaded_subject=loaded_subject,
                        block_idx=block_idx,
                        freq_idx=freq_idx,
                        window_sec=window_sec,
                        pre_stim_sec=pre_stim_sec,
                        latency_correction_sec=latency_correction_sec
                    )

                    one_trial_feature_parts = []

                    for ch_row in range(trial_window.shape[0]):
                        one_channel_signal = np.asarray(trial_window[ch_row, :], dtype=float)

                        feature_vector = extract_feature_vector_one_trial_one_channel_beta(
                            one_channel_signal=one_channel_signal,
                            srate=srate,
                            candidate_freqs_for_feature=candidate_freqs_for_feature,
                            feature_mode=feature_mode,
                            concat_style=concat_style,
                            use_bandpass=use_bandpass,
                            lowcut=lowcut,
                            highcut=highcut,
                            filter_order=filter_order,
                            use_hann_window=use_hann_window,
                            use_zero_padding=use_zero_padding,
                            nfft=nfft,
                            pad_total_sec=pad_total_sec,
                            narrow_neighbor_count=narrow_neighbor_count,
                            wide_n_harmonics=wide_n_harmonics
                        )

                        feature_vector = np.asarray(feature_vector, dtype=float).reshape(-1)
                        one_trial_feature_parts.append(feature_vector)

                    trial_feature_vector = np.concatenate(one_trial_feature_parts, axis=0)

                    X_list.append(trial_feature_vector)
                    y_list.append(local_label)

                    meta_rows.append({
                        'subject_id': str(subject_id),
                        'block_idx': int(block_idx),
                        'freq_idx_global': int(freq_idx),
                        'label_local': int(local_label),
                        'target_freq': float(target_freq),
                        'window_sec': float(window_sec),
                        'latency_correction_sec': float(latency_correction_sec)
                    })

        X = np.asarray(X_list, dtype=float)
        y = np.asarray(y_list, dtype=int)
        meta_df = pd.DataFrame(meta_rows)

        decision_time_sec_fixed = float(window_sec + gaze_shift_sec)

        info_dict = {
            'full_candidate_freqs': ref_info['full_candidate_freqs_ref'],
            'selected_freq_values': ref_info['selected_freq_values_ref'],
            'feature_candidate_freqs': ref_info['feature_candidate_freqs_ref'],
            'srate': float(ref_info['srate_ref']),
            'n_channels': int(len(selected_chans)),
            'selected_chans': selected_chans,
            'n_classes': int(len(ref_info['selected_freq_values_ref'])),
            'window_sec': float(window_sec),
            'T_sec': float(window_sec),
            'latency_correction_sec': float(latency_correction_sec),
            'delta_sec': float(latency_correction_sec),
            'gaze_shift_sec': float(gaze_shift_sec),
            'decision_time_sec_fixed': decision_time_sec_fixed,
            'pre_stim_sec': float(pre_stim_sec),
            'feature_mode': str(feature_mode),
            'concat_style': str(concat_style),
            'use_selected_freqs_as_feature_candidates': bool(use_selected_freqs_as_feature_candidates),
            'use_bandpass': bool(use_bandpass),
            'lowcut': float(lowcut),
            'highcut': float(highcut),
            'filter_order': int(filter_order),
            'use_hann_window': bool(use_hann_window),
            'use_zero_padding': bool(use_zero_padding),
            'nfft': None if nfft is None else int(nfft),
            'pad_total_sec': float(pad_total_sec),
            'narrow_neighbor_count': int(narrow_neighbor_count),
            'wide_n_harmonics': int(wide_n_harmonics)
        }

        data_by_window[float(window_sec)] = {
            'X': X,
            'y': y,
            'meta_df': meta_df,
            'info_dict': info_dict
        }

    return {
        'window_sec_list': [float(x) for x in window_sec_list],
        'data_by_window': data_by_window
    }


# 为 CCA / FBCCA 构造“多窗口 trial 包”。
# 新版：每个 subject 的 .mat 只读取一次，然后在内存里循环不同 window_sec。
def build_multiscale_trial_package_beta(
    base_folder,
    selected_subject_ids,
    selected_chans,
    selected_freq_indices,
    window_sec_list,
    latency_correction_sec=0.13,
    gaze_shift_sec=0.55,
    pre_stim_sec=0.5
):
    if isinstance(selected_chans, str):
        selected_chans = [selected_chans]
    selected_chans = [str(ch) for ch in selected_chans]

    loaded_subjects = []
    for subject_id in selected_subject_ids:
        loaded_subject = _load_single_subject_beta_once(
            base_folder=base_folder,
            subject_id=subject_id,
            selected_chans=selected_chans,
            selected_freq_indices=selected_freq_indices,
            use_selected_freqs_as_feature_candidates=True
        )
        loaded_subjects.append(loaded_subject)

    ref_info = _check_subject_consistency_beta(loaded_subjects)

    data_by_window = {}

    for window_sec in window_sec_list:
        trial_list = []
        y_list = []
        meta_rows = []

        for loaded_subject in loaded_subjects:
            subject_id = loaded_subject['subject_id']
            n_blocks = int(loaded_subject['n_blocks'])
            full_candidate_freqs = loaded_subject['full_candidate_freqs']

            for local_label, freq_idx in enumerate(selected_freq_indices):
                target_freq = float(full_candidate_freqs[freq_idx])

                for block_idx in range(n_blocks):
                    trial_window = _extract_trial_window_from_loaded_subject_beta(
                        loaded_subject=loaded_subject,
                        block_idx=block_idx,
                        freq_idx=freq_idx,
                        window_sec=window_sec,
                        pre_stim_sec=pre_stim_sec,
                        latency_correction_sec=latency_correction_sec
                    )

                    trial_window = np.asarray(trial_window, dtype=float)

                    trial_list.append(trial_window)
                    y_list.append(local_label)

                    meta_rows.append({
                        'subject_id': str(subject_id),
                        'block_idx': int(block_idx),
                        'freq_idx_global': int(freq_idx),
                        'label_local': int(local_label),
                        'target_freq': float(target_freq),
                        'window_sec': float(window_sec),
                        'latency_correction_sec': float(latency_correction_sec)
                    })

        y = np.asarray(y_list, dtype=int)
        meta_df = pd.DataFrame(meta_rows)

        decision_time_sec_fixed = float(window_sec + gaze_shift_sec)

        info_dict = {
            'full_candidate_freqs': ref_info['full_candidate_freqs_ref'],
            'selected_freq_values': ref_info['selected_freq_values_ref'],
            'srate': float(ref_info['srate_ref']),
            'n_channels': int(len(selected_chans)),
            'selected_chans': selected_chans,
            'n_classes': int(len(ref_info['selected_freq_values_ref'])),
            'window_sec': float(window_sec),
            'T_sec': float(window_sec),
            'latency_correction_sec': float(latency_correction_sec),
            'delta_sec': float(latency_correction_sec),
            'gaze_shift_sec': float(gaze_shift_sec),
            'decision_time_sec_fixed': decision_time_sec_fixed,
            'pre_stim_sec': float(pre_stim_sec)
        }

        data_by_window[float(window_sec)] = {
            'trial_list': trial_list,
            'y': y,
            'meta_df': meta_df,
            'info_dict': info_dict
        }

    return {
        'window_sec_list': [float(x) for x in window_sec_list],
        'data_by_window': data_by_window
    }


# 汇总“单个 fold 下、trial 级别”的 early-stop 结果。
# 输入是每个 trial 的结果行，输出：
# 1）trial_results_df：每个 trial 的详细结果
# 2）fold_summary_row：这个 fold 的 accuracy / 平均决策时间 / ITR
def summarize_one_fold_nonstable_early_stop_beta(
    trial_result_rows,
    model_name,
    split_mode,
    split_value,
    threshold,
    n_classes
):
    trial_results_df = pd.DataFrame(trial_result_rows).copy()

    if len(trial_results_df) > 0 and 'is_correct' not in trial_results_df.columns:
        trial_results_df['is_correct'] = (
            trial_results_df['y_true'].astype(int).values
            == trial_results_df['y_pred'].astype(int).values
        )

    overall_accuracy = float((trial_results_df['y_true'].values == trial_results_df['y_pred'].values).mean())
    mean_decision_time_sec = float(trial_results_df['decision_time_sec'].mean())
    mean_stop_window_sec = float(trial_results_df['stop_window_sec'].mean())
    early_stop_rate = float(trial_results_df['stopped_early'].mean())

    if 'recognition_runtime_sec' in trial_results_df.columns:
        mean_recognition_runtime_sec = float(trial_results_df['recognition_runtime_sec'].mean())
    else:
        mean_recognition_runtime_sec = 0.0

    if 'decision_time_with_runtime_sec' in trial_results_df.columns:
        mean_decision_time_with_runtime_sec = float(trial_results_df['decision_time_with_runtime_sec'].mean())
    else:
        mean_decision_time_with_runtime_sec = float(mean_decision_time_sec + mean_recognition_runtime_sec)

    itr_bits_per_min = float(compute_itr_bits_per_min(
        n_classes=n_classes,
        accuracy=overall_accuracy,
        decision_time_sec=mean_decision_time_sec
    ))

    itr_with_runtime_bits_per_min = float(compute_itr_bits_per_min(
        n_classes=n_classes,
        accuracy=overall_accuracy,
        decision_time_sec=mean_decision_time_with_runtime_sec
    ))

    fold_summary_row = {
        'model_name': str(model_name),
        'split_mode': str(split_mode),
        'split_value': split_value,
        'threshold': float(threshold) if np.isfinite(threshold) else np.inf,
        'fold_accuracy': overall_accuracy,
        'mean_stop_window_sec': mean_stop_window_sec,
        'mean_decision_time_sec': mean_decision_time_sec,
        'mean_recognition_runtime_sec': mean_recognition_runtime_sec,
        'mean_decision_time_with_runtime_sec': mean_decision_time_with_runtime_sec,
        'early_stop_rate': early_stop_rate,
        'itr_bits_per_min': itr_bits_per_min,
        'itr_with_runtime_bits_per_min': itr_with_runtime_bits_per_min,
        'n_test': int(len(trial_results_df))
    }

    fold_summary_row = _add_fold_identity_to_summary_row_beta(
        fold_summary_row=fold_summary_row,
        trial_results_df=trial_results_df,
        split_mode=split_mode,
        split_value=split_value
    )

    return {
        'trial_results_df': trial_results_df,
        'fold_summary_row': fold_summary_row
    }


# 汇总“整个模型在所有 outer fold 上”的最终结果。
# 输入是每个 fold 的 fold_summary_row 列表，输出 summary_df 和 fold_results_df。
def summarize_outer_folds_nonstable_early_stop_beta(
    fold_summary_rows,
    model_name,
    split_mode,
    n_classes
):
    fold_results_df = pd.DataFrame(fold_summary_rows).copy()

    summary_row = {
        'model_name': str(model_name),
        'split_mode': str(split_mode),
        'mean_fold_accuracy': float(fold_results_df['fold_accuracy'].mean()),
        'std_fold_accuracy': float(fold_results_df['fold_accuracy'].std(ddof=0)),
        'mean_decision_time_sec': float(fold_results_df['mean_decision_time_sec'].mean()),
        'std_decision_time_sec': float(fold_results_df['mean_decision_time_sec'].std(ddof=0)),
        'mean_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].mean()) if 'mean_recognition_runtime_sec' in fold_results_df.columns else 0.0,
        'std_recognition_runtime_sec': float(fold_results_df['mean_recognition_runtime_sec'].std(ddof=0)) if 'mean_recognition_runtime_sec' in fold_results_df.columns else 0.0,
        'mean_decision_time_with_runtime_sec': float(fold_results_df['mean_decision_time_with_runtime_sec'].mean()) if 'mean_decision_time_with_runtime_sec' in fold_results_df.columns else float(fold_results_df['mean_decision_time_sec'].mean()),
        'std_decision_time_with_runtime_sec': float(fold_results_df['mean_decision_time_with_runtime_sec'].std(ddof=0)) if 'mean_decision_time_with_runtime_sec' in fold_results_df.columns else float(fold_results_df['mean_decision_time_sec'].std(ddof=0)),
        'mean_early_stop_rate': float(fold_results_df['early_stop_rate'].mean()),
        'itr_bits_per_min': float(fold_results_df['itr_bits_per_min'].mean()),
        'std_fold_itr_bits_per_min': float(fold_results_df['itr_bits_per_min'].std(ddof=0)),
        'itr_with_runtime_bits_per_min': float(fold_results_df['itr_with_runtime_bits_per_min'].mean()) if 'itr_with_runtime_bits_per_min' in fold_results_df.columns else float(fold_results_df['itr_bits_per_min'].mean()),
        'std_fold_itr_with_runtime_bits_per_min': float(fold_results_df['itr_with_runtime_bits_per_min'].std(ddof=0)) if 'itr_with_runtime_bits_per_min' in fold_results_df.columns else float(fold_results_df['itr_bits_per_min'].std(ddof=0)),
        'n_classes': int(n_classes),
        'n_folds': int(len(fold_results_df))
    }

    summary_df = pd.DataFrame([summary_row])

    return {
        'summary_df': summary_df,
        'fold_results_df': fold_results_df
    }


def generate_candidate_threshold_list_from_inner_fold_caches_beta(
    inner_fold_cache_dict,
    fallback_candidate_threshold_list=None,
    n_thresholds=50,
    q_min=0.01,
    q_max=0.995,
    include_zero=False,
    round_decimals=10
):
    """
    根据当前 outer training blocks 对应的 inner fold cache 中的 margin 分布自动生成候选阈值列表。
    注意：只使用当前 outer training blocks 的 margin，不使用当前 outer test block。
    """
    margin_parts = []

    if isinstance(inner_fold_cache_dict, dict):
        for _, fold_cache in inner_fold_cache_dict.items():
            if not isinstance(fold_cache, dict) or 'margin_matrix' not in fold_cache:
                continue
            margin_array = np.asarray(fold_cache['margin_matrix'], dtype=float).reshape(-1)
            margin_parts.append(margin_array)

    if len(margin_parts) == 0:
        if fallback_candidate_threshold_list is not None:
            return [float(x) for x in fallback_candidate_threshold_list]
        return [0.0]

    margin_values = np.concatenate(margin_parts, axis=0)
    margin_values = margin_values[np.isfinite(margin_values)]
    margin_values = margin_values[margin_values >= 0]

    if len(margin_values) == 0:
        if fallback_candidate_threshold_list is not None:
            return [float(x) for x in fallback_candidate_threshold_list]
        return [0.0]

    n_thresholds = int(n_thresholds)
    if n_thresholds <= 1:
        return [0.0] if include_zero else [float(np.quantile(margin_values, 0.5))]

    n_quantiles = n_thresholds - 1 if include_zero else n_thresholds
    q_min = float(q_min)
    q_max = float(q_max)

    if q_min < 0 or q_max > 1 or q_min >= q_max:
        raise ValueError('q_min/q_max 应满足 0 <= q_min < q_max <= 1。')

    quantile_points = np.linspace(q_min, q_max, n_quantiles)
    thresholds = np.quantile(margin_values, quantile_points)
    thresholds = np.round(thresholds, int(round_decimals))
    thresholds = thresholds[np.isfinite(thresholds)]
    thresholds = thresholds[thresholds >= 0]

    if include_zero:
        thresholds = np.concatenate([[0.0], thresholds])

    candidate_threshold_list = sorted(set([float(x) for x in thresholds.tolist()]))

    if len(candidate_threshold_list) == 0:
        if fallback_candidate_threshold_list is not None:
            return [float(x) for x in fallback_candidate_threshold_list]
        return [0.0]

    return candidate_threshold_list


def _subset_multiscale_feature_package_by_subject_beta(
    multiscale_feature_package,
    target_subject_id
):
    target_subject_id = str(target_subject_id)

    new_data_by_window = {}
    for window_sec in multiscale_feature_package['window_sec_list']:
        one_window_data = multiscale_feature_package['data_by_window'][float(window_sec)]
        meta_df = one_window_data['meta_df']
        subject_mask = meta_df['subject_id'].astype(str).values == target_subject_id

        new_data_by_window[float(window_sec)] = {
            'X': np.asarray(one_window_data['X'][subject_mask], dtype=float),
            'y': np.asarray(one_window_data['y'][subject_mask], dtype=int),
            'meta_df': meta_df.loc[subject_mask].reset_index(drop=True),
            'info_dict': dict(one_window_data['info_dict'])
        }

    return {
        'window_sec_list': [float(x) for x in multiscale_feature_package['window_sec_list']],
        'data_by_window': new_data_by_window
    }


def _subset_multiscale_trial_package_by_subject_beta(
    multiscale_trial_package,
    target_subject_id
):
    target_subject_id = str(target_subject_id)

    new_data_by_window = {}
    for window_sec in multiscale_trial_package['window_sec_list']:
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]
        meta_df = one_window_data['meta_df']
        subject_mask = meta_df['subject_id'].astype(str).values == target_subject_id
        subject_indices = np.where(subject_mask)[0]

        new_data_by_window[float(window_sec)] = {
            'trial_list': [one_window_data['trial_list'][idx] for idx in subject_indices],
            'y': np.asarray(one_window_data['y'][subject_mask], dtype=int),
            'meta_df': meta_df.loc[subject_mask].reset_index(drop=True),
            'info_dict': dict(one_window_data['info_dict'])
        }

    return {
        'window_sec_list': [float(x) for x in multiscale_trial_package['window_sec_list']],
        'data_by_window': new_data_by_window
    }


def _aggregate_subject_level_outer_cv_results_beta(
    subject_result_list,
    model_name,
    split_mode
):
    subject_summary_df = pd.concat(
        [one_result['summary_df'].copy() for one_result in subject_result_list],
        axis=0,
        ignore_index=True
    )

    fold_results_df = pd.concat(
        [one_result['fold_results_df'].copy() for one_result in subject_result_list],
        axis=0,
        ignore_index=True
    )

    first_summary_row = subject_summary_df.iloc[0].to_dict()

    summary_row = dict(first_summary_row)
    summary_row['model_name'] = str(model_name)
    summary_row['split_mode'] = str(split_mode)
    summary_row['mean_fold_accuracy'] = float(subject_summary_df['mean_fold_accuracy'].mean())
    summary_row['std_fold_accuracy'] = float(subject_summary_df['mean_fold_accuracy'].std(ddof=0))
    summary_row['mean_decision_time_sec'] = float(subject_summary_df['mean_decision_time_sec'].mean())
    summary_row['std_decision_time_sec'] = float(subject_summary_df['mean_decision_time_sec'].std(ddof=0))
    if 'mean_recognition_runtime_sec' in subject_summary_df.columns:
        summary_row['mean_recognition_runtime_sec'] = float(subject_summary_df['mean_recognition_runtime_sec'].mean())
        summary_row['std_recognition_runtime_sec'] = float(subject_summary_df['mean_recognition_runtime_sec'].std(ddof=0))
    if 'mean_decision_time_with_runtime_sec' in subject_summary_df.columns:
        summary_row['mean_decision_time_with_runtime_sec'] = float(subject_summary_df['mean_decision_time_with_runtime_sec'].mean())
        summary_row['std_decision_time_with_runtime_sec'] = float(subject_summary_df['mean_decision_time_with_runtime_sec'].std(ddof=0))
    summary_row['mean_early_stop_rate'] = float(subject_summary_df['mean_early_stop_rate'].mean())
    summary_row['itr_bits_per_min'] = float(subject_summary_df['itr_bits_per_min'].mean())
    summary_row['std_fold_itr_bits_per_min'] = float(subject_summary_df['itr_bits_per_min'].std(ddof=0))
    if 'itr_with_runtime_bits_per_min' in subject_summary_df.columns:
        summary_row['itr_with_runtime_bits_per_min'] = float(subject_summary_df['itr_with_runtime_bits_per_min'].mean())
        summary_row['std_fold_itr_with_runtime_bits_per_min'] = float(subject_summary_df['itr_with_runtime_bits_per_min'].std(ddof=0))

    if 'n_folds' in summary_row and len(fold_results_df) > 0:
        if 'split_value' in fold_results_df.columns:
            summary_row['n_folds'] = int(pd.Series(fold_results_df['split_value']).nunique())

    if 'min_stable_windows' in subject_summary_df.columns:
        summary_row['min_stable_windows'] = int(subject_summary_df['min_stable_windows'].iloc[0])

    summary_df = pd.DataFrame([summary_row])

    threshold_search_by_outer_fold = {}
    for one_result in subject_result_list:
        subject_key = None
        if 'fold_results_df' in one_result and len(one_result['fold_results_df']) > 0:
            if 'subject_id' in one_result['fold_results_df'].columns:
                subject_key = str(one_result['fold_results_df']['subject_id'].iloc[0])
        if subject_key is None:
            subject_key = f'subject_{len(threshold_search_by_outer_fold)}'
        threshold_search_by_outer_fold[subject_key] = one_result.get('threshold_search_by_outer_fold', {})

    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for one_result in subject_result_list:
        subject_key = None
        if 'fold_results_df' in one_result and len(one_result['fold_results_df']) > 0:
            if 'subject_id' in one_result['fold_results_df'].columns:
                subject_key = str(one_result['fold_results_df']['subject_id'].iloc[0])
        if subject_key is None:
            subject_key = f'subject_{len(inner_fold_cache_by_outer_fold)}'

        inner_fold_cache_by_outer_fold[subject_key] = one_result.get('inner_fold_cache_by_outer_fold', {})
        outer_fold_cache_by_outer_fold[subject_key] = one_result.get('outer_fold_cache_by_outer_fold', {})

    return {
        'summary_df': summary_df,
        'fold_results_df': fold_results_df,
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'subject_summary_df': subject_summary_df,
        'subject_result_list': subject_result_list,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }


### 统一的分数、概率、margin 处理函数

In [ ]:
# 将任意 score_vector 转成“类似概率”的向量。
# 对 SVM / LDA，如果本身已经是概率，就直接使用；
# 对 CCA / FBCCA，则使用 softmax 做归一化。
def score_vector_to_prob_like_beta(
    score_vector,
    model_name='GENERIC'
):
    score_vector = np.asarray(score_vector, dtype=float).reshape(-1)
    model_name_upper = str(model_name).upper()

    if model_name_upper in ['SVM', 'LDA']:
        sum_value = np.sum(score_vector)
        is_prob_like = (
            np.all(score_vector >= -1e-8) and
            np.all(score_vector <= 1.0 + 1e-8) and
            np.isclose(sum_value, 1.0, atol=1e-4)
        )
        if is_prob_like:
            return score_vector.copy()
        return softmax_score_vector_beta(score_vector)

    return softmax_score_vector_beta(score_vector)


# 从一个分数向量或概率向量中，统一得到：
# 1）预测类别 pred_label
# 2）margin_value = top1 - top2
# 3）prob_like_vector
def get_pred_margin_from_vector_beta(
    vector,
    model_name='GENERIC',
    vector_type='score'
):
    vector = np.asarray(vector, dtype=float).reshape(-1)

    if vector_type == 'prob':
        prob_like_vector = vector.copy()
    elif vector_type == 'score':
        prob_like_vector = score_vector_to_prob_like_beta(
            score_vector=vector,
            model_name=model_name
        )
    else:
        raise ValueError("vector_type 只能写 'score' 或 'prob'。")

    pred_label = int(np.argmax(prob_like_vector))
    margin_value = float(compute_margin_from_prob_vector_beta(prob_like_vector))

    return {
        'pred_label': pred_label,
        'margin_value': margin_value,
        'prob_like_vector': prob_like_vector
    }

### 四个模型的“当前窗口预测”函数

In [ ]:
from sklearn.base import clone


# 作用：
# 用于 SVM，在“当前窗口”下对单个样本做预测。
# 这里修复的关键点是：
# 最终类别 pred_label 不再用 argmax(prob_vector)，
# 而是直接使用 trained_model.predict(x_row)[0]。
# 概率或分数只用于计算 margin，不用于决定最终类别。
def predict_one_window_svm_beta(
    trained_model,
    x_row
):
    x_row = np.asarray(x_row, dtype=float).reshape(1, -1)

    # 最终类别：统一交给模型自己的 predict()
    pred_label_final = int(trained_model.predict(x_row)[0])

    # SVM 早停置信度：使用 predict_proba 的前两名概率差。
    if not hasattr(trained_model, 'predict_proba'):
        raise ValueError('当前 SVM 模型没有 predict_proba，无法用概率差计算 margin。')

    prob_vector = np.asarray(trained_model.predict_proba(x_row), dtype=float).reshape(-1)

    if prob_vector.shape[0] == 1:
        margin_value = 1.0
    else:
        sorted_probs = np.sort(prob_vector)
        margin_value = float(sorted_probs[-1] - sorted_probs[-2])

    return {
        'pred_label': pred_label_final,
        'margin_value': float(margin_value),
        'prob_like_vector': prob_vector,
        'raw_score_vector': None,
        'pred_label_from_prob_argmax': int(np.argmax(prob_vector))
    }

def predict_one_window_lda_beta(
    trained_model,
    x_row
):
    x_row = np.asarray(x_row, dtype=float).reshape(1, -1)

    # 最终类别：统一交给模型自己的 predict()
    pred_label_final = int(trained_model.predict(x_row)[0])

    # LDA 早停置信度：使用 predict_proba 的前两名概率差。
    if not hasattr(trained_model, 'predict_proba'):
        raise ValueError('当前 LDA 模型没有 predict_proba，无法用概率差计算 margin。')

    prob_vector = np.asarray(trained_model.predict_proba(x_row), dtype=float).reshape(-1)

    if prob_vector.shape[0] == 1:
        margin_value = 1.0
    else:
        sorted_probs = np.sort(prob_vector)
        margin_value = float(sorted_probs[-1] - sorted_probs[-2])

    return {
        'pred_label': pred_label_final,
        'margin_value': float(margin_value),
        'prob_like_vector': prob_vector,
        'raw_score_vector': None,
        'pred_label_from_prob_argmax': int(np.argmax(prob_vector))
    }

def prepare_cca_reference_cache_beta(
    multiscale_trial_package,
    n_harmonics=5
):
    reference_cache = {}

    for window_sec in multiscale_trial_package['window_sec_list']:
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]
        info_dict = one_window_data['info_dict']
        trial_list = one_window_data['trial_list']

        n_samples = int(trial_list[0].shape[1])
        srate = float(info_dict['srate'])
        selected_freq_values = info_dict['selected_freq_values']

        reference_signal_list = generate_cca_reference_signals_beta(
            candidate_freqs=selected_freq_values,
            srate=srate,
            n_samples=n_samples,
            n_harmonics=n_harmonics
        )

        reference_cache[float(window_sec)] = reference_signal_list

    return reference_cache


# 为 FBCCA 的每个窗口预先构造 reference cache，避免重复生成参考信号。
def prepare_fbcca_reference_cache_beta(
    multiscale_trial_package,
    n_harmonics=5
):
    reference_cache = {}

    for window_sec in multiscale_trial_package['window_sec_list']:
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]
        info_dict = one_window_data['info_dict']
        trial_list = one_window_data['trial_list']

        n_samples = int(trial_list[0].shape[1])
        srate = float(info_dict['srate'])
        selected_freq_values = info_dict['selected_freq_values']

        reference_signal_list = generate_fbcca_reference_signals_beta(
            candidate_freqs=selected_freq_values,
            srate=srate,
            n_samples=n_samples,
            n_harmonics=n_harmonics
        )

        reference_cache[float(window_sec)] = reference_signal_list

    return reference_cache


# 用于 CCA，在“当前窗口”下对单个 trial 做预测，
# 返回 pred_label、margin_value、raw_score_vector。
def predict_one_window_cca_beta(
    trial_eeg,
    reference_signal_list,
    srate,
    apply_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4
):
    trial_eeg = np.asarray(trial_eeg, dtype=float)

    if apply_bandpass:
        trial_eeg = bandpass_filter_multichannel_signal_beta(
            multichannel_signal=trial_eeg,
            srate=srate,
            lowcut=lowcut,
            highcut=highcut,
            filter_order=filter_order
        )

    pred_label, cca_score_array = predict_one_trial_cca_beta(
        trial_eeg=trial_eeg,
        reference_signal_list=reference_signal_list
    )

    result_dict = get_pred_margin_from_vector_beta(
        vector=cca_score_array,
        model_name='CCA',
        vector_type='score'
    )
    result_dict['raw_score_vector'] = np.asarray(cca_score_array, dtype=float)
    result_dict['pred_label_raw'] = int(pred_label)

    return result_dict


# 用于 FBCCA，在“当前窗口”下对单个 trial 做预测，
# 返回 pred_label、margin_value、raw_score_vector。
def predict_one_window_fbcca_beta(
    trial_eeg,
    srate,
    reference_signal_list,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25
):
    trial_eeg = np.asarray(trial_eeg, dtype=float)

    pred_label, fbcca_score_array = predict_one_trial_fbcca_beta(
        trial_eeg=trial_eeg,
        srate=srate,
        reference_signal_list=reference_signal_list,
        subband_low_list=subband_low_list,
        subband_highcut=subband_highcut,
        filter_order=filter_order,
        weight_a=weight_a,
        weight_b=weight_b
    )

    result_dict = get_pred_margin_from_vector_beta(
        vector=fbcca_score_array,
        model_name='FBCCA',
        vector_type='score'
    )
    result_dict['raw_score_vector'] = np.asarray(fbcca_score_array, dtype=float)
    result_dict['pred_label_raw'] = int(pred_label)

    return result_dict

### 单个 trial 的非稳定性早停函数

In [ ]:
# 对单个 trial 执行“非稳定性早停”。
# 规则是：
# 1）从最短窗口开始依次预测；
# 2）如果当前窗口的 margin >= threshold，就立刻停止；
# 3）否则继续看更长窗口；
# 4）如果最后一个窗口仍未达到阈值，则强制输出最后一个窗口的结果。
def run_nonstable_early_stop_one_trial_beta(
    window_sec_list,
    predict_func_for_one_window,
    threshold,
    gaze_shift_sec=0.55
):
    last_result_dict = None

    for window_sec in window_sec_list:
        current_result_dict = predict_func_for_one_window(float(window_sec)).copy()
        current_result_dict['stop_window_sec'] = float(window_sec)
        current_result_dict['decision_time_sec'] = float(window_sec + gaze_shift_sec)

        last_result_dict = current_result_dict

        if float(current_result_dict['margin_value']) >= float(threshold):
            output_dict = current_result_dict.copy()
            output_dict['stopped_early'] = True
            output_dict['stop_reason'] = 'margin_reached'
            return output_dict

    output_dict = last_result_dict.copy()
    output_dict['stopped_early'] = False
    output_dict['stop_reason'] = 'forced_at_last_window'
    return output_dict

### 单个fold的非稳定性早停评估函数

In [ ]:
# 在给定 train_mask 的情况下，为每个窗口训练一个 SVM 模型。
# 因为不同窗口的特征维度/数值不同，所以每个窗口都要单独训练一个模型。
def fit_svm_models_by_window_beta(
    multiscale_feature_package,
    train_mask,
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
):
    model_by_window = {}

    for window_sec in multiscale_feature_package['window_sec_list']:
        one_window_data = multiscale_feature_package['data_by_window'][float(window_sec)]
        X = one_window_data['X']
        y = one_window_data['y']

        current_model = define_svm_model_beta(
            C=C,
            kernel=kernel,
            gamma=gamma,
            probability=probability,
            random_state=random_state
        )
        current_model.fit(X[train_mask], y[train_mask])

        model_by_window[float(window_sec)] = current_model

    return model_by_window

# 在给定 train_mask 的情况下，为每个窗口训练一个 LDA 模型。
def fit_lda_models_by_window_beta(
    multiscale_feature_package,
    train_mask,
    solver='lsqr',
    shrinkage='auto'
):
    model_by_window = {}

    for window_sec in multiscale_feature_package['window_sec_list']:
        one_window_data = multiscale_feature_package['data_by_window'][float(window_sec)]
        X = one_window_data['X']
        y = one_window_data['y']

        current_model = define_lda_model_beta(
            solver=solver,
            shrinkage=shrinkage
        )
        current_model.fit(X[train_mask], y[train_mask])

        model_by_window[float(window_sec)] = current_model

    return model_by_window
# 已清理：旧版 evaluate_one_fold_nonstable_early_stop_svm_beta 已删除，统一使用后面的 cache 版本。
# 已清理：旧版 evaluate_one_fold_nonstable_early_stop_lda_beta 已删除，统一使用后面的 cache 版本。
# 已清理：旧版 evaluate_one_fold_nonstable_early_stop_cca_beta 已删除，统一使用后面的 cache 版本。
# 已清理：旧版 evaluate_one_fold_nonstable_early_stop_fbcca_beta 已删除，统一使用后面的 cache 版本。


### 阈值搜索函数

In [ ]:
def search_best_threshold_nonstable_early_stop_beta(
    inner_split_values,
    candidate_threshold_list,
    evaluate_one_inner_fold_func,
    baseline_accuracy=None,
    max_accuracy_drop=None
):
    search_rows = []
    inner_fold_search_rows = []

    if baseline_accuracy is not None and max_accuracy_drop is not None:
        min_required_accuracy = float(baseline_accuracy) - float(max_accuracy_drop)
    else:
        min_required_accuracy = None

    for threshold_order, threshold in enumerate(candidate_threshold_list):
        one_threshold_fold_rows = []

        for inner_val_split_value in inner_split_values:
            fold_result = evaluate_one_inner_fold_func(
                inner_val_split_value=inner_val_split_value,
                threshold=threshold
            )

            fold_summary_row = fold_result['fold_summary_row'].copy()
            fold_summary_row['threshold'] = float(threshold)
            fold_summary_row['threshold_order'] = int(threshold_order)
            fold_summary_row['inner_val_split_value'] = inner_val_split_value

            one_threshold_fold_rows.append(fold_summary_row)
            inner_fold_search_rows.append(fold_summary_row)

        one_threshold_df = pd.DataFrame(one_threshold_fold_rows)

        mean_fold_accuracy = float(one_threshold_df['fold_accuracy'].mean())
        mean_decision_time_sec = float(one_threshold_df['mean_decision_time_sec'].mean())
        mean_recognition_runtime_sec = float(one_threshold_df['mean_recognition_runtime_sec'].mean()) if 'mean_recognition_runtime_sec' in one_threshold_df.columns else 0.0
        mean_decision_time_with_runtime_sec = float(one_threshold_df['mean_decision_time_with_runtime_sec'].mean()) if 'mean_decision_time_with_runtime_sec' in one_threshold_df.columns else float(mean_decision_time_sec + mean_recognition_runtime_sec)
        mean_early_stop_rate = float(one_threshold_df['early_stop_rate'].mean())
        mean_itr_bits_per_min = float(one_threshold_df['itr_bits_per_min'].mean())
        mean_itr_with_runtime_bits_per_min = float(one_threshold_df['itr_with_runtime_bits_per_min'].mean()) if 'itr_with_runtime_bits_per_min' in one_threshold_df.columns else mean_itr_bits_per_min

        if min_required_accuracy is None:
            passes_accuracy_constraint = True
        else:
            passes_accuracy_constraint = bool(mean_fold_accuracy >= min_required_accuracy)

        search_rows.append({
            'threshold': float(threshold),
            'threshold_order': int(threshold_order),
            'mean_fold_accuracy': mean_fold_accuracy,
            'mean_decision_time_sec': mean_decision_time_sec,
            'mean_recognition_runtime_sec': mean_recognition_runtime_sec,
            'mean_decision_time_with_runtime_sec': mean_decision_time_with_runtime_sec,
            'mean_early_stop_rate': mean_early_stop_rate,
            'mean_itr_bits_per_min': mean_itr_bits_per_min,
            'mean_itr_with_runtime_bits_per_min': mean_itr_with_runtime_bits_per_min,
            'baseline_accuracy': float(baseline_accuracy) if baseline_accuracy is not None else np.nan,
            'min_required_accuracy': float(min_required_accuracy) if min_required_accuracy is not None else np.nan,
            'passes_accuracy_constraint': passes_accuracy_constraint
        })

    search_df = pd.DataFrame(search_rows)
    inner_fold_search_df = pd.DataFrame(inner_fold_search_rows)

    # 额外记录：每个内层验证 Block 如果单独按 ITR 选阈值，会选中哪个阈值。
    # 这只用于后续统计阈值出现次数，不改变原来的三折平均阈值选择逻辑。
    if len(inner_fold_search_df) > 0:
        inner_best_threshold_df = inner_fold_search_df.sort_values(
            by=['inner_val_split_value', 'itr_bits_per_min', 'fold_accuracy', 'threshold_order'],
            ascending=[True, False, False, True],
            kind='mergesort'
        ).groupby('inner_val_split_value', as_index=False).head(1).reset_index(drop=True)

        inner_best_count_series = inner_best_threshold_df.groupby('threshold').size()
        inner_best_block_series = inner_best_threshold_df.groupby('threshold')['inner_val_split_value'].apply(
            lambda values: ', '.join([str(x) for x in values.tolist()])
        )

        search_df['inner_best_threshold_count'] = search_df['threshold'].map(inner_best_count_series).fillna(0).astype(int)
        search_df['inner_best_threshold_blocks'] = search_df['threshold'].map(inner_best_block_series).fillna('')
    else:
        inner_best_threshold_df = pd.DataFrame()
        search_df['inner_best_threshold_count'] = 0
        search_df['inner_best_threshold_blocks'] = ''

    if min_required_accuracy is None:
        valid_df = search_df.copy()
        selection_mode = 'best_itr_without_accuracy_constraint'
    else:
        valid_df = search_df.loc[search_df['passes_accuracy_constraint']].copy()
        selection_mode = 'best_itr_under_accuracy_constraint'

    if len(valid_df) > 0:
        valid_df = valid_df.sort_values(
            by=['mean_itr_bits_per_min', 'mean_fold_accuracy'],
            ascending=[False, False]
        ).reset_index(drop=True)

        best_threshold = float(valid_df.loc[0, 'threshold'])
    else:
        best_threshold = np.inf
        selection_mode = 'fallback_to_no_early_stop'

    search_df = search_df.sort_values(
        by=['passes_accuracy_constraint', 'mean_itr_bits_per_min', 'mean_fold_accuracy'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return {
        'best_threshold': best_threshold,
        'threshold_search_df': search_df,
        'valid_threshold_df': valid_df,
        'inner_fold_threshold_search_df': inner_fold_search_df,
        'inner_best_threshold_df': inner_best_threshold_df,
        'baseline_accuracy': float(baseline_accuracy) if baseline_accuracy is not None else np.nan,
        'min_required_accuracy': float(min_required_accuracy) if min_required_accuracy is not None else np.nan,
        'selection_mode': selection_mode
    }

### 四个模型的 outer 实验主函数

In [ ]:
# 本 cell 中旧版 run_nonstable_early_stop_experiment_* 主函数已清理。
# 统一使用后面保留的基于 prediction cache 的最终版本：
# - SVM / LDA：见下一 cell
# - CCA / FBCCA：见再下一 cell


In [ ]:
def predict_many_window_svm_beta(
    trained_model,
    X_rows
):
    X_rows = np.asarray(X_rows, dtype=float)

    pred_label_array = np.asarray(
        trained_model.predict(X_rows),
        dtype=int
    ).reshape(-1)

    # SVM 早停 margin 使用 predict_proba 的前两名概率差。
    if not hasattr(trained_model, 'predict_proba'):
        raise ValueError('当前 SVM 模型没有 predict_proba，无法用概率差计算 margin。')

    prob_matrix = np.asarray(trained_model.predict_proba(X_rows), dtype=float)

    if prob_matrix.ndim == 1:
        prob_matrix = prob_matrix.reshape(-1, 1)

    if prob_matrix.shape[1] == 1:
        margin_array = np.ones(prob_matrix.shape[0], dtype=float)
    else:
        sorted_probs = np.sort(prob_matrix, axis=1)
        margin_array = sorted_probs[:, -1] - sorted_probs[:, -2]

    return {
        'pred_label_array': pred_label_array,
        'margin_array': np.asarray(margin_array, dtype=float),
        'prob_like_matrix': np.asarray(prob_matrix, dtype=float),
        'raw_score_matrix': None
    }

def predict_many_window_lda_beta(
    trained_model,
    X_rows
):
    X_rows = np.asarray(X_rows, dtype=float)

    pred_label_array = np.asarray(
        trained_model.predict(X_rows),
        dtype=int
    ).reshape(-1)

    # LDA 早停 margin 使用 predict_proba 的前两名概率差。
    if not hasattr(trained_model, 'predict_proba'):
        raise ValueError('当前 LDA 模型没有 predict_proba，无法用概率差计算 margin。')

    prob_matrix = np.asarray(trained_model.predict_proba(X_rows), dtype=float)

    if prob_matrix.ndim == 1:
        prob_matrix = prob_matrix.reshape(-1, 1)

    if prob_matrix.shape[1] == 1:
        margin_array = np.ones(prob_matrix.shape[0], dtype=float)
    else:
        sorted_probs = np.sort(prob_matrix, axis=1)
        margin_array = sorted_probs[:, -1] - sorted_probs[:, -2]

    return {
        'pred_label_array': pred_label_array,
        'margin_array': np.asarray(margin_array, dtype=float),
        'prob_like_matrix': np.asarray(prob_matrix, dtype=float),
        'raw_score_matrix': None
    }

def evaluate_nonstable_early_stop_from_cache_beta(
    fold_cache,
    threshold,
    model_name
):
    pred_label_matrix = np.asarray(fold_cache['pred_label_matrix'], dtype=int)
    margin_matrix = np.asarray(fold_cache['margin_matrix'], dtype=float)
    y_true = np.asarray(fold_cache['y_true'], dtype=int)
    sample_indices = np.asarray(fold_cache['sample_indices'], dtype=int)
    window_sec_list = np.asarray(fold_cache['window_sec_list'], dtype=float)
    runtime_matrix = np.asarray(
        fold_cache.get('runtime_matrix', np.zeros_like(margin_matrix, dtype=float)),
        dtype=float
    )
    if runtime_matrix.shape != margin_matrix.shape:
        runtime_matrix = np.zeros_like(margin_matrix, dtype=float)
    cumulative_runtime_matrix = np.cumsum(runtime_matrix, axis=1)

    n_samples, n_windows = margin_matrix.shape

    reached_matrix = margin_matrix >= float(threshold)
    has_reached = reached_matrix.any(axis=1)

    first_reach_idx = np.argmax(reached_matrix, axis=1)
    stop_window_idx = np.where(has_reached, first_reach_idx, n_windows - 1)

    row_idx = np.arange(n_samples)

    y_pred = pred_label_matrix[row_idx, stop_window_idx]
    selected_margin = margin_matrix[row_idx, stop_window_idx]
    stop_window_sec = window_sec_list[stop_window_idx]
    decision_time_sec = stop_window_sec + float(fold_cache['gaze_shift_sec'])
    recognition_runtime_sec = cumulative_runtime_matrix[row_idx, stop_window_idx]
    decision_time_with_runtime_sec = decision_time_sec + recognition_runtime_sec
    stop_reason = np.where(has_reached, 'margin_reached', 'forced_at_last_window')

    trial_results_df = pd.DataFrame({
        'sample_idx': sample_indices.astype(int),
        'y_true': y_true.astype(int),
        'y_pred': y_pred.astype(int),
        'margin_value': selected_margin.astype(float),
        'stop_window_sec': stop_window_sec.astype(float),
        'decision_time_sec': decision_time_sec.astype(float),
        'recognition_runtime_sec': recognition_runtime_sec.astype(float),
        'decision_time_with_runtime_sec': decision_time_with_runtime_sec.astype(float),
        'stopped_early': has_reached.astype(bool),
        'stop_reason': stop_reason
    })

    trial_results_df = _add_fold_cache_metadata_to_trial_results_beta(
        trial_results_df=trial_results_df,
        fold_cache=fold_cache
    )

    return summarize_one_fold_nonstable_early_stop_beta(
        trial_result_rows=trial_results_df.to_dict('records'),
        model_name=model_name,
        split_mode=fold_cache['split_mode'],
        split_value=fold_cache['split_value'],
        threshold=threshold,
        n_classes=fold_cache['n_classes']
    )


def build_svm_fold_prediction_cache_beta(
    multiscale_feature_package,
    split_mode,
    test_split_value,
    threshold=None,
    allowed_split_values=None,
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
):
    first_window_sec = float(multiscale_feature_package['window_sec_list'][0])
    first_window_data = multiscale_feature_package['data_by_window'][first_window_sec]

    y = first_window_data['y']
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    split_key = get_split_key_beta(split_mode=split_mode)

    if allowed_split_values is None:
        subset_mask = np.ones(len(meta_df), dtype=bool)
    else:
        subset_mask = meta_df[split_key].isin(list(allowed_split_values)).values

    test_mask = subset_mask & (meta_df[split_key].values == test_split_value)
    train_mask = subset_mask & (meta_df[split_key].values != test_split_value)

    model_by_window = fit_svm_models_by_window_beta(
        multiscale_feature_package=multiscale_feature_package,
        train_mask=train_mask,
        C=C,
        kernel=kernel,
        gamma=gamma,
        probability=probability,
        random_state=random_state
    )

    window_sec_list = [float(x) for x in multiscale_feature_package['window_sec_list']]
    test_indices = np.where(test_mask)[0]

    pred_label_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=int)
    margin_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=float)
    runtime_matrix = np.zeros((len(test_indices), len(window_sec_list)), dtype=float)

    for w_idx, window_sec in enumerate(window_sec_list):
        one_window_data = multiscale_feature_package['data_by_window'][float(window_sec)]
        X_window = one_window_data['X']
        trained_model = model_by_window[float(window_sec)]

        _runtime_start = time.perf_counter()
        batch_result = predict_many_window_svm_beta(
            trained_model=trained_model,
            X_rows=X_window[test_indices]
        )
        _batch_runtime_sec = float(time.perf_counter() - _runtime_start)
        _per_sample_runtime_sec = _batch_runtime_sec / max(len(test_indices), 1)

        runtime_matrix[:, w_idx] = _per_sample_runtime_sec
        pred_label_matrix[:, w_idx] = batch_result['pred_label_array']
        margin_matrix[:, w_idx] = batch_result['margin_array']

    return {
        'pred_label_matrix': pred_label_matrix,
        'margin_matrix': margin_matrix,
        'runtime_matrix': runtime_matrix,
        'y_true': y[test_indices],
        'sample_indices': test_indices,
        'test_meta_df': meta_df.iloc[test_indices].reset_index(drop=True),
        'window_sec_list': window_sec_list,
        'gaze_shift_sec': float(info_dict.get('gaze_shift_sec', 0.55)),
        'n_classes': int(info_dict['n_classes']),
        'split_mode': split_mode,
        'split_value': test_split_value
    }


def build_lda_fold_prediction_cache_beta(
    multiscale_feature_package,
    split_mode,
    test_split_value,
    threshold=None,
    allowed_split_values=None,
    solver='lsqr',
    shrinkage='auto'
):
    first_window_sec = float(multiscale_feature_package['window_sec_list'][0])
    first_window_data = multiscale_feature_package['data_by_window'][first_window_sec]

    y = first_window_data['y']
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    split_key = get_split_key_beta(split_mode=split_mode)

    if allowed_split_values is None:
        subset_mask = np.ones(len(meta_df), dtype=bool)
    else:
        subset_mask = meta_df[split_key].isin(list(allowed_split_values)).values

    test_mask = subset_mask & (meta_df[split_key].values == test_split_value)
    train_mask = subset_mask & (meta_df[split_key].values != test_split_value)

    model_by_window = fit_lda_models_by_window_beta(
        multiscale_feature_package=multiscale_feature_package,
        train_mask=train_mask,
        solver=solver,
        shrinkage=shrinkage
    )

    window_sec_list = [float(x) for x in multiscale_feature_package['window_sec_list']]
    test_indices = np.where(test_mask)[0]

    pred_label_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=int)
    margin_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=float)
    runtime_matrix = np.zeros((len(test_indices), len(window_sec_list)), dtype=float)

    for w_idx, window_sec in enumerate(window_sec_list):
        one_window_data = multiscale_feature_package['data_by_window'][float(window_sec)]
        X_window = one_window_data['X']
        trained_model = model_by_window[float(window_sec)]

        _runtime_start = time.perf_counter()
        batch_result = predict_many_window_lda_beta(
            trained_model=trained_model,
            X_rows=X_window[test_indices]
        )
        _batch_runtime_sec = float(time.perf_counter() - _runtime_start)
        _per_sample_runtime_sec = _batch_runtime_sec / max(len(test_indices), 1)

        runtime_matrix[:, w_idx] = _per_sample_runtime_sec
        pred_label_matrix[:, w_idx] = batch_result['pred_label_array']
        margin_matrix[:, w_idx] = batch_result['margin_array']

    return {
        'pred_label_matrix': pred_label_matrix,
        'margin_matrix': margin_matrix,
        'runtime_matrix': runtime_matrix,
        'y_true': y[test_indices],
        'sample_indices': test_indices,
        'test_meta_df': meta_df.iloc[test_indices].reset_index(drop=True),
        'window_sec_list': window_sec_list,
        'gaze_shift_sec': float(info_dict.get('gaze_shift_sec', 0.55)),
        'n_classes': int(info_dict['n_classes']),
        'split_mode': split_mode,
        'split_value': test_split_value
    }


def evaluate_one_fold_nonstable_early_stop_svm_beta(
    multiscale_feature_package,
    split_mode,
    test_split_value,
    threshold,
    allowed_split_values=None,
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
):
    fold_cache = build_svm_fold_prediction_cache_beta(
        multiscale_feature_package=multiscale_feature_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        C=C,
        kernel=kernel,
        gamma=gamma,
        probability=probability,
        random_state=random_state
    )

    return evaluate_nonstable_early_stop_from_cache_beta(
        fold_cache=fold_cache,
        threshold=threshold,
        model_name='SVM'
    )


def evaluate_one_fold_nonstable_early_stop_lda_beta(
    multiscale_feature_package,
    split_mode,
    test_split_value,
    threshold,
    allowed_split_values=None,
    solver='lsqr',
    shrinkage='auto'
):
    fold_cache = build_lda_fold_prediction_cache_beta(
        multiscale_feature_package=multiscale_feature_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        solver=solver,
        shrinkage=shrinkage
    )

    return evaluate_nonstable_early_stop_from_cache_beta(
        fold_cache=fold_cache,
        threshold=threshold,
        model_name='LDA'
    )


def run_nonstable_early_stop_experiment_svm_beta(
    multiscale_feature_package,
    candidate_threshold_list,
    split_mode='block',
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42,
    max_accuracy_drop=None
):
    first_window_sec = float(multiscale_feature_package['window_sec_list'][0])
    first_window_data = multiscale_feature_package['data_by_window'][first_window_sec]
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            one_subject_package = _subset_multiscale_feature_package_by_subject_beta(
                multiscale_feature_package=multiscale_feature_package,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_nonstable_early_stop_experiment_svm_beta(
                multiscale_feature_package=one_subject_package,
                candidate_threshold_list=candidate_threshold_list,
                split_mode=split_mode,
                C=C,
                kernel=kernel,
                gamma=gamma,
                probability=probability,
                random_state=random_state,
                max_accuracy_drop=max_accuracy_drop
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name='SVM',
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}
    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        inner_fold_cache_dict = {}

        for inner_val_split_value in outer_train_split_values:
            inner_fold_cache_dict[inner_val_split_value] = build_svm_fold_prediction_cache_beta(
                multiscale_feature_package=multiscale_feature_package,
                split_mode=split_mode,
                test_split_value=inner_val_split_value,
                allowed_split_values=outer_train_split_values,
                C=C,
                kernel=kernel,
                gamma=gamma,
                probability=probability,
                random_state=random_state
            )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name='SVM'
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name='SVM'
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = 'SVM'
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df
        inner_fold_cache_by_outer_fold[outer_test_split_value] = inner_fold_cache_dict

        outer_fold_cache = build_svm_fold_prediction_cache_beta(
            multiscale_feature_package=multiscale_feature_package,
            split_mode=split_mode,
            test_split_value=outer_test_split_value,
            allowed_split_values=None,
            C=C,
            kernel=kernel,
            gamma=gamma,
            probability=probability,
            random_state=random_state
        )

        outer_fold_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name='SVM'
        )

        outer_fold_cache_by_outer_fold[outer_test_split_value] = outer_fold_cache

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name='SVM',
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }


def run_nonstable_early_stop_experiment_lda_beta(
    multiscale_feature_package,
    candidate_threshold_list,
    split_mode='block',
    solver='lsqr',
    shrinkage='auto',
    max_accuracy_drop=None
):
    first_window_sec = float(multiscale_feature_package['window_sec_list'][0])
    first_window_data = multiscale_feature_package['data_by_window'][first_window_sec]
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            one_subject_package = _subset_multiscale_feature_package_by_subject_beta(
                multiscale_feature_package=multiscale_feature_package,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_nonstable_early_stop_experiment_lda_beta(
                multiscale_feature_package=one_subject_package,
                candidate_threshold_list=candidate_threshold_list,
                split_mode=split_mode,
                solver=solver,
                shrinkage=shrinkage,
                max_accuracy_drop=max_accuracy_drop
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name='LDA',
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}
    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        inner_fold_cache_dict = {}

        for inner_val_split_value in outer_train_split_values:
            inner_fold_cache_dict[inner_val_split_value] = build_lda_fold_prediction_cache_beta(
                multiscale_feature_package=multiscale_feature_package,
                split_mode=split_mode,
                test_split_value=inner_val_split_value,
                allowed_split_values=outer_train_split_values,
                solver=solver,
                shrinkage=shrinkage
            )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name='LDA'
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name='LDA'
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = 'LDA'
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df
        inner_fold_cache_by_outer_fold[outer_test_split_value] = inner_fold_cache_dict

        outer_fold_cache = build_lda_fold_prediction_cache_beta(
            multiscale_feature_package=multiscale_feature_package,
            split_mode=split_mode,
            test_split_value=outer_test_split_value,
            allowed_split_values=None,
            solver=solver,
            shrinkage=shrinkage
        )

        outer_fold_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name='LDA'
        )

        outer_fold_cache_by_outer_fold[outer_test_split_value] = outer_fold_cache

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name='LDA',
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }

In [ ]:
def build_cca_fold_prediction_cache_beta(
    multiscale_trial_package,
    cca_reference_cache,
    split_mode,
    test_split_value,
    allowed_split_values=None,
    apply_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]

    y = first_window_data['y']
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']
    split_key = get_split_key_beta(split_mode=split_mode)

    if allowed_split_values is None:
        subset_mask = np.ones(len(meta_df), dtype=bool)
    else:
        subset_mask = meta_df[split_key].isin(list(allowed_split_values)).values

    test_mask = subset_mask & (meta_df[split_key].values == test_split_value)
    test_indices = np.where(test_mask)[0]

    window_sec_list = [float(x) for x in multiscale_trial_package['window_sec_list']]

    pred_label_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=int)
    margin_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=float)
    runtime_matrix = np.zeros((len(test_indices), len(window_sec_list)), dtype=float)

    for w_idx, window_sec in enumerate(window_sec_list):
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]
        trial_list = one_window_data['trial_list']
        srate = float(one_window_data['info_dict']['srate'])
        reference_signal_list = cca_reference_cache[float(window_sec)]

        for row_idx, sample_idx in enumerate(test_indices):
            trial_eeg = trial_list[sample_idx]

            _runtime_start = time.perf_counter()
            result_dict = predict_one_window_cca_beta(
                trial_eeg=trial_eeg,
                reference_signal_list=reference_signal_list,
                srate=srate,
                apply_bandpass=apply_bandpass,
                lowcut=lowcut,
                highcut=highcut,
                filter_order=filter_order
            )

            runtime_matrix[row_idx, w_idx] = float(time.perf_counter() - _runtime_start)
            pred_label_matrix[row_idx, w_idx] = int(result_dict['pred_label'])
            margin_matrix[row_idx, w_idx] = float(result_dict['margin_value'])

    return {
        'pred_label_matrix': pred_label_matrix,
        'margin_matrix': margin_matrix,
        'runtime_matrix': runtime_matrix,
        'y_true': y[test_indices],
        'sample_indices': test_indices,
        'test_meta_df': meta_df.iloc[test_indices].reset_index(drop=True),
        'window_sec_list': window_sec_list,
        'gaze_shift_sec': float(info_dict.get('gaze_shift_sec', 0.55)),
        'n_classes': int(info_dict['n_classes']),
        'split_mode': split_mode,
        'split_value': test_split_value
    }


def build_fbcca_fold_prediction_cache_beta(
    multiscale_trial_package,
    fbcca_reference_cache,
    split_mode,
    test_split_value,
    allowed_split_values=None,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]

    y = first_window_data['y']
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']
    split_key = get_split_key_beta(split_mode=split_mode)

    if allowed_split_values is None:
        subset_mask = np.ones(len(meta_df), dtype=bool)
    else:
        subset_mask = meta_df[split_key].isin(list(allowed_split_values)).values

    test_mask = subset_mask & (meta_df[split_key].values == test_split_value)
    test_indices = np.where(test_mask)[0]

    window_sec_list = [float(x) for x in multiscale_trial_package['window_sec_list']]

    pred_label_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=int)
    margin_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=float)
    runtime_matrix = np.zeros((len(test_indices), len(window_sec_list)), dtype=float)

    for w_idx, window_sec in enumerate(window_sec_list):
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]
        trial_list = one_window_data['trial_list']
        srate = float(one_window_data['info_dict']['srate'])
        reference_signal_list = fbcca_reference_cache[float(window_sec)]

        for row_idx, sample_idx in enumerate(test_indices):
            trial_eeg = trial_list[sample_idx]

            _runtime_start = time.perf_counter()
            result_dict = predict_one_window_fbcca_beta(
                trial_eeg=trial_eeg,
                srate=srate,
                reference_signal_list=reference_signal_list,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b
            )

            runtime_matrix[row_idx, w_idx] = float(time.perf_counter() - _runtime_start)
            pred_label_matrix[row_idx, w_idx] = int(result_dict['pred_label'])
            margin_matrix[row_idx, w_idx] = float(result_dict['margin_value'])

    return {
        'pred_label_matrix': pred_label_matrix,
        'margin_matrix': margin_matrix,
        'runtime_matrix': runtime_matrix,
        'y_true': y[test_indices],
        'sample_indices': test_indices,
        'test_meta_df': meta_df.iloc[test_indices].reset_index(drop=True),
        'window_sec_list': window_sec_list,
        'gaze_shift_sec': float(info_dict.get('gaze_shift_sec', 0.55)),
        'n_classes': int(info_dict['n_classes']),
        'split_mode': split_mode,
        'split_value': test_split_value
    }


def evaluate_one_fold_nonstable_early_stop_cca_beta(
    multiscale_trial_package,
    cca_reference_cache,
    split_mode,
    test_split_value,
    threshold,
    allowed_split_values=None,
    n_harmonics=5,
    apply_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4
):
    fold_cache = build_cca_fold_prediction_cache_beta(
        multiscale_trial_package=multiscale_trial_package,
        cca_reference_cache=cca_reference_cache,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        apply_bandpass=apply_bandpass,
        lowcut=lowcut,
        highcut=highcut,
        filter_order=filter_order
    )

    return evaluate_nonstable_early_stop_from_cache_beta(
        fold_cache=fold_cache,
        threshold=threshold,
        model_name='CCA'
    )


def evaluate_one_fold_nonstable_early_stop_fbcca_beta(
    multiscale_trial_package,
    fbcca_reference_cache,
    split_mode,
    test_split_value,
    threshold,
    allowed_split_values=None,
    n_harmonics=5,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25
):
    fold_cache = build_fbcca_fold_prediction_cache_beta(
        multiscale_trial_package=multiscale_trial_package,
        fbcca_reference_cache=fbcca_reference_cache,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        subband_low_list=subband_low_list,
        subband_highcut=subband_highcut,
        filter_order=filter_order,
        weight_a=weight_a,
        weight_b=weight_b
    )

    return evaluate_nonstable_early_stop_from_cache_beta(
        fold_cache=fold_cache,
        threshold=threshold,
        model_name='FBCCA'
    )


def run_nonstable_early_stop_experiment_cca_beta(
    multiscale_trial_package,
    candidate_threshold_list,
    split_mode='block',
    n_harmonics=5,
    apply_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4,
    max_accuracy_drop=None
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            one_subject_package = _subset_multiscale_trial_package_by_subject_beta(
                multiscale_trial_package=multiscale_trial_package,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_nonstable_early_stop_experiment_cca_beta(
                multiscale_trial_package=one_subject_package,
                candidate_threshold_list=candidate_threshold_list,
                split_mode=split_mode,
                n_harmonics=n_harmonics,
                apply_bandpass=apply_bandpass,
                lowcut=lowcut,
                highcut=highcut,
                filter_order=filter_order,
                max_accuracy_drop=max_accuracy_drop
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name='CCA',
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    cca_reference_cache = prepare_cca_reference_cache_beta(
        multiscale_trial_package=multiscale_trial_package,
        n_harmonics=n_harmonics
    )

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}
    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        inner_fold_cache_dict = {}

        for inner_val_split_value in outer_train_split_values:
            inner_fold_cache_dict[inner_val_split_value] = build_cca_fold_prediction_cache_beta(
                multiscale_trial_package=multiscale_trial_package,
                cca_reference_cache=cca_reference_cache,
                split_mode=split_mode,
                test_split_value=inner_val_split_value,
                allowed_split_values=outer_train_split_values,
                apply_bandpass=apply_bandpass,
                lowcut=lowcut,
                highcut=highcut,
                filter_order=filter_order
            )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name='CCA'
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name='CCA'
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = 'CCA'
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df
        inner_fold_cache_by_outer_fold[outer_test_split_value] = inner_fold_cache_dict

        outer_fold_cache = build_cca_fold_prediction_cache_beta(
            multiscale_trial_package=multiscale_trial_package,
            cca_reference_cache=cca_reference_cache,
            split_mode=split_mode,
            test_split_value=outer_test_split_value,
            allowed_split_values=None,
            apply_bandpass=apply_bandpass,
            lowcut=lowcut,
            highcut=highcut,
            filter_order=filter_order
        )

        outer_fold_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name='CCA'
        )

        outer_fold_cache_by_outer_fold[outer_test_split_value] = outer_fold_cache

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name='CCA',
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }


def run_nonstable_early_stop_experiment_fbcca_beta(
    multiscale_trial_package,
    candidate_threshold_list,
    split_mode='block',
    n_harmonics=5,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    max_accuracy_drop=None
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            one_subject_package = _subset_multiscale_trial_package_by_subject_beta(
                multiscale_trial_package=multiscale_trial_package,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_nonstable_early_stop_experiment_fbcca_beta(
                multiscale_trial_package=one_subject_package,
                candidate_threshold_list=candidate_threshold_list,
                split_mode=split_mode,
                n_harmonics=n_harmonics,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b,
                max_accuracy_drop=max_accuracy_drop
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name='FBCCA',
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    fbcca_reference_cache = prepare_fbcca_reference_cache_beta(
        multiscale_trial_package=multiscale_trial_package,
        n_harmonics=n_harmonics
    )

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}
    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        inner_fold_cache_dict = {}

        for inner_val_split_value in outer_train_split_values:
            inner_fold_cache_dict[inner_val_split_value] = build_fbcca_fold_prediction_cache_beta(
                multiscale_trial_package=multiscale_trial_package,
                fbcca_reference_cache=fbcca_reference_cache,
                split_mode=split_mode,
                test_split_value=inner_val_split_value,
                allowed_split_values=outer_train_split_values,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b
            )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name='FBCCA'
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name='FBCCA'
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = 'FBCCA'
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df
        inner_fold_cache_by_outer_fold[outer_test_split_value] = inner_fold_cache_dict

        outer_fold_cache = build_fbcca_fold_prediction_cache_beta(
            multiscale_trial_package=multiscale_trial_package,
            fbcca_reference_cache=fbcca_reference_cache,
            split_mode=split_mode,
            test_split_value=outer_test_split_value,
            allowed_split_values=None,
            subband_low_list=subband_low_list,
            subband_highcut=subband_highcut,
            filter_order=filter_order,
            weight_a=weight_a,
            weight_b=weight_b
        )

        outer_fold_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name='FBCCA'
        )

        outer_fold_cache_by_outer_fold[outer_test_split_value] = outer_fold_cache

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name='FBCCA',
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }

### TRCA和msTRCA非稳定早停函数

In [ ]:
# ------------ 用于 TRCA / msTRCA 的“当前窗口预测函数” ------------

# 过程：
# ① 对单个窗口下的单个 trial 做 TRCA / msTRCA 预测
# ② 输出 pred_label、margin_value、raw_score_vector
# ③ 风格与当前 notebook 中的 CCA / FBCCA 当前窗口预测函数保持一致

def predict_one_window_trca_beta(
    trial_eeg,
    srate,
    model_state
):
    trial_eeg = np.asarray(trial_eeg, dtype=float)

    pred_label, trca_score_array = _predict_one_trial_trca_like_beta(
        trial_eeg=trial_eeg,
        srate=srate,
        model_state=model_state
    )

    result_dict = get_pred_margin_from_vector_beta(
        vector=trca_score_array,
        model_name='TRCA',
        vector_type='score'
    )
    result_dict['raw_score_vector'] = np.asarray(trca_score_array, dtype=float)
    result_dict['pred_label_raw'] = int(pred_label)

    return result_dict


def predict_one_window_mstrca_beta(
    trial_eeg,
    srate,
    model_state
):
    trial_eeg = np.asarray(trial_eeg, dtype=float)

    pred_label, mstrca_score_array = _predict_one_trial_trca_like_beta(
        trial_eeg=trial_eeg,
        srate=srate,
        model_state=model_state
    )

    result_dict = get_pred_margin_from_vector_beta(
        vector=mstrca_score_array,
        model_name='msTRCA',
        vector_type='score'
    )
    result_dict['raw_score_vector'] = np.asarray(mstrca_score_array, dtype=float)
    result_dict['pred_label_raw'] = int(pred_label)

    return result_dict


# ------------ 构造 TRCA 的 fold prediction cache ------------

# 过程：
# ① 对每一个 window_sec 单独训练一个 TRCA 模型
# ② 对当前 test fold 中所有样本，记录每个窗口下的 pred_label 和 margin
# ③ 最终拼成 pred_label_matrix / margin_matrix
# ④ 后续直接交给 evaluate_nonstable_early_stop_from_cache_beta 做阈值早停评估

def build_trca_fold_prediction_cache_beta(
    multiscale_trial_package,
    split_mode,
    test_split_value,
    allowed_split_values=None,
    use_ensemble=True,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]

    y = first_window_data['y']
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']
    split_key = get_split_key_beta(split_mode=split_mode)

    if allowed_split_values is None:
        subset_mask = np.ones(len(meta_df), dtype=bool)
    else:
        subset_mask = meta_df[split_key].isin(list(allowed_split_values)).values

    test_mask = subset_mask & (meta_df[split_key].values == test_split_value)
    test_indices = np.where(test_mask)[0]

    window_sec_list = [float(x) for x in multiscale_trial_package['window_sec_list']]

    pred_label_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=int)
    margin_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=float)
    runtime_matrix = np.zeros((len(test_indices), len(window_sec_list)), dtype=float)

    for w_idx, window_sec in enumerate(window_sec_list):
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]

        trial_list = one_window_data['trial_list']
        y_one_window = one_window_data['y']
        meta_df_one_window = one_window_data['meta_df']
        info_dict_one_window = one_window_data['info_dict']
        srate = float(info_dict_one_window['srate'])

        if allowed_split_values is None:
            subset_mask_one_window = np.ones(len(meta_df_one_window), dtype=bool)
        else:
            subset_mask_one_window = meta_df_one_window[split_key].isin(list(allowed_split_values)).values

        train_mask = subset_mask_one_window & (meta_df_one_window[split_key].values != test_split_value)
        train_indices = np.where(train_mask)[0]

        model_state = _fit_trca_like_model_beta(
            trial_list=trial_list,
            y=y_one_window,
            train_indices=train_indices,
            info_dict=info_dict_one_window,
            subband_low_list=subband_low_list,
            subband_highcut=subband_highcut,
            filter_order=filter_order,
            weight_a=weight_a,
            weight_b=weight_b,
            regularization=regularization,
            use_ensemble=use_ensemble,
            use_multistimulus=False,
            neighbor_template_count=0
        )

        for row_idx, sample_idx in enumerate(test_indices):
            trial_eeg = trial_list[sample_idx]

            _runtime_start = time.perf_counter()
            result_dict = predict_one_window_trca_beta(
                trial_eeg=trial_eeg,
                srate=srate,
                model_state=model_state
            )

            runtime_matrix[row_idx, w_idx] = float(time.perf_counter() - _runtime_start)
            pred_label_matrix[row_idx, w_idx] = int(result_dict['pred_label'])
            margin_matrix[row_idx, w_idx] = float(result_dict['margin_value'])

    return {
        'pred_label_matrix': pred_label_matrix,
        'margin_matrix': margin_matrix,
        'runtime_matrix': runtime_matrix,
        'y_true': y[test_indices],
        'sample_indices': test_indices,
        'test_meta_df': meta_df.iloc[test_indices].reset_index(drop=True),
        'window_sec_list': window_sec_list,
        'gaze_shift_sec': float(info_dict.get('gaze_shift_sec', 0.55)),
        'n_classes': int(info_dict['n_classes']),
        'split_mode': split_mode,
        'split_value': test_split_value
    }


# ------------ 构造 msTRCA 的 fold prediction cache ------------

# 过程：
# ① 对每一个 window_sec 单独训练一个 msTRCA 模型
# ② 训练时将目标频率及其邻近频率 trial 一起用于空间滤波器学习
# ③ 对当前 test fold 中所有样本，记录每个窗口下的 pred_label 和 margin
# ④ 最终拼成 pred_label_matrix / margin_matrix，供后续阈值早停评估使用

def build_mstrca_fold_prediction_cache_beta(
    multiscale_trial_package,
    split_mode,
    test_split_value,
    allowed_split_values=None,
    use_ensemble=True,
    neighbor_template_count=2,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]

    y = first_window_data['y']
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']
    split_key = get_split_key_beta(split_mode=split_mode)

    if allowed_split_values is None:
        subset_mask = np.ones(len(meta_df), dtype=bool)
    else:
        subset_mask = meta_df[split_key].isin(list(allowed_split_values)).values

    test_mask = subset_mask & (meta_df[split_key].values == test_split_value)
    test_indices = np.where(test_mask)[0]

    window_sec_list = [float(x) for x in multiscale_trial_package['window_sec_list']]

    pred_label_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=int)
    margin_matrix = np.empty((len(test_indices), len(window_sec_list)), dtype=float)
    runtime_matrix = np.zeros((len(test_indices), len(window_sec_list)), dtype=float)

    for w_idx, window_sec in enumerate(window_sec_list):
        one_window_data = multiscale_trial_package['data_by_window'][float(window_sec)]

        trial_list = one_window_data['trial_list']
        y_one_window = one_window_data['y']
        meta_df_one_window = one_window_data['meta_df']
        info_dict_one_window = one_window_data['info_dict']
        srate = float(info_dict_one_window['srate'])

        if allowed_split_values is None:
            subset_mask_one_window = np.ones(len(meta_df_one_window), dtype=bool)
        else:
            subset_mask_one_window = meta_df_one_window[split_key].isin(list(allowed_split_values)).values

        train_mask = subset_mask_one_window & (meta_df_one_window[split_key].values != test_split_value)
        train_indices = np.where(train_mask)[0]

        model_state = _fit_trca_like_model_beta(
            trial_list=trial_list,
            y=y_one_window,
            train_indices=train_indices,
            info_dict=info_dict_one_window,
            subband_low_list=subband_low_list,
            subband_highcut=subband_highcut,
            filter_order=filter_order,
            weight_a=weight_a,
            weight_b=weight_b,
            regularization=regularization,
            use_ensemble=use_ensemble,
            use_multistimulus=True,
            neighbor_template_count=neighbor_template_count
        )

        for row_idx, sample_idx in enumerate(test_indices):
            trial_eeg = trial_list[sample_idx]

            _runtime_start = time.perf_counter()
            result_dict = predict_one_window_mstrca_beta(
                trial_eeg=trial_eeg,
                srate=srate,
                model_state=model_state
            )

            runtime_matrix[row_idx, w_idx] = float(time.perf_counter() - _runtime_start)
            pred_label_matrix[row_idx, w_idx] = int(result_dict['pred_label'])
            margin_matrix[row_idx, w_idx] = float(result_dict['margin_value'])

    return {
        'pred_label_matrix': pred_label_matrix,
        'margin_matrix': margin_matrix,
        'runtime_matrix': runtime_matrix,
        'y_true': y[test_indices],
        'sample_indices': test_indices,
        'test_meta_df': meta_df.iloc[test_indices].reset_index(drop=True),
        'window_sec_list': window_sec_list,
        'gaze_shift_sec': float(info_dict.get('gaze_shift_sec', 0.55)),
        'n_classes': int(info_dict['n_classes']),
        'split_mode': split_mode,
        'split_value': test_split_value
    }


# ------------ 单个 fold 的 TRCA 非稳定早停评估函数 ------------

def evaluate_one_fold_nonstable_early_stop_trca_beta(
    multiscale_trial_package,
    split_mode,
    test_split_value,
    threshold,
    allowed_split_values=None,
    use_ensemble=True,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8
):
    fold_cache = build_trca_fold_prediction_cache_beta(
        multiscale_trial_package=multiscale_trial_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        use_ensemble=use_ensemble,
        subband_low_list=subband_low_list,
        subband_highcut=subband_highcut,
        filter_order=filter_order,
        weight_a=weight_a,
        weight_b=weight_b,
        regularization=regularization
    )

    return evaluate_nonstable_early_stop_from_cache_beta(
        fold_cache=fold_cache,
        threshold=threshold,
        model_name='TRCA'
    )


# ------------ 单个 fold 的 msTRCA 非稳定早停评估函数 ------------

def evaluate_one_fold_nonstable_early_stop_mstrca_beta(
    multiscale_trial_package,
    split_mode,
    test_split_value,
    threshold,
    allowed_split_values=None,
    use_ensemble=True,
    neighbor_template_count=2,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8
):
    fold_cache = build_mstrca_fold_prediction_cache_beta(
        multiscale_trial_package=multiscale_trial_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        use_ensemble=use_ensemble,
        neighbor_template_count=neighbor_template_count,
        subband_low_list=subband_low_list,
        subband_highcut=subband_highcut,
        filter_order=filter_order,
        weight_a=weight_a,
        weight_b=weight_b,
        regularization=regularization
    )

    return evaluate_nonstable_early_stop_from_cache_beta(
        fold_cache=fold_cache,
        threshold=threshold,
        model_name='msTRCA'
    )


# ------------ 整体 TRCA 非稳定早停实验（含 nested CV 阈值搜索） ------------

# 过程：
# ① outer fold：留一个 block 做最终测试
# ② inner fold：在 outer train 上做阈值搜索
# ③ 找到最优 threshold 后，在 outer test 上评估
# ④ 最后汇总所有 outer fold 的准确率、平均决策时间、早停率、ITR

def run_nonstable_early_stop_experiment_trca_beta(
    multiscale_trial_package,
    candidate_threshold_list,
    split_mode='block',
    use_ensemble=True,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8,
    max_accuracy_drop=None
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            one_subject_package = _subset_multiscale_trial_package_by_subject_beta(
                multiscale_trial_package=multiscale_trial_package,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_nonstable_early_stop_experiment_trca_beta(
                multiscale_trial_package=one_subject_package,
                candidate_threshold_list=candidate_threshold_list,
                split_mode=split_mode,
                use_ensemble=use_ensemble,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b,
                regularization=regularization,
                max_accuracy_drop=max_accuracy_drop
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name='TRCA',
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}
    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        inner_fold_cache_dict = {}

        for inner_val_split_value in outer_train_split_values:
            inner_fold_cache_dict[inner_val_split_value] = build_trca_fold_prediction_cache_beta(
                multiscale_trial_package=multiscale_trial_package,
                split_mode=split_mode,
                test_split_value=inner_val_split_value,
                allowed_split_values=outer_train_split_values,
                use_ensemble=use_ensemble,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b,
                regularization=regularization
            )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name='TRCA'
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name='TRCA'
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = 'TRCA'
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df
        inner_fold_cache_by_outer_fold[outer_test_split_value] = inner_fold_cache_dict

        outer_fold_cache = build_trca_fold_prediction_cache_beta(
            multiscale_trial_package=multiscale_trial_package,
            split_mode=split_mode,
            test_split_value=outer_test_split_value,
            allowed_split_values=None,
            use_ensemble=use_ensemble,
            subband_low_list=subband_low_list,
            subband_highcut=subband_highcut,
            filter_order=filter_order,
            weight_a=weight_a,
            weight_b=weight_b,
            regularization=regularization
        )

        outer_fold_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name='TRCA'
        )

        outer_fold_cache_by_outer_fold[outer_test_split_value] = outer_fold_cache

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name='TRCA',
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }


# ------------ 整体 msTRCA 非稳定早停实验（含 nested CV 阈值搜索） ------------

def run_nonstable_early_stop_experiment_mstrca_beta(
    multiscale_trial_package,
    candidate_threshold_list,
    split_mode='block',
    use_ensemble=True,
    neighbor_template_count=2,
    subband_low_list=None,
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8,
    max_accuracy_drop=None
):
    first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
    first_window_data = multiscale_trial_package['data_by_window'][first_window_sec]
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            one_subject_package = _subset_multiscale_trial_package_by_subject_beta(
                multiscale_trial_package=multiscale_trial_package,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_nonstable_early_stop_experiment_mstrca_beta(
                multiscale_trial_package=one_subject_package,
                candidate_threshold_list=candidate_threshold_list,
                split_mode=split_mode,
                use_ensemble=use_ensemble,
                neighbor_template_count=neighbor_template_count,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b,
                regularization=regularization,
                max_accuracy_drop=max_accuracy_drop
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name='msTRCA',
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}
    inner_fold_cache_by_outer_fold = {}
    outer_fold_cache_by_outer_fold = {}

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        inner_fold_cache_dict = {}

        for inner_val_split_value in outer_train_split_values:
            inner_fold_cache_dict[inner_val_split_value] = build_mstrca_fold_prediction_cache_beta(
                multiscale_trial_package=multiscale_trial_package,
                split_mode=split_mode,
                test_split_value=inner_val_split_value,
                allowed_split_values=outer_train_split_values,
                use_ensemble=use_ensemble,
                neighbor_template_count=neighbor_template_count,
                subband_low_list=subband_low_list,
                subband_highcut=subband_highcut,
                filter_order=filter_order,
                weight_a=weight_a,
                weight_b=weight_b,
                regularization=regularization
            )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name='msTRCA'
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_nonstable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name='msTRCA'
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = 'msTRCA'
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df
        inner_fold_cache_by_outer_fold[outer_test_split_value] = inner_fold_cache_dict

        outer_fold_cache = build_mstrca_fold_prediction_cache_beta(
            multiscale_trial_package=multiscale_trial_package,
            split_mode=split_mode,
            test_split_value=outer_test_split_value,
            allowed_split_values=None,
            use_ensemble=use_ensemble,
            neighbor_template_count=neighbor_template_count,
            subband_low_list=subband_low_list,
            subband_highcut=subband_highcut,
            filter_order=filter_order,
            weight_a=weight_a,
            weight_b=weight_b,
            regularization=regularization
        )

        outer_fold_result = evaluate_nonstable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name='msTRCA'
        )

        outer_fold_cache_by_outer_fold[outer_test_split_value] = outer_fold_cache

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name='msTRCA',
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold,
        'inner_fold_cache_by_outer_fold': inner_fold_cache_by_outer_fold,
        'outer_fold_cache_by_outer_fold': outer_fold_cache_by_outer_fold
    }

### 主调用示例

In [ ]:
# 先构造“非稳定性早停”要用的多窗口数据。
base_folder = '/Users/mikuru/SSVEP/DATA/BETA DATA'

selected_subject_ids = [
    'S16', 'S17', 'S18', 'S19', 'S20',
    'S21', 'S22', 'S23', 'S24', 'S25', 'S26', 'S27', 'S28', 'S29', 'S30',
    'S31', 'S32', 'S33', 'S34', 'S35', 'S36', 'S37', 'S38', 'S39', 'S40',
    'S41', 'S42', 'S43', 'S44', 'S45', 'S46', 'S47', 'S48', 'S49', 'S50',
    'S51', 'S52', 'S53', 'S54', 'S55', 'S56', 'S57', 'S58', 'S59', 'S60',
    'S61', 'S62', 'S63', 'S64', 'S65', 'S66', 'S67', 'S68', 'S69', 'S70'
]
selected_chans = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
selected_freq_indices = list(range(40))

window_sec_list = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]

# candidate_threshold_list_fbcca = [0.005, 0.006, 0.007, 0.0075, 0.008, 0.009, 0.01, 0.011, 0.0125, 0.015]
# candidate_threshold_list_cca = [0.001, 0.0015, 0.002, 0.0025, 0.003, 0.004, 0.005, 0.0075, 0.01]
# candidate_threshold_list_svm = [0.005, 0.0075, 0.01, 0.0125, 0.015, 0.0175, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045, 0.05]
# candidate_threshold_list_lda = [0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]
# candidate_threshold_list_trca = [0.002, 0.0025, 0.003, 0.0035, 0.004, 0.0045, 0.005, 0.006, 0.0075]
# candidate_threshold_list_mstrca = [0.0015, 0.002, 0.0025, 0.003, 0.0035, 0.004, 0.005, 0.0075]

# candidate_threshold_list_fbcca = [0.005, 0.006, 0.007, 0.0075, 0.008, 0.0085, 0.009, 0.01, 0.011]
# candidate_threshold_list_cca = [0.001, 0.0015, 0.0016, 0.0017, 0.00175, 0.002, 0.0025, 0.003, 0.0035, 0.004, 0.005]
# candidate_threshold_list_svm = [
#     0.001, 0.01, 0.05, 0.1, 0.2, 0.5,
#     1.0, 1.5, 2.0, 2.5,
#     3.0, 3.5, 4.0, 4.5,
#     5.0, 5.5, 6.0,
#     7.0, 8.0, 10.0, 12.0
# ]
# candidate_threshold_list_lda = sorted(set([
#     # Based on the LDA-only validation run after applying chance-level ITR=0:
#     # non-stable selected thresholds were mainly in 0-30, with max about 70;
#     # stable selected thresholds were mainly in 0-10, with max about 12.5.
#     # Therefore this list keeps dense coverage in 0-30 and only a few safety values above it.
#     0.0,
#     0.001, 0.005, 0.01, 0.02, 0.05,
#     0.1, 0.2, 0.3, 0.4, 0.5,
#     0.6, 0.8,
#     1.0, 1.2, 1.5, 1.8,
#     2.0, 2.2, 2.5, 2.8,
#     3.0, 3.3, 3.5, 3.8,
#     4.0, 4.5,
#     5.0, 5.5,
#     6.0, 6.5,
#     7.0, 7.5,
#     8.0, 8.5,
#     9.0, 9.5,
#     10.0, 11.0, 12.0, 13.0, 14.0, 15.0,
#     16.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 30.0,
#     35.0, 40.0, 50.0, 70.0, 100.0
# ]))
# candidate_threshold_list_trca = [0.001, 0.0015, 0.002, 0.0025, 0.003, 0.0035, 0.0036, 0.00375, 0.004, 0.0045, 0.005]
# CANDIDATE_THRESHOLD_LIST_MSTRCA = [
#     0.002, 0.0025, 0.003, 0.0035,
#     0.004, 0.0045, 0.005, 0.0055,
#     0.006, 0.007, 0.008, 0.010,
# ]
# candidate_threshold_list_mstrca = CANDIDATE_THRESHOLD_LIST_MSTRCA

candidate_threshold_list_cca = [
    0.0010, 0.0011, 0.0012, 0.0013, 0.0014,
    0.0015, 0.00155, 0.0016, 0.00165, 0.0017,
    0.00175, 0.0018, 0.0019, 0.0020, 0.00225,
    0.0025, 0.0030, 0.0035, 0.0040, 0.0050
]

candidate_threshold_list_fbcca = [
    0.0045, 0.0050, 0.0055, 0.0060, 0.0065,
    0.0070, 0.00725, 0.0075, 0.00775, 0.0080,
    0.00825, 0.0085, 0.00875, 0.0090, 0.0095,
    0.0100, 0.0110, 0.0120, 0.0135, 0.0150
]

candidate_threshold_list_svm = [
    0.001, 0.01, 0.05, 0.1, 0.2,
    0.5, 0.75, 1.0, 1.25, 1.5,
    2.0, 2.5, 3.0, 3.5, 4.0,
    4.5, 5.0, 5.5, 6.0, 7.0
]

candidate_threshold_list_lda = [
    0.5, 1.0, 1.5, 2.0, 2.5,
    3.0, 4.0, 5.0, 6.0, 7.5,
    9.0, 10.0, 12.5, 15.0, 18.0,
    20.0, 25.0, 30.0, 50.0, 70.0
]

candidate_threshold_list_trca = [
    0.0010, 0.00125, 0.0015, 0.00175, 0.0020,
    0.00225, 0.0025, 0.00275, 0.0030, 0.00325,
    0.0035, 0.0036, 0.00375, 0.0040, 0.00425,
    0.0045, 0.00475, 0.0050, 0.0055, 0.0060
]

CANDIDATE_THRESHOLD_LIST_MSTRCA = [
    0.0020, 0.00225, 0.0025, 0.00275, 0.0030,
    0.00325, 0.0035, 0.00375, 0.0040, 0.00425,
    0.0045, 0.00475, 0.0050, 0.0055, 0.0060,
    0.0070, 0.0080, 0.0090, 0.0100, 0.0120
]
candidate_threshold_list_mstrca = CANDIDATE_THRESHOLD_LIST_MSTRCA


multiscale_feature_package = build_multiscale_feature_package_beta(
    base_folder=base_folder,
    selected_subject_ids=selected_subject_ids,
    selected_chans=selected_chans,
    selected_freq_indices=selected_freq_indices,
    window_sec_list=window_sec_list,
    latency_correction_sec=0.13,
    gaze_shift_sec=0.55,
    feature_mode='wide',
    concat_style='by_type',
    use_selected_freqs_as_feature_candidates=True,
    use_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4,
    use_hann_window=False,
    use_zero_padding=True,
    nfft=None,
    pad_total_sec=5.0,
    narrow_neighbor_count=5,
    wide_n_harmonics=5
)

multiscale_trial_package = build_multiscale_trial_package_beta(
    base_folder=base_folder,
    selected_subject_ids=selected_subject_ids,
    selected_chans=selected_chans,
    selected_freq_indices=selected_freq_indices,
    window_sec_list=window_sec_list,
    latency_correction_sec=0.13,
    gaze_shift_sec=0.55
)

In [ ]:
# 先测试 FBCCA 的“非稳定性早停”实验。
fbcca_nonstable_result = run_nonstable_early_stop_experiment_fbcca_beta(
    multiscale_trial_package=multiscale_trial_package,
    candidate_threshold_list=candidate_threshold_list_fbcca,
    split_mode='block',
    n_harmonics=5,
    subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25
)

display(fbcca_nonstable_result['summary_df'])
display(fbcca_nonstable_result['fold_results_df'])

# 保存非稳定早停结果，同时从 prediction cache 保存固定窗 0.2-2.0s sweep/window detail。

save_early_stop_tables_beta(
    result=fbcca_nonstable_result,
    model_name='FBCCA',
    strategy_name='nonstable_early_stop',
    cache_result=fbcca_nonstable_result,
    min_stable_windows=np.nan,
    save_window_cache=True,
    save_fixed_window_from_cache=True
)


In [ ]:
# 测试 CCA 的“非稳定性早停”实验。
cca_nonstable_result = run_nonstable_early_stop_experiment_cca_beta(
    multiscale_trial_package=multiscale_trial_package,
    candidate_threshold_list=candidate_threshold_list_cca,
    split_mode='block',
    n_harmonics=5,
    apply_bandpass=True,
    lowcut=6.0,
    highcut=80.0,
    filter_order=4
)

display(cca_nonstable_result['summary_df'])
display(cca_nonstable_result['fold_results_df'])

# 保存非稳定早停结果，同时从 prediction cache 保存固定窗 0.2-2.0s sweep/window detail。

save_early_stop_tables_beta(
    result=cca_nonstable_result,
    model_name='CCA',
    strategy_name='nonstable_early_stop',
    cache_result=cca_nonstable_result,
    min_stable_windows=np.nan,
    save_window_cache=True,
    save_fixed_window_from_cache=True
)


In [ ]:
# 测试 SVM 的“非稳定性早停”实验。
svm_nonstable_result = run_nonstable_early_stop_experiment_svm_beta(
    multiscale_feature_package=multiscale_feature_package,
    candidate_threshold_list=candidate_threshold_list_svm,
    split_mode='block',
    C=1.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
)

display(svm_nonstable_result['summary_df'])
display(svm_nonstable_result['fold_results_df'])

# 保存 SVM 非稳定早停结果，同时保存固定窗 0.2-2.0s sweep/window detail。
save_early_stop_tables_beta(
    result=svm_nonstable_result,
    model_name='SVM',
    strategy_name='nonstable_early_stop',
    cache_result=svm_nonstable_result,
    min_stable_windows=np.nan,
    save_window_cache=True,
    save_fixed_window_from_cache=True
)


# 测试 LDA 的“非稳定性早停”实验。
lda_nonstable_result = run_nonstable_early_stop_experiment_lda_beta(
    multiscale_feature_package=multiscale_feature_package,
    candidate_threshold_list=candidate_threshold_list_lda,
    split_mode='block',
    solver='lsqr',
    shrinkage='auto'
)

display(lda_nonstable_result['summary_df'])
display(lda_nonstable_result['fold_results_df'])

# 保存 LDA 非稳定早停结果，同时保存固定窗 0.2-2.0s sweep/window detail。
save_early_stop_tables_beta(
    result=lda_nonstable_result,
    model_name='LDA',
    strategy_name='nonstable_early_stop',
    cache_result=lda_nonstable_result,
    min_stable_windows=np.nan,
    save_window_cache=True,
    save_fixed_window_from_cache=True
)

In [ ]:
# ------------ 非稳定早停：加入 TRCA 和 msTRCA 的主调用示例 ------------


# 测试 TRCA 的“非稳定性早停”实验。
trca_nonstable_result = run_nonstable_early_stop_experiment_trca_beta(
    multiscale_trial_package=multiscale_trial_package,
    candidate_threshold_list=candidate_threshold_list_trca,
    split_mode='block',
    use_ensemble=True,
    subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8
)

display(trca_nonstable_result['summary_df'])
display(trca_nonstable_result['fold_results_df'])

# 保存 TRCA 非稳定早停结果，同时保存固定窗 0.2-2.0s sweep/window detail。
save_early_stop_tables_beta(
    result=trca_nonstable_result,
    model_name='TRCA',
    strategy_name='nonstable_early_stop',
    cache_result=trca_nonstable_result,
    min_stable_windows=np.nan,
    save_window_cache=True,
    save_fixed_window_from_cache=True
)


# 测试 msTRCA 的“非稳定性早停”实验。
mstrca_nonstable_result = run_nonstable_early_stop_experiment_mstrca_beta(
    multiscale_trial_package=multiscale_trial_package,
    candidate_threshold_list=candidate_threshold_list_mstrca,
    split_mode='block',
    use_ensemble=True,
    neighbor_template_count=2,  # 2个邻近频率 + 目标频率 = 共3个频率参与ms-eTRCA空间滤波器训练
    subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
    subband_highcut=90.0,
    filter_order=4,
    weight_a=1.25,
    weight_b=0.25,
    regularization=1e-8
)

display(mstrca_nonstable_result['summary_df'])
display(mstrca_nonstable_result['fold_results_df'])

# 保存 msTRCA 非稳定早停结果，同时保存固定窗 0.2-2.0s sweep/window detail。
save_early_stop_tables_beta(
    result=mstrca_nonstable_result,
    model_name='msTRCA',
    strategy_name='nonstable_early_stop',
    cache_result=mstrca_nonstable_result,
    min_stable_windows=np.nan,
    save_window_cache=True,
    save_fixed_window_from_cache=True
)

In [ ]:
# 无精度限制汇总结果表格
nonstable_summary_df = pd.concat([
    fbcca_nonstable_result['summary_df'],
    cca_nonstable_result['summary_df'],
    svm_nonstable_result['summary_df'],
    lda_nonstable_result['summary_df'],
    trca_nonstable_result['summary_df'],
    mstrca_nonstable_result['summary_df']
], ignore_index=True)

nonstable_summary_df[[
    'model_name',
    'mean_fold_accuracy',
    'mean_decision_time_sec',
    'mean_recognition_runtime_sec',
    'mean_decision_time_with_runtime_sec',
    'mean_early_stop_rate',
    'itr_bits_per_min',
    'itr_with_runtime_bits_per_min'
]]

## 稳定性早停下的模型

### 稳定性早停评估函数

In [ ]:
# ------------ 稳定性早停：从 cache 直接评估一个 fold ------------

def evaluate_stable_early_stop_from_cache_beta(
    fold_cache,
    threshold,
    model_name,
    min_stable_windows=2
):
    pred_label_matrix = np.asarray(fold_cache['pred_label_matrix'], dtype=int)
    margin_matrix = np.asarray(fold_cache['margin_matrix'], dtype=float)
    y_true = np.asarray(fold_cache['y_true'], dtype=int)
    sample_indices = np.asarray(fold_cache['sample_indices'], dtype=int)
    window_sec_list = np.asarray(fold_cache['window_sec_list'], dtype=float)
    runtime_matrix = np.asarray(
        fold_cache.get('runtime_matrix', np.zeros_like(margin_matrix, dtype=float)),
        dtype=float
    )
    if runtime_matrix.shape != margin_matrix.shape:
        runtime_matrix = np.zeros_like(margin_matrix, dtype=float)
    cumulative_runtime_matrix = np.cumsum(runtime_matrix, axis=1)

    n_samples, n_windows = margin_matrix.shape
    trial_result_rows = []

    for sample_i in range(n_samples):
        pred_history = []
        stop_window_idx = n_windows - 1
        stopped_early = False
        stop_reason = 'forced_at_last_window'

        for window_idx in range(n_windows):
            current_pred = int(pred_label_matrix[sample_i, window_idx])
            current_margin = float(margin_matrix[sample_i, window_idx])

            pred_history.append(current_pred)

            is_stable = check_recent_predictions_stable_beta(
                pred_history=pred_history,
                min_stable_windows=min_stable_windows
            )

            if (current_margin >= float(threshold)) and is_stable:
                stop_window_idx = window_idx
                stopped_early = True
                stop_reason = f'margin_and_stable_{int(min_stable_windows)}'
                break

        recognition_runtime_sec = float(cumulative_runtime_matrix[sample_i, stop_window_idx])
        decision_time_sec = float(window_sec_list[stop_window_idx] + float(fold_cache['gaze_shift_sec']))

        trial_result_rows.append({
            'sample_idx': int(sample_indices[sample_i]),
            'y_true': int(y_true[sample_i]),
            'y_pred': int(pred_label_matrix[sample_i, stop_window_idx]),
            'margin_value': float(margin_matrix[sample_i, stop_window_idx]),
            'stop_window_sec': float(window_sec_list[stop_window_idx]),
            'decision_time_sec': decision_time_sec,
            'recognition_runtime_sec': recognition_runtime_sec,
            'decision_time_with_runtime_sec': float(decision_time_sec + recognition_runtime_sec),
            'stopped_early': bool(stopped_early),
            'stop_reason': stop_reason
        })

    trial_results_df = pd.DataFrame(trial_result_rows).copy()
    trial_results_df = _add_fold_cache_metadata_to_trial_results_beta(
        trial_results_df=trial_results_df,
        fold_cache=fold_cache
    )

    result = summarize_one_fold_nonstable_early_stop_beta(
        trial_result_rows=trial_results_df.to_dict('records'),
        model_name=model_name,
        split_mode=fold_cache['split_mode'],
        split_value=fold_cache['split_value'],
        threshold=threshold,
        n_classes=fold_cache['n_classes']
    )

    result['fold_summary_row']['min_stable_windows'] = int(min_stable_windows)
    return result

### 通用 stable nested CV 外壳

In [ ]:
# ------------ 稳定性早停：通用 nested CV 外壳 ------------

def run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data,
    split_mode,
    candidate_threshold_list,
    min_stable_windows,
    model_name,
    build_fold_cache_func,
    max_accuracy_drop=None,
    precomputed_nonstable_result=None
):
    meta_df = first_window_data['meta_df']
    info_dict = first_window_data['info_dict']

    def _extract_subject_level_nonstable_result(one_result, target_subject_id):
        if one_result is None:
            return None

        subject_result_list = one_result.get('subject_result_list', None)
        if subject_result_list is None:
            return None

        for one_subject_result in subject_result_list:
            one_fold_results_df = one_subject_result.get('fold_results_df', pd.DataFrame())
            if len(one_fold_results_df) == 0:
                continue
            if 'subject_id' not in one_fold_results_df.columns:
                continue
            current_subject_id = str(one_fold_results_df['subject_id'].iloc[0])
            if current_subject_id == str(target_subject_id):
                return one_subject_result

        return None

    def _get_precomputed_cache_dicts(one_result):
        if one_result is None:
            return None, None

        inner_cache_dict = one_result.get('inner_fold_cache_by_outer_fold', None)
        outer_cache_dict = one_result.get('outer_fold_cache_by_outer_fold', None)

        if isinstance(inner_cache_dict, dict) and isinstance(outer_cache_dict, dict):
            return inner_cache_dict, outer_cache_dict

        return None, None

    if _should_run_subjectwise_block_cv_beta(meta_df=meta_df, split_mode=split_mode):
        subject_result_list = []

        subject_id_list = sorted(pd.Series(meta_df['subject_id']).astype(str).unique().tolist())

        for one_subject_id in subject_id_list:
            subject_mask = meta_df['subject_id'].astype(str).values == str(one_subject_id)
            subject_first_window_data = {
                'X': np.asarray(first_window_data['X'][subject_mask], dtype=float) if 'X' in first_window_data else None,
                'trial_list': [first_window_data['trial_list'][idx] for idx in np.where(subject_mask)[0]] if 'trial_list' in first_window_data else None,
                'y': np.asarray(first_window_data['y'][subject_mask], dtype=int),
                'meta_df': meta_df.loc[subject_mask].reset_index(drop=True),
                'info_dict': dict(info_dict)
            }

            subject_build_fold_cache_func = (
                lambda test_split_value, allowed_split_values, target_subject_id=one_subject_id:
                    build_fold_cache_func(
                        test_split_value=test_split_value,
                        allowed_split_values=allowed_split_values,
                        target_subject_id=target_subject_id
                    )
            )

            one_subject_precomputed_nonstable_result = _extract_subject_level_nonstable_result(
                one_result=precomputed_nonstable_result,
                target_subject_id=one_subject_id
            )

            one_subject_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
                first_window_data=subject_first_window_data,
                split_mode=split_mode,
                candidate_threshold_list=candidate_threshold_list,
                min_stable_windows=min_stable_windows,
                model_name=model_name,
                build_fold_cache_func=subject_build_fold_cache_func,
                max_accuracy_drop=max_accuracy_drop,
                precomputed_nonstable_result=one_subject_precomputed_nonstable_result
            )
            subject_result_list.append(one_subject_result)

        return _aggregate_subject_level_outer_cv_results_beta(
            subject_result_list=subject_result_list,
            model_name=model_name,
            split_mode=split_mode
        )

    split_key = get_split_key_beta(split_mode=split_mode)
    outer_split_values = sorted(meta_df[split_key].unique().tolist())

    outer_fold_rows = []
    threshold_search_by_outer_fold = {}

    precomputed_inner_cache_by_outer_fold, precomputed_outer_cache_by_outer_fold = _get_precomputed_cache_dicts(
        one_result=precomputed_nonstable_result
    )

    for outer_test_split_value in outer_split_values:
        outer_train_split_values = [x for x in outer_split_values if x != outer_test_split_value]

        if (
            isinstance(precomputed_inner_cache_by_outer_fold, dict)
            and outer_test_split_value in precomputed_inner_cache_by_outer_fold
        ):
            inner_fold_cache_dict = precomputed_inner_cache_by_outer_fold[outer_test_split_value]
        else:
            inner_fold_cache_dict = {}

            for inner_val_split_value in outer_train_split_values:
                inner_fold_cache_dict[inner_val_split_value] = build_fold_cache_func(
                    test_split_value=inner_val_split_value,
                    allowed_split_values=outer_train_split_values
                )

        def evaluate_one_inner_fold_func(inner_val_split_value, threshold):
            return evaluate_stable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=threshold,
                model_name=model_name,
                min_stable_windows=min_stable_windows
            )

        baseline_fold_rows = []
        for inner_val_split_value in outer_train_split_values:
            baseline_result = evaluate_stable_early_stop_from_cache_beta(
                fold_cache=inner_fold_cache_dict[inner_val_split_value],
                threshold=np.inf,
                model_name=model_name,
                min_stable_windows=min_stable_windows
            )
            baseline_fold_rows.append(baseline_result['fold_summary_row'])

        baseline_df = pd.DataFrame(baseline_fold_rows)
        baseline_accuracy = float(baseline_df['fold_accuracy'].mean())

        current_candidate_threshold_list = generate_candidate_threshold_list_from_inner_fold_caches_beta(
            inner_fold_cache_dict=inner_fold_cache_dict,
            fallback_candidate_threshold_list=candidate_threshold_list,
            n_thresholds=50,
            q_min=0.01,
            q_max=0.995
        )

        threshold_search_result = search_best_threshold_nonstable_early_stop_beta(
            inner_split_values=outer_train_split_values,
            candidate_threshold_list=current_candidate_threshold_list,
            evaluate_one_inner_fold_func=evaluate_one_inner_fold_func,
            baseline_accuracy=baseline_accuracy,
            max_accuracy_drop=max_accuracy_drop
        )

        best_threshold = threshold_search_result['best_threshold']

        threshold_search_df = threshold_search_result['threshold_search_df'].copy()
        threshold_search_df['outer_test_split_value'] = outer_test_split_value
        threshold_search_df['model_name'] = str(model_name)
        threshold_search_df['min_stable_windows'] = int(min_stable_windows)
        threshold_search_by_outer_fold[outer_test_split_value] = threshold_search_df

        if (
            isinstance(precomputed_outer_cache_by_outer_fold, dict)
            and outer_test_split_value in precomputed_outer_cache_by_outer_fold
        ):
            outer_fold_cache = precomputed_outer_cache_by_outer_fold[outer_test_split_value]
        else:
            outer_fold_cache = build_fold_cache_func(
                test_split_value=outer_test_split_value,
                allowed_split_values=None
            )

        outer_fold_result = evaluate_stable_early_stop_from_cache_beta(
            fold_cache=outer_fold_cache,
            threshold=best_threshold,
            model_name=model_name,
            min_stable_windows=min_stable_windows
        )

        fold_summary_row = outer_fold_result['fold_summary_row'].copy()
        fold_summary_row['baseline_inner_accuracy'] = float(threshold_search_result['baseline_accuracy'])
        fold_summary_row['min_required_accuracy'] = float(threshold_search_result['min_required_accuracy'])
        fold_summary_row['best_threshold_from_inner_cv'] = float(best_threshold) if np.isfinite(best_threshold) else np.inf
        fold_summary_row['threshold_selection_mode'] = threshold_search_result['selection_mode']
        outer_fold_rows.append(fold_summary_row)

    summary_result = summarize_outer_folds_nonstable_early_stop_beta(
        fold_summary_rows=outer_fold_rows,
        model_name=model_name,
        split_mode=split_mode,
        n_classes=info_dict['n_classes']
    )

    summary_result['summary_df']['min_stable_windows'] = int(min_stable_windows)

    return {
        'summary_df': summary_result['summary_df'],
        'fold_results_df': summary_result['fold_results_df'],
        'threshold_search_by_outer_fold': threshold_search_by_outer_fold
    }

### 设置统一稳定性参数

In [ ]:
# ------------ 稳定性早停：统一参数设置 ------------

min_stable_windows = 2
split_mode = 'block'
max_accuracy_drop = None

### 六个模型的主调用代码

In [ ]:
# ------------ SVM：稳定性早停主调用 ------------

first_window_sec = float(multiscale_feature_package['window_sec_list'][0])
first_window_data_feature = multiscale_feature_package['data_by_window'][first_window_sec]

def build_svm_cache_func(test_split_value, allowed_split_values, target_subject_id=None):
    current_package = multiscale_feature_package
    if target_subject_id is not None:
        current_package = _subset_multiscale_feature_package_by_subject_beta(
            multiscale_feature_package=multiscale_feature_package,
            target_subject_id=target_subject_id
        )

    return build_svm_fold_prediction_cache_beta(
        multiscale_feature_package=current_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        C=1.0,
        kernel='rbf',
        gamma='scale',
        probability=True,
        random_state=42
    )

svm_stable_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data=first_window_data_feature,
    split_mode=split_mode,
    candidate_threshold_list=candidate_threshold_list_svm,
    min_stable_windows=min_stable_windows,
    model_name='SVM',
    build_fold_cache_func=build_svm_cache_func,
    max_accuracy_drop=max_accuracy_drop,
    precomputed_nonstable_result=globals().get('svm_nonstable_result', None)
)

display(svm_stable_result['summary_df'])
display(svm_stable_result['fold_results_df'])

# 保存 SVM 稳定性早停结果；之后释放对应 nonstable prediction cache，保留 summary/fold/threshold 表。
save_early_stop_tables_beta(
    result=svm_stable_result,
    model_name='SVM',
    strategy_name='stable_early_stop',
    cache_result=svm_nonstable_result,
    min_stable_windows=min_stable_windows,
    save_window_cache=False,
    save_fixed_window_from_cache=False
)
release_prediction_caches_from_result_beta(svm_nonstable_result)


In [ ]:
# ------------ LDA：稳定性早停主调用 ------------

def build_lda_cache_func(test_split_value, allowed_split_values, target_subject_id=None):
    current_package = multiscale_feature_package
    if target_subject_id is not None:
        current_package = _subset_multiscale_feature_package_by_subject_beta(
            multiscale_feature_package=multiscale_feature_package,
            target_subject_id=target_subject_id
        )

    return build_lda_fold_prediction_cache_beta(
        multiscale_feature_package=current_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        solver='lsqr',
        shrinkage='auto'
    )

lda_stable_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data=first_window_data_feature,
    split_mode=split_mode,
    candidate_threshold_list=candidate_threshold_list_lda,
    min_stable_windows=min_stable_windows,
    model_name='LDA',
    build_fold_cache_func=build_lda_cache_func,
    max_accuracy_drop=max_accuracy_drop,
    precomputed_nonstable_result=globals().get('lda_nonstable_result', None)
)

display(lda_stable_result['summary_df'])
display(lda_stable_result['fold_results_df'])

# 保存 LDA 稳定性早停结果；之后释放对应 nonstable prediction cache，保留 summary/fold/threshold 表。
save_early_stop_tables_beta(
    result=lda_stable_result,
    model_name='LDA',
    strategy_name='stable_early_stop',
    cache_result=lda_nonstable_result,
    min_stable_windows=min_stable_windows,
    save_window_cache=False,
    save_fixed_window_from_cache=False
)
release_prediction_caches_from_result_beta(lda_nonstable_result)


In [ ]:
# ------------ CCA：稳定性早停主调用 ------------

first_window_sec = float(multiscale_trial_package['window_sec_list'][0])
first_window_data_trial = multiscale_trial_package['data_by_window'][first_window_sec]

cca_reference_cache = prepare_cca_reference_cache_beta(
    multiscale_trial_package=multiscale_trial_package,
    n_harmonics=5
)

def build_cca_cache_func(test_split_value, allowed_split_values, target_subject_id=None):
    current_package = multiscale_trial_package
    if target_subject_id is not None:
        current_package = _subset_multiscale_trial_package_by_subject_beta(
            multiscale_trial_package=multiscale_trial_package,
            target_subject_id=target_subject_id
        )

    return build_cca_fold_prediction_cache_beta(
        multiscale_trial_package=current_package,
        cca_reference_cache=cca_reference_cache,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        apply_bandpass=True,
        lowcut=6.0,
        highcut=80.0,
        filter_order=4
    )

cca_stable_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data=first_window_data_trial,
    split_mode=split_mode,
    candidate_threshold_list=candidate_threshold_list_cca,
    min_stable_windows=min_stable_windows,
    model_name='CCA',
    build_fold_cache_func=build_cca_cache_func,
    max_accuracy_drop=max_accuracy_drop,
    precomputed_nonstable_result=globals().get('cca_nonstable_result', None)
)

display(cca_stable_result['summary_df'])
display(cca_stable_result['fold_results_df'])

# 保存 CCA 稳定性早停结果；之后释放对应 nonstable prediction cache，保留 summary/fold/threshold 表。
save_early_stop_tables_beta(
    result=cca_stable_result,
    model_name='CCA',
    strategy_name='stable_early_stop',
    cache_result=cca_nonstable_result,
    min_stable_windows=min_stable_windows,
    save_window_cache=False,
    save_fixed_window_from_cache=False
)
release_prediction_caches_from_result_beta(cca_nonstable_result)


In [ ]:
# ------------ FBCCA：稳定性早停主调用 ------------

fbcca_reference_cache = prepare_fbcca_reference_cache_beta(
    multiscale_trial_package=multiscale_trial_package,
    n_harmonics=5
)

def build_fbcca_cache_func(test_split_value, allowed_split_values, target_subject_id=None):
    current_package = multiscale_trial_package
    if target_subject_id is not None:
        current_package = _subset_multiscale_trial_package_by_subject_beta(
            multiscale_trial_package=multiscale_trial_package,
            target_subject_id=target_subject_id
        )

    return build_fbcca_fold_prediction_cache_beta(
        multiscale_trial_package=current_package,
        fbcca_reference_cache=fbcca_reference_cache,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
        subband_highcut=90.0,
        filter_order=4,
        weight_a=1.25,
        weight_b=0.25
    )

fbcca_stable_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data=first_window_data_trial,
    split_mode=split_mode,
    candidate_threshold_list=candidate_threshold_list_fbcca,
    min_stable_windows=min_stable_windows,
    model_name='FBCCA',
    build_fold_cache_func=build_fbcca_cache_func,
    max_accuracy_drop=max_accuracy_drop,
    precomputed_nonstable_result=globals().get('fbcca_nonstable_result', None)
)

display(fbcca_stable_result['summary_df'])
display(fbcca_stable_result['fold_results_df'])

# 保存 FBCCA 稳定性早停结果；之后释放对应 nonstable prediction cache，保留 summary/fold/threshold 表。
save_early_stop_tables_beta(
    result=fbcca_stable_result,
    model_name='FBCCA',
    strategy_name='stable_early_stop',
    cache_result=fbcca_nonstable_result,
    min_stable_windows=min_stable_windows,
    save_window_cache=False,
    save_fixed_window_from_cache=False
)
release_prediction_caches_from_result_beta(fbcca_nonstable_result)


In [ ]:
# ------------ TRCA：稳定性早停主调用 ------------

def build_trca_cache_func(test_split_value, allowed_split_values, target_subject_id=None):
    current_package = multiscale_trial_package
    if target_subject_id is not None:
        current_package = _subset_multiscale_trial_package_by_subject_beta(
            multiscale_trial_package=multiscale_trial_package,
            target_subject_id=target_subject_id
        )

    return build_trca_fold_prediction_cache_beta(
        multiscale_trial_package=current_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        use_ensemble=True,
        subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
        subband_highcut=90.0,
        filter_order=4,
        weight_a=1.25,
        weight_b=0.25,
        regularization=1e-8
    )

trca_stable_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data=first_window_data_trial,
    split_mode=split_mode,
    candidate_threshold_list=candidate_threshold_list_trca,
    min_stable_windows=min_stable_windows,
    model_name='TRCA',
    build_fold_cache_func=build_trca_cache_func,
    max_accuracy_drop=max_accuracy_drop,
    precomputed_nonstable_result=globals().get('trca_nonstable_result', None)
)

display(trca_stable_result['summary_df'])
display(trca_stable_result['fold_results_df'])

# 保存 TRCA 稳定性早停结果；之后释放对应 nonstable prediction cache，保留 summary/fold/threshold 表。
save_early_stop_tables_beta(
    result=trca_stable_result,
    model_name='TRCA',
    strategy_name='stable_early_stop',
    cache_result=trca_nonstable_result,
    min_stable_windows=min_stable_windows,
    save_window_cache=False,
    save_fixed_window_from_cache=False
)
release_prediction_caches_from_result_beta(trca_nonstable_result)


In [ ]:
# ------------ msTRCA：稳定性早停主调用 ------------

def build_mstrca_cache_func(test_split_value, allowed_split_values, target_subject_id=None):
    current_package = multiscale_trial_package
    if target_subject_id is not None:
        current_package = _subset_multiscale_trial_package_by_subject_beta(
            multiscale_trial_package=multiscale_trial_package,
            target_subject_id=target_subject_id
        )

    return build_mstrca_fold_prediction_cache_beta(
        multiscale_trial_package=current_package,
        split_mode=split_mode,
        test_split_value=test_split_value,
        allowed_split_values=allowed_split_values,
        use_ensemble=True,
        neighbor_template_count=2,  # 2个邻近频率 + 目标频率 = 共3个频率参与ms-eTRCA空间滤波器训练
        subband_low_list=[6.0, 14.0, 22.0, 30.0, 38.0],
        subband_highcut=90.0,
        filter_order=4,
        weight_a=1.25,
        weight_b=0.25,
        regularization=1e-8
    )

mstrca_stable_result = run_stable_early_stop_experiment_from_fold_cache_builder_beta(
    first_window_data=first_window_data_trial,
    split_mode=split_mode,
    candidate_threshold_list=candidate_threshold_list_mstrca,
    min_stable_windows=min_stable_windows,
    model_name='msTRCA',
    build_fold_cache_func=build_mstrca_cache_func,
    max_accuracy_drop=max_accuracy_drop,
    precomputed_nonstable_result=globals().get('mstrca_nonstable_result', None)
)

display(mstrca_stable_result['summary_df'])
display(mstrca_stable_result['fold_results_df'])

# 保存 msTRCA 稳定性早停结果；之后释放对应 nonstable prediction cache，保留 summary/fold/threshold 表。
save_early_stop_tables_beta(
    result=mstrca_stable_result,
    model_name='msTRCA',
    strategy_name='stable_early_stop',
    cache_result=mstrca_nonstable_result,
    min_stable_windows=min_stable_windows,
    save_window_cache=False,
    save_fixed_window_from_cache=False
)
release_prediction_caches_from_result_beta(mstrca_nonstable_result)


In [ ]:
# ------------ 稳定性早停：汇总 6 个模型结果 ------------

stable_summary_df = pd.concat([
    svm_stable_result['summary_df'],
    lda_stable_result['summary_df'],
    cca_stable_result['summary_df'],
    fbcca_stable_result['summary_df'],
    trca_stable_result['summary_df'],
    mstrca_stable_result['summary_df']
], axis=0, ignore_index=True)

display(stable_summary_df)

# 导出一个 Excel 汇总文件，便于快速查看 summary/fold/window_sweep/threshold_sweep 小表。
finalize_result_excel_beta()


In [ ]:
# ------------ 对比：非稳定性早停 vs 稳定性早停 ------------

# 如果你当前变量名不是这两个，请把这里改成你实际的变量名
nonstable_df = nonstable_summary_df.copy()
stable_df = stable_summary_df.copy()

# 只保留两边共同需要比较的列
compare_cols_nonstable = [
    'model_name',
    'mean_fold_accuracy',
    'mean_decision_time_sec',
    'mean_decision_time_with_runtime_sec',
    'mean_early_stop_rate',
    'itr_bits_per_min',
    'itr_with_runtime_bits_per_min'
]

compare_cols_stable = [
    'model_name',
    'mean_fold_accuracy',
    'mean_decision_time_sec',
    'mean_decision_time_with_runtime_sec',
    'mean_early_stop_rate',
    'itr_bits_per_min',
    'itr_with_runtime_bits_per_min'
]

nonstable_part = nonstable_df[compare_cols_nonstable].copy()
stable_part = stable_df[compare_cols_stable].copy()

# 重命名，便于区分
nonstable_part = nonstable_part.rename(columns={
    'mean_fold_accuracy': 'accuracy_nonstable',
    'mean_decision_time_sec': 'decision_time_nonstable',
    'mean_decision_time_with_runtime_sec': 'decision_time_with_runtime_nonstable',
    'mean_early_stop_rate': 'early_stop_rate_nonstable',
    'itr_bits_per_min': 'itr_nonstable',
    'itr_with_runtime_bits_per_min': 'itr_with_runtime_nonstable'
})

stable_part = stable_part.rename(columns={
    'mean_fold_accuracy': 'accuracy_stable',
    'mean_decision_time_sec': 'decision_time_stable',
    'mean_decision_time_with_runtime_sec': 'decision_time_with_runtime_stable',
    'mean_early_stop_rate': 'early_stop_rate_stable',
    'itr_bits_per_min': 'itr_stable',
    'itr_with_runtime_bits_per_min': 'itr_with_runtime_stable'
})

# 按模型合并
compare_stable_vs_nonstable_df = pd.merge(
    nonstable_part,
    stable_part,
    on='model_name',
    how='inner'
)

# 计算差值：稳定性早停 - 非稳定性早停
compare_stable_vs_nonstable_df['delta_accuracy'] = (
    compare_stable_vs_nonstable_df['accuracy_stable']
    - compare_stable_vs_nonstable_df['accuracy_nonstable']
)

compare_stable_vs_nonstable_df['delta_decision_time_sec'] = (
    compare_stable_vs_nonstable_df['decision_time_stable']
    - compare_stable_vs_nonstable_df['decision_time_nonstable']
)

compare_stable_vs_nonstable_df['delta_early_stop_rate'] = (
    compare_stable_vs_nonstable_df['early_stop_rate_stable']
    - compare_stable_vs_nonstable_df['early_stop_rate_nonstable']
)

compare_stable_vs_nonstable_df['delta_decision_time_with_runtime_sec'] = (
    compare_stable_vs_nonstable_df['decision_time_with_runtime_stable']
    - compare_stable_vs_nonstable_df['decision_time_with_runtime_nonstable']
)

compare_stable_vs_nonstable_df['delta_itr'] = (
    compare_stable_vs_nonstable_df['itr_stable']
    - compare_stable_vs_nonstable_df['itr_nonstable']
)

compare_stable_vs_nonstable_df['delta_itr_with_runtime'] = (
    compare_stable_vs_nonstable_df['itr_with_runtime_stable']
    - compare_stable_vs_nonstable_df['itr_with_runtime_nonstable']
)

# 调整列顺序
compare_stable_vs_nonstable_df = compare_stable_vs_nonstable_df[
    [
        'model_name',
        'accuracy_nonstable',
        'accuracy_stable',
        'delta_accuracy',
        'decision_time_nonstable',
        'decision_time_stable',
        'delta_decision_time_sec',
        'decision_time_with_runtime_nonstable',
        'decision_time_with_runtime_stable',
        'delta_decision_time_with_runtime_sec',
        'early_stop_rate_nonstable',
        'early_stop_rate_stable',
        'delta_early_stop_rate',
        'itr_nonstable',
        'itr_stable',
        'delta_itr',
        'itr_with_runtime_nonstable',
        'itr_with_runtime_stable',
        'delta_itr_with_runtime'
    ]
].copy()

# 排序（按模型名）
compare_stable_vs_nonstable_df = compare_stable_vs_nonstable_df.sort_values(
    by='model_name'
).reset_index(drop=True)

display(compare_stable_vs_nonstable_df)

In [ ]:
# ------------ 稳定性早停 vs 非稳定性早停：差值对比表（带颜色） ------------

import pandas as pd
import numpy as np

# 1) 取出要比较的列
nonstable_compare_df = nonstable_summary_df[[
    'model_name',
    'mean_fold_accuracy',
    'mean_decision_time_sec',
    'mean_decision_time_with_runtime_sec',
    'mean_early_stop_rate',
    'itr_bits_per_min',
    'itr_with_runtime_bits_per_min'
]].copy()

stable_compare_df = stable_summary_df[[
    'model_name',
    'mean_fold_accuracy',
    'mean_decision_time_sec',
    'mean_decision_time_with_runtime_sec',
    'mean_early_stop_rate',
    'itr_bits_per_min',
    'itr_with_runtime_bits_per_min'
]].copy()

# 2) 按 model_name 对齐
compare_df = pd.merge(
    stable_compare_df,
    nonstable_compare_df,
    on='model_name',
    how='inner',
    suffixes=('_stable', '_nonstable')
)

# 3) 只保留“做差结果”
#    这里统一定义为：稳定性早停 - 非稳定性早停
delta_df = pd.DataFrame()
delta_df['model_name'] = compare_df['model_name']
delta_df['accuracy_diff'] = compare_df['mean_fold_accuracy_stable'] - compare_df['mean_fold_accuracy_nonstable']
delta_df['decision_time_diff_sec'] = compare_df['mean_decision_time_sec_stable'] - compare_df['mean_decision_time_sec_nonstable']
delta_df['decision_time_with_runtime_diff_sec'] = compare_df['mean_decision_time_with_runtime_sec_stable'] - compare_df['mean_decision_time_with_runtime_sec_nonstable']
delta_df['early_stop_rate_diff'] = compare_df['mean_early_stop_rate_stable'] - compare_df['mean_early_stop_rate_nonstable']
delta_df['itr_diff'] = compare_df['itr_bits_per_min_stable'] - compare_df['itr_bits_per_min_nonstable']
delta_df['itr_with_runtime_diff'] = compare_df['itr_with_runtime_bits_per_min_stable'] - compare_df['itr_with_runtime_bits_per_min_nonstable']

# 4) 定义上色函数
def make_diff_color(val, positive_is_better=True, vmax=1.0):
    if pd.isna(val):
        return ''

    if vmax <= 0:
        vmax = 1.0

    strength = min(abs(val) / vmax, 1.0)
    alpha = 0.15 + 0.55 * strength

    if positive_is_better:
        is_good = (val > 0)
    else:
        is_good = (val < 0)

    if val == 0:
        return 'background-color: rgba(240, 240, 240, 0.35); color: black;'

    if is_good:
        return f'background-color: rgba(0, 160, 0, {alpha:.3f}); color: black;'
    else:
        return f'background-color: rgba(220, 0, 0, {alpha:.3f}); color: black;'

# 5) 每列单独控制颜色深浅
vmax_accuracy = max(delta_df['accuracy_diff'].abs().max(), 1e-12)
vmax_time = max(delta_df['decision_time_diff_sec'].abs().max(), 1e-12)
vmax_time_runtime = max(delta_df['decision_time_with_runtime_diff_sec'].abs().max(), 1e-12) if 'decision_time_with_runtime_diff_sec' in delta_df.columns else 1e-12
vmax_early = max(delta_df['early_stop_rate_diff'].abs().max(), 1e-12)
vmax_itr = max(delta_df['itr_diff'].abs().max(), 1e-12)
vmax_itr_runtime = max(delta_df['itr_with_runtime_diff'].abs().max(), 1e-12) if 'itr_with_runtime_diff' in delta_df.columns else 1e-12

styled_delta_df = (
    delta_df.style
    .format({
        'accuracy_diff': '{:+.5f}',
        'decision_time_diff_sec': '{:+.5f}',
        'decision_time_with_runtime_diff_sec': '{:+.5f}',
        'early_stop_rate_diff': '{:+.5f}',
        'itr_diff': '{:+.5f}',
        'itr_with_runtime_diff': '{:+.5f}'
    })
    .map(lambda v: make_diff_color(v, positive_is_better=True,  vmax=vmax_accuracy), subset=['accuracy_diff'])
    .map(lambda v: make_diff_color(v, positive_is_better=False, vmax=vmax_time), subset=['decision_time_diff_sec'])
    .map(lambda v: make_diff_color(v, positive_is_better=False, vmax=vmax_time_runtime), subset=['decision_time_with_runtime_diff_sec'])
    .map(lambda v: make_diff_color(v, positive_is_better=True,  vmax=vmax_early), subset=['early_stop_rate_diff'])
    .map(lambda v: make_diff_color(v, positive_is_better=True,  vmax=vmax_itr), subset=['itr_diff'])
    .map(lambda v: make_diff_color(v, positive_is_better=True,  vmax=vmax_itr_runtime), subset=['itr_with_runtime_diff'])
)

display(styled_delta_df)

In [ ]:
# ------------ 统计：非稳定早停 / 稳定性早停 各模型的平均阈值 ------------

import numpy as np
import pandas as pd

def build_threshold_stat_df(result_dict_map, strategy_name):
    stat_rows = []

    for model_name, one_result in result_dict_map.items():
        if one_result is None:
            continue

        fold_results_df = one_result.get('fold_results_df', None)
        if fold_results_df is None or len(fold_results_df) == 0:
            continue

        if 'best_threshold_from_inner_cv' not in fold_results_df.columns:
            continue

        current_df = fold_results_df.copy()

        # 原始阈值列
        threshold_series_raw = pd.to_numeric(
            current_df['best_threshold_from_inner_cv'],
            errors='coerce'
        )

        # 把 inf 视为“没有有效阈值 / fallback_to_no_early_stop”，均值时不纳入
        threshold_series_valid = threshold_series_raw.replace([np.inf, -np.inf], np.nan)

        # fallback 统计
        if 'threshold_selection_mode' in current_df.columns:
            fallback_mask = current_df['threshold_selection_mode'].astype(str) == 'fallback_to_no_early_stop'
            n_fallback = int(fallback_mask.sum())
        else:
            n_fallback = int(np.isinf(threshold_series_raw).sum())

        stat_rows.append({
            'model_name': str(model_name),
            'strategy': str(strategy_name),
            'mean_threshold': float(threshold_series_valid.mean()) if threshold_series_valid.notna().any() else np.nan,
            'median_threshold': float(threshold_series_valid.median()) if threshold_series_valid.notna().any() else np.nan,
            'std_threshold': float(threshold_series_valid.std(ddof=0)) if threshold_series_valid.notna().any() else np.nan,
            'min_threshold': float(threshold_series_valid.min()) if threshold_series_valid.notna().any() else np.nan,
            'max_threshold': float(threshold_series_valid.max()) if threshold_series_valid.notna().any() else np.nan,
            'n_total_folds': int(len(current_df)),
            'n_valid_threshold_folds': int(threshold_series_valid.notna().sum()),
            'n_fallback_folds': int(n_fallback)
        })

    stat_df = pd.DataFrame(stat_rows)

    if len(stat_df) > 0:
        stat_df = stat_df.sort_values(
            by=['strategy', 'model_name'],
            ascending=[True, True]
        ).reset_index(drop=True)

    return stat_df


# 1) 收集当前 notebook 中的“非稳定早停”结果变量
nonstable_result_dict = {
    'SVM': globals().get('svm_nonstable_result', None),
    'LDA': globals().get('lda_nonstable_result', None),
    'CCA': globals().get('cca_nonstable_result', None),
    'FBCCA': globals().get('fbcca_nonstable_result', None),
    'TRCA': globals().get('trca_nonstable_result', None),
    'msTRCA': globals().get('mstrca_nonstable_result', None)
}

# 2) 收集当前 notebook 中的“稳定性早停”结果变量
stable_result_dict = {
    'SVM': globals().get('svm_stable_result', None),
    'LDA': globals().get('lda_stable_result', None),
    'CCA': globals().get('cca_stable_result', None),
    'FBCCA': globals().get('fbcca_stable_result', None),
    'TRCA': globals().get('trca_stable_result', None),
    'msTRCA': globals().get('mstrca_stable_result', None)
}

# 3) 分别统计
nonstable_threshold_stat_df = build_threshold_stat_df(
    result_dict_map=nonstable_result_dict,
    strategy_name='nonstable'
)

stable_threshold_stat_df = build_threshold_stat_df(
    result_dict_map=stable_result_dict,
    strategy_name='stable'
)

# 4) 合并成总表
all_threshold_stat_df = pd.concat(
    [nonstable_threshold_stat_df, stable_threshold_stat_df],
    axis=0,
    ignore_index=True
)

print('========== 非稳定早停：各模型阈值统计 ==========')
display(nonstable_threshold_stat_df)

print('========== 稳定性早停：各模型阈值统计 ==========')
display(stable_threshold_stat_df)

print('========== 全部阈值统计总表 ==========')
display(all_threshold_stat_df)


# 5) 额外生成一个“非稳定 vs 稳定”的平均阈值对比表
threshold_mean_compare_df = pd.merge(
    nonstable_threshold_stat_df[['model_name', 'mean_threshold']].rename(
        columns={'mean_threshold': 'mean_threshold_nonstable'}
    ),
    stable_threshold_stat_df[['model_name', 'mean_threshold']].rename(
        columns={'mean_threshold': 'mean_threshold_stable'}
    ),
    on='model_name',
    how='outer'
)

threshold_mean_compare_df['delta_mean_threshold'] = (
    threshold_mean_compare_df['mean_threshold_stable']
    - threshold_mean_compare_df['mean_threshold_nonstable']
)

print('========== 平均阈值对比表（稳定 - 非稳定） ==========')
display(threshold_mean_compare_df)

In [ ]:
# ------------ 统计：每个内层验证 Block 单独选出的最佳阈值出现次数 ------------

import numpy as np
import pandas as pd


def iter_threshold_search_df_from_result_beta(one_result):
    if one_result is None:
        return

    threshold_search_obj = one_result.get('threshold_search_by_outer_fold', None)
    if threshold_search_obj is None:
        return

    def _walk(obj):
        if isinstance(obj, pd.DataFrame):
            yield obj
        elif isinstance(obj, dict):
            for value in obj.values():
                yield from _walk(value)

    yield from _walk(threshold_search_obj)


def build_inner_block_best_threshold_count_df(result_dict_map, strategy_name):
    count_rows = []

    for model_name, one_result in result_dict_map.items():
        if one_result is None:
            continue

        one_model_threshold_rows = []

        for threshold_search_df in iter_threshold_search_df_from_result_beta(one_result):
            if threshold_search_df is None or len(threshold_search_df) == 0:
                continue

            if 'inner_best_threshold_count' not in threshold_search_df.columns:
                continue

            current_df = threshold_search_df.copy()
            current_df['threshold'] = pd.to_numeric(current_df['threshold'], errors='coerce')
            current_df['inner_best_threshold_count'] = pd.to_numeric(
                current_df['inner_best_threshold_count'],
                errors='coerce'
            ).fillna(0).astype(int)

            # 每个 outer fold 的候选阈值理论上不重复；这里仍先按 threshold 聚合一次，避免候选列表意外重复时重复计数。
            current_count_df = current_df[['threshold', 'inner_best_threshold_count']].groupby(
                'threshold',
                as_index=False
            ).agg(
                inner_best_threshold_count=('inner_best_threshold_count', 'max')
            )

            one_model_threshold_rows.append(current_count_df)

        if len(one_model_threshold_rows) == 0:
            continue

        one_model_count_df = pd.concat(
            one_model_threshold_rows,
            axis=0,
            ignore_index=True
        )

        one_model_count_df = one_model_count_df.groupby('threshold', as_index=False).agg(
            count=('inner_best_threshold_count', 'sum')
        )

        total_count = int(one_model_count_df['count'].sum())
        one_model_count_df['percentage'] = (
            one_model_count_df['count'] / total_count * 100.0
            if total_count > 0
            else 0.0
        )

        one_model_count_df['model_name'] = str(model_name)
        one_model_count_df['strategy'] = str(strategy_name)
        one_model_count_df['total_inner_validation_blocks'] = int(total_count)

        count_rows.append(one_model_count_df)

    if len(count_rows) == 0:
        return pd.DataFrame(columns=[
            'strategy',
            'model_name',
            'threshold',
            'count',
            'percentage',
            'total_inner_validation_blocks'
        ])

    count_df = pd.concat(count_rows, axis=0, ignore_index=True)
    count_df = count_df[[
        'strategy',
        'model_name',
        'threshold',
        'count',
        'percentage',
        'total_inner_validation_blocks'
    ]]

    count_df = count_df.sort_values(
        by=['strategy', 'model_name', 'count', 'threshold'],
        ascending=[True, True, False, True]
    ).reset_index(drop=True)

    return count_df


# 1) 收集当前 notebook 中的“非稳定早停”结果变量
nonstable_result_dict = {
    'SVM': globals().get('svm_nonstable_result', None),
    'LDA': globals().get('lda_nonstable_result', None),
    'CCA': globals().get('cca_nonstable_result', None),
    'FBCCA': globals().get('fbcca_nonstable_result', None),
    'TRCA': globals().get('trca_nonstable_result', None),
    'msTRCA': globals().get('mstrca_nonstable_result', None)
}

# 2) 收集当前 notebook 中的“稳定性早停”结果变量
stable_result_dict = {
    'SVM': globals().get('svm_stable_result', None),
    'LDA': globals().get('lda_stable_result', None),
    'CCA': globals().get('cca_stable_result', None),
    'FBCCA': globals().get('fbcca_stable_result', None),
    'TRCA': globals().get('trca_stable_result', None),
    'msTRCA': globals().get('mstrca_stable_result', None)
}

# 3) 分别统计“每个 inner validation block 自己单独选出的最佳阈值”出现次数
nonstable_inner_best_threshold_count_df = build_inner_block_best_threshold_count_df(
    result_dict_map=nonstable_result_dict,
    strategy_name='nonstable'
)

stable_inner_best_threshold_count_df = build_inner_block_best_threshold_count_df(
    result_dict_map=stable_result_dict,
    strategy_name='stable'
)

all_inner_best_threshold_count_df = pd.concat(
    [nonstable_inner_best_threshold_count_df, stable_inner_best_threshold_count_df],
    axis=0,
    ignore_index=True
)

print('========== 非稳定早停：每个内层验证 Block 单独最佳阈值出现次数 ==========' )
display(nonstable_inner_best_threshold_count_df)

print('========== 稳定性早停：每个内层验证 Block 单独最佳阈值出现次数 ==========' )
display(stable_inner_best_threshold_count_df)

print('========== 全部：每个内层验证 Block 单独最佳阈值出现次数 ==========' )
display(all_inner_best_threshold_count_df)
